In [43]:
# Import general Packages and Functions
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import copy
import os
import random
import warnings
from mpl_toolkits.mplot3d import Axes3D

# Import from sklearn (scikit-learn)
# Data Preprocessing
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.decomposition import PCA
# Metrics
from sklearn.metrics import (
    # classification
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    
    # regression
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    
    # scoring utilities
    make_scorer)
# kNN
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
# LDA & QDA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
# Naive Bayes
from sklearn.naive_bayes import GaussianNB
# Multi Layer Perceptrons
from sklearn.neural_network import MLPClassifier, MLPRegressor
# Support Vector Machines
from sklearn.svm import SVC, SVR
# Tree based Methods & Ensemble Methods
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingRegressor
# k-means 
from sklearn.cluster import KMeans
# Logistic Regression
from sklearn.linear_model import LogisticRegression
# AdaBoost
from sklearn.ensemble import AdaBoostClassifier
# Grid Search for period based CV
from sklearn.model_selection import GridSearchCV, GroupKFold, LeaveOneGroupOut
# Linear Regression Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV
# Other
from sklearn.pipeline import Pipeline
from sklearn.base import clone

# Import XGBoost
from xgboost import XGBClassifier

# Import PyTorch for more complex DeepLearning (Joint Optimisation Model)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, Subset, Dataset

# Ignore Warnings
warnings.filterwarnings("ignore")

In [44]:
# Read Data & Set.Seed
seed = 123
train_url = "https://raw.githubusercontent.com/jmuelleo/Oxford-MSc.-Statistical-Science---Statistical-Machine-Learning---Group-Practical/refs/heads/main/train.csv"
public_test_url = "https://raw.githubusercontent.com/jmuelleo/Oxford-MSc.-Statistical-Science---Statistical-Machine-Learning---Group-Practical/refs/heads/main/public_test.csv"
private_test_url = "https://raw.githubusercontent.com/jmuelleo/Oxford-MSc.-Statistical-Science---Statistical-Machine-Learning---Group-Practical/refs/heads/main/private_test.csv"

train_df = pd.read_csv(train_url)
public_test_df = pd.read_csv(public_test_url)
private_test_df = pd.read_csv(private_test_url)

In [45]:
# Make Train_Train, Train_Val, Pub_test, Priv_test Split
X_train = train_df.iloc[:,3:]
X_pub_test = public_test_df.iloc[:,3:]
X_priv_test = private_test_df.iloc[:,1:]

ys_train = train_df.iloc[:,0]
yc_train = train_df.iloc[:,1]
period_train = train_df.iloc[:,2]

ys_pub_test = public_test_df.iloc[:,0]
yc_pub_test = public_test_df.iloc[:,1]
period_pub_test = public_test_df.iloc[:,2]

period_priv_test = private_test_df.iloc[:,0]

n_train = X_train.shape[0]
p_train = X_train.shape[1]

In [46]:
# Substance Specific Sub Groups:
X_train_s1 = X_train.loc[ys_train == 1, :]
ys_train_s1 = ys_train.loc[ys_train == 1]
yc_train_s1 = yc_train.loc[ys_train == 1]

X_train_s2 = X_train.loc[ys_train == 2, :]
ys_train_s2 = ys_train.loc[ys_train == 2]
yc_train_s2 = yc_train.loc[ys_train == 2]

X_train_s3 = X_train.loc[ys_train == 3, :]
ys_train_s3 = ys_train.loc[ys_train == 3]
yc_train_s3 = yc_train.loc[ys_train == 3]

X_train_s4 = X_train.loc[ys_train == 4, :]
ys_train_s4 = ys_train.loc[ys_train == 4]
yc_train_s4 = yc_train.loc[ys_train == 4]

X_train_s5 = X_train.loc[ys_train == 5, :]
ys_train_s5 = ys_train.loc[ys_train == 5]
yc_train_s5 = yc_train.loc[ys_train == 5]

X_train_s6 = X_train.loc[ys_train == 6, :]
ys_train_s6 = ys_train.loc[ys_train == 6]
yc_train_s6 = yc_train.loc[ys_train == 6]


period_train_s1 = period_train.loc[ys_train == 1]
period_train_s2 = period_train.loc[ys_train == 2]
period_train_s3 = period_train.loc[ys_train == 3]
period_train_s4 = period_train.loc[ys_train == 4]
period_train_s5 = period_train.loc[ys_train == 5]
period_train_s6 = period_train.loc[ys_train == 6]

In [47]:
# Create General period based group splits
gkf = GroupKFold(n_splits = len(np.unique(period_train)))

In [48]:
# Create substance specific period based group splits
gkf1 = GroupKFold(n_splits = len(np.unique(period_train_s1)))
gkf2 = GroupKFold(n_splits = len(np.unique(period_train_s2)))
gkf3 = GroupKFold(n_splits = len(np.unique(period_train_s3)))
gkf4 = GroupKFold(n_splits = len(np.unique(period_train_s4)))
gkf5 = GroupKFold(n_splits = len(np.unique(period_train_s5)))
gkf6 = GroupKFold(n_splits = len(np.unique(period_train_s6)))

# Logistic Regression - Standardised Data

In [49]:
# =========================================================
# LOGISTIC REGRESSION - STANDARDISED DATA
# =========================================================

# Define pipeline of how to transform data and which model to use
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=3000)) # We made max_iter = 3000 to the standard if convergence does not occur under base settings
])

# Define parameter grid to search over
param_grid = [
    {
        "model__penalty": ["l2"],
        "model__C": np.logspace(-2, 4, 30)
    },
    {
        "model__penalty": [None]
    }
]

# Define metrics we want to evaluate
scoring = {
    "balanced_accuracy": "balanced_accuracy",
    "accuracy": "accuracy",
    "f1_macro": "f1_macro"
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=gkf,
    scoring=scoring,
    refit="balanced_accuracy",   # Choose best model based on balanced accuracy
    n_jobs=-1,
    verbose=1,
    return_train_score=False
)

grid.fit(X_train, ys_train, groups=period_train)

# =========================================================
# OVERALL BEST RESULT
# =========================================================
print("=" * 60)
print("BEST MODEL")
print("=" * 60)
print("Best params:", grid.best_params_)
print(f"Best mean CV balanced_accuracy: {grid.best_score_:.4f}") # format to 4 decimal places

results = pd.DataFrame(grid.cv_results_)
best_idx = grid.best_index_

print("\nMean CV metrics for best model:")
print(f"  balanced_accuracy: {results.loc[best_idx, 'mean_test_balanced_accuracy']:.4f}")
print(f"  accuracy:          {results.loc[best_idx, 'mean_test_accuracy']:.4f}")
print(f"  f1_macro:          {results.loc[best_idx, 'mean_test_f1_macro']:.4f}")

# =========================================================
# PER-FOLD SCORES FOR BEST MODEL
# =========================================================
n_splits = gkf.get_n_splits(X_train, ys_train, groups=period_train)

print("\n" + "=" * 60)
print("PER-FOLD SCORES FOR BEST MODEL")
print("=" * 60)

for i in range(n_splits):
    ba = results.loc[best_idx, f"split{i}_test_balanced_accuracy"]
    acc = results.loc[best_idx, f"split{i}_test_accuracy"]
    f1 = results.loc[best_idx, f"split{i}_test_f1_macro"]

    print(f"Fold {i}:")
    print(f"  balanced_accuracy = {ba:.4f}")
    print(f"  accuracy          = {acc:.4f}")
    print(f"  f1_macro          = {f1:.4f}")


# =========================================================
# WEIGHTED MEAN (based on fold sizes) FOR BEST MODEL
# =========================================================
# Get sizes of the validation sets
fold_sizes = []
for _, val_idx in gkf.split(X_train, ys_train, groups=period_train):
    fold_sizes.append(len(val_idx))
fold_sizes = np.array(fold_sizes)

print("\nFold sizes:", fold_sizes)

fold_ba = np.array([
    results.loc[best_idx, f"split{i}_test_balanced_accuracy"]
    for i in range(n_splits)
])

fold_acc = np.array([
    results.loc[best_idx, f"split{i}_test_accuracy"]
    for i in range(n_splits)
])

fold_f1 = np.array([
    results.loc[best_idx, f"split{i}_test_f1_macro"]
    for i in range(n_splits)
])

weighted_ba = np.average(fold_ba, weights=fold_sizes)
weighted_acc = np.average(fold_acc, weights=fold_sizes)
weighted_f1 = np.average(fold_f1, weights=fold_sizes)

print("\n" + "=" * 60)
print("WEIGHTED MEAN CV SCORES")
print("=" * 60)
print(f"Weighted balanced_accuracy: {weighted_ba:.4f}")
print(f"Weighted accuracy:          {weighted_acc:.4f}")
print(f"Weighted f1_macro:          {weighted_f1:.4f}")


# =========================================================
# BEST MODEL ON PUB TEST (Classification)
# =========================================================
best_model = grid.best_estimator_

y_pred = best_model.predict(X_pub_test)
cm = confusion_matrix(ys_pub_test, y_pred)

pub_acc = accuracy_score(ys_pub_test, y_pred)
pub_ba = balanced_accuracy_score(ys_pub_test, y_pred)
pub_f1 = f1_score(ys_pub_test, y_pred, average="macro")

print("\n" + "=" * 60)
print("PUBLIC TEST PERFORMANCE")
print("=" * 60)
print(f"Accuracy:          {pub_acc:.4f}")
print(f"Balanced Accuracy: {pub_ba:.4f}")
print(f"F1 Macro:          {pub_f1:.4f}")

print("\nConfusion Matrix:")
print(cm)

Fitting 6 folds for each of 31 candidates, totalling 186 fits
BEST MODEL
Best params: {'model__C': np.float64(0.2807216203941177), 'model__penalty': 'l2'}
Best mean CV balanced_accuracy: 0.8859

Mean CV metrics for best model:
  balanced_accuracy: 0.8859
  accuracy:          0.8918
  f1_macro:          0.8280

PER-FOLD SCORES FOR BEST MODEL
Fold 0:
  balanced_accuracy = 0.8120
  accuracy          = 0.7813
  f1_macro          = 0.6707
Fold 1:
  balanced_accuracy = 0.9794
  accuracy          = 0.9811
  f1_macro          = 0.9815
Fold 2:
  balanced_accuracy = 0.9763
  accuracy          = 0.9751
  f1_macro          = 0.9744
Fold 3:
  balanced_accuracy = 0.6567
  accuracy          = 0.7416
  f1_macro          = 0.6371
Fold 4:
  balanced_accuracy = 0.9857
  accuracy          = 0.9898
  f1_macro          = 0.9876
Fold 5:
  balanced_accuracy = 0.9052
  accuracy          = 0.8820
  f1_macro          = 0.7170

Fold sizes: [2300 1586 1244  445  197  161]

WEIGHTED MEAN CV SCORES
Weighted balanced

In [81]:
print("CUSTOM CV:")
for i, (_, val_idx) in enumerate(LeaveOneGroupOut().split(X_train, ys_train, groups=period_train)):
    print(i, np.unique(period_train[val_idx]))

print("\nGRID CV:")
for i, (_, val_idx) in enumerate(gkf.split(X_train, ys_train, groups=period_train)):
    print(i, np.unique(period_train[val_idx]))

CUSTOM CV:
0 [1]
1 [2]
2 [3]
3 [4]
4 [5]
5 [6]

GRID CV:
0 [6]
1 [3]
2 [2]
3 [1]
4 [5]
5 [4]


# Linear Discriminant Analysis - Standardised Data

In [50]:
# =========================================================
# LINEAR DISCRIMINANT ANALYSIS - STANDARDISED DATA
# =========================================================

# Define the pipeline of how to transform the data and which model to fit
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearDiscriminantAnalysis())
])

# Define a parameter grid over which we conduct out search
param_grid = [
    {"model__solver": ["svd"]},
    {"model__solver": ["lsqr", "eigen"], "model__shrinkage": [None, "auto"]}
]

# Technically we don't need to redefine scoring, as we defined it before, but we opted to redefine it in each codeblock
# such that there is a lower chance of incoherent choices due to different individuals working on the code on different devices.
# This logic was applied to several such cases
scoring = {
    "balanced_accuracy": "balanced_accuracy",  
    "accuracy": "accuracy",
    "f1_macro": "f1_macro"
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=gkf,
    scoring=scoring,
    refit="balanced_accuracy",   # Choose best model based on balanced accuracy
    n_jobs=-1,
    verbose=1,
    return_train_score=False
)

grid.fit(X_train, ys_train, groups=period_train)

# =========================================================
# OVERALL BEST RESULT
# =========================================================
print("=" * 60)
print("BEST MODEL")
print("=" * 60)
print("Best params:", grid.best_params_)
print(f"Best mean CV balanced_accuracy: {grid.best_score_:.4f}")

results = pd.DataFrame(grid.cv_results_)
best_idx = grid.best_index_

print("\nMean CV metrics for best model:")
print(f"  balanced_accuracy: {results.loc[best_idx, 'mean_test_balanced_accuracy']:.4f}")
print(f"  accuracy:          {results.loc[best_idx, 'mean_test_accuracy']:.4f}")
print(f"  f1_macro:          {results.loc[best_idx, 'mean_test_f1_macro']:.4f}")

# =========================================================
# PER-FOLD SCORES FOR BEST MODEL
# =========================================================
n_splits = gkf.get_n_splits(X_train, ys_train, groups=period_train)

print("\n" + "=" * 60)
print("PER-FOLD SCORES FOR BEST MODEL")
print("=" * 60)

for i in range(n_splits):
    ba = results.loc[best_idx, f"split{i}_test_balanced_accuracy"]
    acc = results.loc[best_idx, f"split{i}_test_accuracy"]
    f1 = results.loc[best_idx, f"split{i}_test_f1_macro"]

    print(f"Fold {i}:")
    print(f"  balanced_accuracy = {ba:.4f}")
    print(f"  accuracy          = {acc:.4f}")
    print(f"  f1_macro          = {f1:.4f}")


# =========================================================
# WEIGHTED MEAN
# =========================================================

# 1) Number of samples
fold_sizes = []
for _, val_idx in gkf.split(X_train, ys_train, groups=period_train):
    fold_sizes.append(len(val_idx))

fold_sizes = np.array(fold_sizes)

print("\nFold sizes:", fold_sizes)

# 2) Scores of the best model
fold_ba = np.array([
    results.loc[best_idx, f"split{i}_test_balanced_accuracy"]
    for i in range(n_splits)
])

fold_acc = np.array([
    results.loc[best_idx, f"split{i}_test_accuracy"]
    for i in range(n_splits)
])

fold_f1 = np.array([
    results.loc[best_idx, f"split{i}_test_f1_macro"]
    for i in range(n_splits)
])

print("Per-fold balanced_accuracy:", fold_ba)
print("Per-fold accuracy:", fold_acc)
print("Per-fold f1_macro:", fold_f1)

# 3) Weighted Means berechnen
weighted_ba = np.average(fold_ba, weights=fold_sizes)
weighted_acc = np.average(fold_acc, weights=fold_sizes)
weighted_f1 = np.average(fold_f1, weights=fold_sizes)

print("\n" + "=" * 60)
print("WEIGHTED MEAN CV SCORES")
print("=" * 60)
print(f"Weighted balanced_accuracy: {weighted_ba:.4f}")
print(f"Weighted accuracy:          {weighted_acc:.4f}")
print(f"Weighted f1_macro:          {weighted_f1:.4f}")

# =========================================================
# BEST MODEL ON PUB TEST (Classification)
# =========================================================
best_model = grid.best_estimator_

y_pred = best_model.predict(X_pub_test)
cm = confusion_matrix(ys_pub_test, y_pred)

pub_acc = accuracy_score(ys_pub_test, y_pred)
pub_ba = balanced_accuracy_score(ys_pub_test, y_pred)
pub_f1 = f1_score(ys_pub_test, y_pred, average="macro")

print("\n" + "=" * 60)
print("PUBLIC TEST PERFORMANCE")
print("=" * 60)
print(f"Accuracy:          {pub_acc:.4f}")
print(f"Balanced Accuracy: {pub_ba:.4f}")
print(f"F1 Macro:          {pub_f1:.4f}")

print("\nConfusion Matrix:")
print(cm)

Fitting 6 folds for each of 5 candidates, totalling 30 fits
BEST MODEL
Best params: {'model__shrinkage': 'auto', 'model__solver': 'lsqr'}
Best mean CV balanced_accuracy: 0.8363

Mean CV metrics for best model:
  balanced_accuracy: 0.8363
  accuracy:          0.8378
  f1_macro:          0.7493

PER-FOLD SCORES FOR BEST MODEL
Fold 0:
  balanced_accuracy = 0.7414
  accuracy          = 0.6743
  f1_macro          = 0.6229
Fold 1:
  balanced_accuracy = 0.9645
  accuracy          = 0.9710
  f1_macro          = 0.8054
Fold 2:
  balanced_accuracy = 0.9129
  accuracy          = 0.9413
  f1_macro          = 0.8352
Fold 3:
  balanced_accuracy = 0.4935
  accuracy          = 0.5371
  f1_macro          = 0.4831
Fold 4:
  balanced_accuracy = 0.9868
  accuracy          = 0.9898
  f1_macro          = 0.8264
Fold 5:
  balanced_accuracy = 0.9188
  accuracy          = 0.9130
  f1_macro          = 0.9224

Fold sizes: [2300 1586 1244  445  197  161]
Per-fold balanced_accuracy: [0.74136229 0.96453427 0.9129  

# Multi Layer Perceptron - Standardised Data -> PCA

In [51]:
# =========================================================
# MULTI LAYER PERCEPTRON - STANDARDISED DATA -> PCA
# =========================================================

# Define pipeline that defines how the data is preprocessed and which model is applied
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA()),
    ("model", MLPClassifier(max_iter=3000,
                            random_state=seed)) # Set seed for reproducibility
])

# Define parameter grid over which we optimise
param_grid = [
    {
        "pca__n_components": [5, 10, 20, 50, 75],
        "model__hidden_layer_sizes": [
            (128, 64),
            (128, 96, 64),
        ],
        "model__activation": ["relu"],
        "model__alpha": np.logspace(-5, -1, 5),
        "model__learning_rate_init": [1e-4],
        "model__solver": ["adam"] # Use ADAM as optimisation algorithm, as it adapts the learning rate
    }
]

# Define metrics we are interested in
scoring = {
    "balanced_accuracy": "balanced_accuracy",
    "accuracy": "accuracy",
    "f1_macro": "f1_macro"
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=gkf,
    scoring=scoring,
    refit="balanced_accuracy",   # Choose best model based on balanced accuracy
    n_jobs=-1,
    verbose=1,
    return_train_score=False
)

grid.fit(X_train, ys_train, groups=period_train)

# =========================================================
# OVERALL BEST RESULT
# =========================================================
print("=" * 60)
print("BEST MODEL")
print("=" * 60)
print("Best params:", grid.best_params_)
print(f"Best mean CV balanced_accuracy: {grid.best_score_:.4f}")

results = pd.DataFrame(grid.cv_results_)
best_idx = grid.best_index_

print("\nMean CV metrics for best model:")
print(f"  balanced_accuracy: {results.loc[best_idx, 'mean_test_balanced_accuracy']:.4f}")
print(f"  accuracy:          {results.loc[best_idx, 'mean_test_accuracy']:.4f}")
print(f"  f1_macro:          {results.loc[best_idx, 'mean_test_f1_macro']:.4f}")

# =========================================================
# PER-FOLD SCORES FOR BEST MODEL
# =========================================================
n_splits = gkf.get_n_splits(X_train, ys_train, groups=period_train)

print("\n" + "=" * 60)
print("PER-FOLD SCORES FOR BEST MODEL")
print("=" * 60)

for i in range(n_splits):
    ba = results.loc[best_idx, f"split{i}_test_balanced_accuracy"]
    acc = results.loc[best_idx, f"split{i}_test_accuracy"]
    f1 = results.loc[best_idx, f"split{i}_test_f1_macro"]

    print(f"Fold {i}:")
    print(f"  balanced_accuracy = {ba:.4f}")
    print(f"  accuracy          = {acc:.4f}")
    print(f"  f1_macro          = {f1:.4f}")


# =========================================================
# WEIGHTED MEAN (based on number of observations per fold)
# =========================================================

# 1) Anzahl Samples pro Fold bestimmen
fold_sizes = []
for _, val_idx in gkf.split(X_train, ys_train, groups=period_train):
    fold_sizes.append(len(val_idx))

fold_sizes = np.array(fold_sizes)

print("\nFold sizes:", fold_sizes)

# 2) Scores des BESTEN MODELLS extrahieren
fold_ba = np.array([
    results.loc[best_idx, f"split{i}_test_balanced_accuracy"]
    for i in range(n_splits)
])

fold_acc = np.array([
    results.loc[best_idx, f"split{i}_test_accuracy"]
    for i in range(n_splits)
])

fold_f1 = np.array([
    results.loc[best_idx, f"split{i}_test_f1_macro"]
    for i in range(n_splits)
])

print("Per-fold balanced_accuracy:", fold_ba)
print("Per-fold accuracy:", fold_acc)
print("Per-fold f1_macro:", fold_f1)

# 3) Weighted Means berechnen
weighted_ba = np.average(fold_ba, weights=fold_sizes)
weighted_acc = np.average(fold_acc, weights=fold_sizes)
weighted_f1 = np.average(fold_f1, weights=fold_sizes)

print("\n" + "=" * 60)
print("WEIGHTED MEAN CV SCORES")
print("=" * 60)
print(f"Weighted balanced_accuracy: {weighted_ba:.4f}")
print(f"Weighted accuracy:          {weighted_acc:.4f}")
print(f"Weighted f1_macro:          {weighted_f1:.4f}")

# =========================================================
# BEST MODEL ON PUB TEST (Classification)
# =========================================================
best_model = grid.best_estimator_

y_pred = best_model.predict(X_pub_test)
cm = confusion_matrix(ys_pub_test, y_pred)

pub_acc = accuracy_score(ys_pub_test, y_pred)
pub_ba = balanced_accuracy_score(ys_pub_test, y_pred)
pub_f1 = f1_score(ys_pub_test, y_pred, average="macro")

print("\n" + "=" * 60)
print("PUBLIC TEST PERFORMANCE")
print("=" * 60)
print(f"Accuracy:          {pub_acc:.4f}")
print(f"Balanced Accuracy: {pub_ba:.4f}")
print(f"F1 Macro:          {pub_f1:.4f}")

print("\nConfusion Matrix:")
print(cm)

Fitting 6 folds for each of 50 candidates, totalling 300 fits
BEST MODEL
Best params: {'model__activation': 'relu', 'model__alpha': np.float64(0.01), 'model__hidden_layer_sizes': (128, 96, 64), 'model__learning_rate_init': 0.0001, 'model__solver': 'adam', 'pca__n_components': 20}
Best mean CV balanced_accuracy: 0.8951

Mean CV metrics for best model:
  balanced_accuracy: 0.8951
  accuracy:          0.9065
  f1_macro:          0.8637

PER-FOLD SCORES FOR BEST MODEL
Fold 0:
  balanced_accuracy = 0.8923
  accuracy          = 0.8730
  f1_macro          = 0.8196
Fold 1:
  balanced_accuracy = 0.9832
  accuracy          = 0.9836
  f1_macro          = 0.9840
Fold 2:
  balanced_accuracy = 0.9608
  accuracy          = 0.9598
  f1_macro          = 0.9380
Fold 3:
  balanced_accuracy = 0.6561
  accuracy          = 0.7506
  f1_macro          = 0.6042
Fold 4:
  balanced_accuracy = 0.9857
  accuracy          = 0.9898
  f1_macro          = 0.9876
Fold 5:
  balanced_accuracy = 0.8925
  accuracy         

# Support Vector Machine - Standardised Data

In [52]:
# =========================================================
# Support Vector Machine - STANDARDISED DATA
# =========================================================

# Define pipeline, that explains how the data is preprocessed and which model is used
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC(kernel='rbf', class_weight='balanced'))
])

# Define parameter grid over which we optimise
param_grid = [
    {
        "model__C": [0.1, 1, 10, 100],
        "model__gamma": ['scale', 'auto', 0.01, 0.1, 1]
    }
]

# Define metrics we want to evaluate
scoring = {
    "balanced_accuracy": "balanced_accuracy",
    "accuracy": "accuracy",
    "f1_macro": "f1_macro"
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=gkf,
    scoring=scoring,
    refit="balanced_accuracy",   # Choose best model based on balanced accuracy
    n_jobs=-1,
    verbose=1,
    return_train_score=False
)

grid.fit(X_train, ys_train, groups=period_train)

# =========================================================
# OVERALL BEST RESULT
# =========================================================
print("=" * 60)
print("BEST MODEL")
print("=" * 60)
print("Best params:", grid.best_params_)
print(f"Best mean CV balanced_accuracy: {grid.best_score_:.4f}")

results = pd.DataFrame(grid.cv_results_)
best_idx = grid.best_index_

print("\nMean CV metrics for best model:")
print(f"  balanced_accuracy: {results.loc[best_idx, 'mean_test_balanced_accuracy']:.4f}")
print(f"  accuracy:          {results.loc[best_idx, 'mean_test_accuracy']:.4f}")
print(f"  f1_macro:          {results.loc[best_idx, 'mean_test_f1_macro']:.4f}")

# =========================================================
# PER-FOLD SCORES FOR BEST MODEL
# =========================================================
n_splits = gkf.get_n_splits(X_train, ys_train, groups=period_train)

print("\n" + "=" * 60)
print("PER-FOLD SCORES FOR BEST MODEL")
print("=" * 60)

for i in range(n_splits):
    ba = results.loc[best_idx, f"split{i}_test_balanced_accuracy"]
    acc = results.loc[best_idx, f"split{i}_test_accuracy"]
    f1 = results.loc[best_idx, f"split{i}_test_f1_macro"]

    print(f"Fold {i}:")
    print(f"  balanced_accuracy = {ba:.4f}")
    print(f"  accuracy          = {acc:.4f}")
    print(f"  f1_macro          = {f1:.4f}")


# =========================================================
# WEIGHTED MEAN (based on fold sizes) FOR BEST MODEL
# =========================================================
fold_sizes = []
for _, val_idx in gkf.split(X_train, ys_train, groups=period_train):
    fold_sizes.append(len(val_idx))
fold_sizes = np.array(fold_sizes)

print("\nFold sizes:", fold_sizes)

fold_ba = np.array([
    results.loc[best_idx, f"split{i}_test_balanced_accuracy"]
    for i in range(n_splits)
])

fold_acc = np.array([
    results.loc[best_idx, f"split{i}_test_accuracy"]
    for i in range(n_splits)
])

fold_f1 = np.array([
    results.loc[best_idx, f"split{i}_test_f1_macro"]
    for i in range(n_splits)
])

weighted_ba = np.average(fold_ba, weights=fold_sizes)
weighted_acc = np.average(fold_acc, weights=fold_sizes)
weighted_f1 = np.average(fold_f1, weights=fold_sizes)

print("\n" + "=" * 60)
print("WEIGHTED MEAN CV SCORES")
print("=" * 60)
print(f"Weighted balanced_accuracy: {weighted_ba:.4f}")
print(f"Weighted accuracy:          {weighted_acc:.4f}")
print(f"Weighted f1_macro:          {weighted_f1:.4f}")


# =========================================================
# BEST MODEL ON PUB TEST (Classification)
# =========================================================
best_model = grid.best_estimator_

y_pred = best_model.predict(X_pub_test)
cm = confusion_matrix(ys_pub_test, y_pred)

pub_acc = accuracy_score(ys_pub_test, y_pred)
pub_ba = balanced_accuracy_score(ys_pub_test, y_pred)
pub_f1 = f1_score(ys_pub_test, y_pred, average="macro")

print("\n" + "=" * 60)
print("PUBLIC TEST PERFORMANCE")
print("=" * 60)
print(f"Accuracy:          {pub_acc:.4f}")
print(f"Balanced Accuracy: {pub_ba:.4f}")
print(f"F1 Macro:          {pub_f1:.4f}")

print("\nConfusion Matrix:")
print(cm)

Fitting 6 folds for each of 20 candidates, totalling 120 fits
BEST MODEL
Best params: {'model__C': 10, 'model__gamma': 'scale'}
Best mean CV balanced_accuracy: 0.8537

Mean CV metrics for best model:
  balanced_accuracy: 0.8537
  accuracy:          0.8549
  f1_macro:          0.8039

PER-FOLD SCORES FOR BEST MODEL
Fold 0:
  balanced_accuracy = 0.7814
  accuracy          = 0.7374
  f1_macro          = 0.6361
Fold 1:
  balanced_accuracy = 0.9718
  accuracy          = 0.9729
  f1_macro          = 0.8099
Fold 2:
  balanced_accuracy = 0.9826
  accuracy          = 0.9791
  f1_macro          = 0.9654
Fold 3:
  balanced_accuracy = 0.4990
  accuracy          = 0.5596
  f1_macro          = 0.5168
Fold 4:
  balanced_accuracy = 0.9805
  accuracy          = 0.9797
  f1_macro          = 0.9783
Fold 5:
  balanced_accuracy = 0.9070
  accuracy          = 0.9006
  f1_macro          = 0.9171

Fold sizes: [2300 1586 1244  445  197  161]

WEIGHTED MEAN CV SCORES
Weighted balanced_accuracy: 0.8633
Weighted 

In [53]:
# =========================================================
# Support Vector Machine - STANDARDISED DATA -> PCA
# =========================================================

# Define the pipeline of data preprocessing and model implementation
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA()),
    ("model", SVC(kernel='rbf', class_weight='balanced'))
])

# Define parameter grid over which we optimise
param_grid = [
    {
        "pca__n_components": [5, 10, 20, 50, 75],
        "model__C": [0.1, 1, 10, 100],
        "model__gamma": ['scale', 'auto', 0.01, 0.1]
    }
]

# Define metrics we want to evaluate
scoring = {
    "balanced_accuracy": "balanced_accuracy",
    "accuracy": "accuracy",
    "f1_macro": "f1_macro"
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=gkf,
    scoring=scoring,
    refit="balanced_accuracy",   # Choose best model based on balanced accuracy
    n_jobs=-1,
    verbose=1,
    return_train_score=False
)

grid.fit(X_train, ys_train, groups=period_train)

# =========================================================
# OVERALL BEST RESULT
# =========================================================
print("=" * 60)
print("BEST MODEL")
print("=" * 60)
print("Best params:", grid.best_params_)
print(f"Best mean CV balanced_accuracy: {grid.best_score_:.4f}")

results = pd.DataFrame(grid.cv_results_)
best_idx = grid.best_index_

print("\nMean CV metrics for best model:")
print(f"  balanced_accuracy: {results.loc[best_idx, 'mean_test_balanced_accuracy']:.4f}")
print(f"  accuracy:          {results.loc[best_idx, 'mean_test_accuracy']:.4f}")
print(f"  f1_macro:          {results.loc[best_idx, 'mean_test_f1_macro']:.4f}")

# =========================================================
# PER-FOLD SCORES FOR BEST MODEL
# =========================================================
n_splits = gkf.get_n_splits(X_train, ys_train, groups=period_train)

print("\n" + "=" * 60)
print("PER-FOLD SCORES FOR BEST MODEL")
print("=" * 60)

for i in range(n_splits):
    ba = results.loc[best_idx, f"split{i}_test_balanced_accuracy"]
    acc = results.loc[best_idx, f"split{i}_test_accuracy"]
    f1 = results.loc[best_idx, f"split{i}_test_f1_macro"]

    print(f"Fold {i}:")
    print(f"  balanced_accuracy = {ba:.4f}")
    print(f"  accuracy          = {acc:.4f}")
    print(f"  f1_macro          = {f1:.4f}")


# =========================================================
# WEIGHTED MEAN (based on fold sizes) FOR BEST MODEL
# =========================================================
fold_sizes = []
for _, val_idx in gkf.split(X_train, ys_train, groups=period_train):
    fold_sizes.append(len(val_idx))
fold_sizes = np.array(fold_sizes)

print("\nFold sizes:", fold_sizes)

fold_ba = np.array([
    results.loc[best_idx, f"split{i}_test_balanced_accuracy"]
    for i in range(n_splits)
])

fold_acc = np.array([
    results.loc[best_idx, f"split{i}_test_accuracy"]
    for i in range(n_splits)
])

fold_f1 = np.array([
    results.loc[best_idx, f"split{i}_test_f1_macro"]
    for i in range(n_splits)
])

weighted_ba = np.average(fold_ba, weights=fold_sizes)
weighted_acc = np.average(fold_acc, weights=fold_sizes)
weighted_f1 = np.average(fold_f1, weights=fold_sizes)

print("\n" + "=" * 60)
print("WEIGHTED MEAN CV SCORES")
print("=" * 60)
print(f"Weighted balanced_accuracy: {weighted_ba:.4f}")
print(f"Weighted accuracy:          {weighted_acc:.4f}")
print(f"Weighted f1_macro:          {weighted_f1:.4f}")


# =========================================================
# BEST MODEL ON PUB TEST (Classification)
# =========================================================
best_model = grid.best_estimator_

y_pred = best_model.predict(X_pub_test)
cm = confusion_matrix(ys_pub_test, y_pred)

pub_acc = accuracy_score(ys_pub_test, y_pred)
pub_ba = balanced_accuracy_score(ys_pub_test, y_pred)
pub_f1 = f1_score(ys_pub_test, y_pred, average="macro")

print("\n" + "=" * 60)
print("PUBLIC TEST PERFORMANCE")
print("=" * 60)
print(f"Accuracy:          {pub_acc:.4f}")
print(f"Balanced Accuracy: {pub_ba:.4f}")
print(f"F1 Macro:          {pub_f1:.4f}")

print("\nConfusion Matrix:")
print(cm)

Fitting 6 folds for each of 80 candidates, totalling 480 fits
BEST MODEL
Best params: {'model__C': 10, 'model__gamma': 'scale', 'pca__n_components': 50}
Best mean CV balanced_accuracy: 0.8537

Mean CV metrics for best model:
  balanced_accuracy: 0.8537
  accuracy:          0.8549
  f1_macro:          0.8038

PER-FOLD SCORES FOR BEST MODEL
Fold 0:
  balanced_accuracy = 0.7813
  accuracy          = 0.7374
  f1_macro          = 0.6361
Fold 1:
  balanced_accuracy = 0.9734
  accuracy          = 0.9748
  f1_macro          = 0.8116
Fold 2:
  balanced_accuracy = 0.9831
  accuracy          = 0.9799
  f1_macro          = 0.9662
Fold 3:
  balanced_accuracy = 0.4968
  accuracy          = 0.5573
  f1_macro          = 0.5138
Fold 4:
  balanced_accuracy = 0.9805
  accuracy          = 0.9797
  f1_macro          = 0.9783
Fold 5:
  balanced_accuracy = 0.9070
  accuracy          = 0.9006
  f1_macro          = 0.9171

Fold sizes: [2300 1586 1244  445  197  161]

WEIGHTED MEAN CV SCORES
Weighted balanced_a

# Naive Bayes - Standardised Data:

In [54]:
# =========================================================
# Naive Bayes - STANDARDISED DATA
# =========================================================

# Define the pipeline of how to preprocess the data and which model to implement
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", GaussianNB())
])

# Define the parameter grid over which we optimise
param_grid = [
    {
        "model__var_smoothing": np.logspace(-9, 0, 10)
    }
]


# Define metrics we want to evaluate
scoring = {
    "balanced_accuracy": "balanced_accuracy",
    "accuracy": "accuracy",
    "f1_macro": "f1_macro"
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=gkf,
    scoring=scoring,
    refit="balanced_accuracy",   # Choose best model based on balanced accuracy
    n_jobs=-1,
    verbose=1,
    return_train_score=False
)

grid.fit(X_train, ys_train, groups=period_train)

# =========================================================
# OVERALL BEST RESULT
# =========================================================
print("=" * 60)
print("BEST MODEL")
print("=" * 60)
print("Best params:", grid.best_params_)
print(f"Best mean CV balanced_accuracy: {grid.best_score_:.4f}")

results = pd.DataFrame(grid.cv_results_)
best_idx = grid.best_index_

print("\nMean CV metrics for best model:")
print(f"  balanced_accuracy: {results.loc[best_idx, 'mean_test_balanced_accuracy']:.4f}")
print(f"  accuracy:          {results.loc[best_idx, 'mean_test_accuracy']:.4f}")
print(f"  f1_macro:          {results.loc[best_idx, 'mean_test_f1_macro']:.4f}")

# =========================================================
# PER-FOLD SCORES FOR BEST MODEL
# =========================================================
n_splits = gkf.get_n_splits(X_train, ys_train, groups=period_train)

print("\n" + "=" * 60)
print("PER-FOLD SCORES FOR BEST MODEL")
print("=" * 60)

for i in range(n_splits):
    ba = results.loc[best_idx, f"split{i}_test_balanced_accuracy"]
    acc = results.loc[best_idx, f"split{i}_test_accuracy"]
    f1 = results.loc[best_idx, f"split{i}_test_f1_macro"]

    print(f"Fold {i}:")
    print(f"  balanced_accuracy = {ba:.4f}")
    print(f"  accuracy          = {acc:.4f}")
    print(f"  f1_macro          = {f1:.4f}")


# =========================================================
# WEIGHTED MEAN (based on fold sizes) FOR BEST MODEL
# =========================================================
fold_sizes = []
for _, val_idx in gkf.split(X_train, ys_train, groups=period_train):
    fold_sizes.append(len(val_idx))
fold_sizes = np.array(fold_sizes)

print("\nFold sizes:", fold_sizes)

fold_ba = np.array([
    results.loc[best_idx, f"split{i}_test_balanced_accuracy"]
    for i in range(n_splits)
])

fold_acc = np.array([
    results.loc[best_idx, f"split{i}_test_accuracy"]
    for i in range(n_splits)
])

fold_f1 = np.array([
    results.loc[best_idx, f"split{i}_test_f1_macro"]
    for i in range(n_splits)
])

weighted_ba = np.average(fold_ba, weights=fold_sizes)
weighted_acc = np.average(fold_acc, weights=fold_sizes)
weighted_f1 = np.average(fold_f1, weights=fold_sizes)

print("\n" + "=" * 60)
print("WEIGHTED MEAN CV SCORES")
print("=" * 60)
print(f"Weighted balanced_accuracy: {weighted_ba:.4f}")
print(f"Weighted accuracy:          {weighted_acc:.4f}")
print(f"Weighted f1_macro:          {weighted_f1:.4f}")


# =========================================================
# BEST MODEL ON PUB TEST (Classification)
# =========================================================
best_model = grid.best_estimator_

y_pred = best_model.predict(X_pub_test)
cm = confusion_matrix(ys_pub_test, y_pred)

pub_acc = accuracy_score(ys_pub_test, y_pred)
pub_ba = balanced_accuracy_score(ys_pub_test, y_pred)
pub_f1 = f1_score(ys_pub_test, y_pred, average="macro")

print("\n" + "=" * 60)
print("PUBLIC TEST PERFORMANCE")
print("=" * 60)
print(f"Accuracy:          {pub_acc:.4f}")
print(f"Balanced Accuracy: {pub_ba:.4f}")
print(f"F1 Macro:          {pub_f1:.4f}")

print("\nConfusion Matrix:")
print(cm)

Fitting 6 folds for each of 10 candidates, totalling 60 fits
BEST MODEL
Best params: {'model__var_smoothing': np.float64(1e-06)}
Best mean CV balanced_accuracy: 0.6931

Mean CV metrics for best model:
  balanced_accuracy: 0.6931
  accuracy:          0.6652
  f1_macro:          0.6050

PER-FOLD SCORES FOR BEST MODEL
Fold 0:
  balanced_accuracy = 0.5256
  accuracy          = 0.3700
  f1_macro          = 0.3380
Fold 1:
  balanced_accuracy = 0.7573
  accuracy          = 0.7654
  f1_macro          = 0.6582
Fold 2:
  balanced_accuracy = 0.7365
  accuracy          = 0.7291
  f1_macro          = 0.5947
Fold 3:
  balanced_accuracy = 0.4074
  accuracy          = 0.4517
  f1_macro          = 0.3469
Fold 4:
  balanced_accuracy = 0.8715
  accuracy          = 0.8426
  f1_macro          = 0.8550
Fold 5:
  balanced_accuracy = 0.8604
  accuracy          = 0.8323
  f1_macro          = 0.8375

Fold sizes: [2300 1586 1244  445  197  161]

WEIGHTED MEAN CV SCORES
Weighted balanced_accuracy: 0.6435
Weighted

# AdaBoost - Standardised Data:

In [55]:
# =========================================================
# AdaBoost - STANDARDISED DATA
# =========================================================

# We use a shallow tree (stump) to keep computation time low
base_estimator = DecisionTreeClassifier(max_depth=3, class_weight='balanced')

# Define pipeline of how we preprocess the data and which model to implement
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", AdaBoostClassifier(estimator=base_estimator,
                                 random_state=seed)) # Set seed for reproducibility
])

# Define the parameter grid over which we optimise
param_grid = [
    {
        "model__n_estimators": [50, 100],
        "model__learning_rate": [0.01, 0.1, 1.0]
    }
]

# Define metrics we want to evaluate
scoring = {
    "balanced_accuracy": "balanced_accuracy",
    "accuracy": "accuracy",
    "f1_macro": "f1_macro"
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=gkf,
    scoring=scoring,
    refit="balanced_accuracy",   # Choose best model based on balanced accuracy
    n_jobs=-1,
    verbose=1,
    return_train_score=False
)

grid.fit(X_train, ys_train, groups=period_train)

# =========================================================
# OVERALL BEST RESULT
# =========================================================
print("=" * 60)
print("BEST MODEL")
print("=" * 60)
print("Best params:", grid.best_params_)
print(f"Best mean CV balanced_accuracy: {grid.best_score_:.4f}")

results = pd.DataFrame(grid.cv_results_)
best_idx = grid.best_index_

print("\nMean CV metrics for best model:")
print(f"  balanced_accuracy: {results.loc[best_idx, 'mean_test_balanced_accuracy']:.4f}")
print(f"  accuracy:          {results.loc[best_idx, 'mean_test_accuracy']:.4f}")
print(f"  f1_macro:          {results.loc[best_idx, 'mean_test_f1_macro']:.4f}")

# =========================================================
# PER-FOLD SCORES FOR BEST MODEL
# =========================================================
n_splits = gkf.get_n_splits(X_train, ys_train, groups=period_train)

print("\n" + "=" * 60)
print("PER-FOLD SCORES FOR BEST MODEL")
print("=" * 60)

for i in range(n_splits):
    ba = results.loc[best_idx, f"split{i}_test_balanced_accuracy"]
    acc = results.loc[best_idx, f"split{i}_test_accuracy"]
    f1 = results.loc[best_idx, f"split{i}_test_f1_macro"]

    print(f"Fold {i}:")
    print(f"  balanced_accuracy = {ba:.4f}")
    print(f"  accuracy          = {acc:.4f}")
    print(f"  f1_macro          = {f1:.4f}")


# =========================================================
# WEIGHTED MEAN (based on fold sizes) FOR BEST MODEL
# =========================================================
fold_sizes = []
for _, val_idx in gkf.split(X_train, ys_train, groups=period_train):
    fold_sizes.append(len(val_idx))
fold_sizes = np.array(fold_sizes)

print("\nFold sizes:", fold_sizes)

fold_ba = np.array([
    results.loc[best_idx, f"split{i}_test_balanced_accuracy"]
    for i in range(n_splits)
])

fold_acc = np.array([
    results.loc[best_idx, f"split{i}_test_accuracy"]
    for i in range(n_splits)
])

fold_f1 = np.array([
    results.loc[best_idx, f"split{i}_test_f1_macro"]
    for i in range(n_splits)
])

weighted_ba = np.average(fold_ba, weights=fold_sizes)
weighted_acc = np.average(fold_acc, weights=fold_sizes)
weighted_f1 = np.average(fold_f1, weights=fold_sizes)

print("\n" + "=" * 60)
print("WEIGHTED MEAN CV SCORES")
print("=" * 60)
print(f"Weighted balanced_accuracy: {weighted_ba:.4f}")
print(f"Weighted accuracy:          {weighted_acc:.4f}")
print(f"Weighted f1_macro:          {weighted_f1:.4f}")


# =========================================================
# BEST MODEL ON PUB TEST (Classification)
# =========================================================
best_model = grid.best_estimator_

y_pred = best_model.predict(X_pub_test)
cm = confusion_matrix(ys_pub_test, y_pred)

pub_acc = accuracy_score(ys_pub_test, y_pred)
pub_ba = balanced_accuracy_score(ys_pub_test, y_pred)
pub_f1 = f1_score(ys_pub_test, y_pred, average="macro")

print("\n" + "=" * 60)
print("PUBLIC TEST PERFORMANCE")
print("=" * 60)
print(f"Accuracy:          {pub_acc:.4f}")
print(f"Balanced Accuracy: {pub_ba:.4f}")
print(f"F1 Macro:          {pub_f1:.4f}")

print("\nConfusion Matrix:")
print(cm)

Fitting 6 folds for each of 6 candidates, totalling 36 fits
BEST MODEL
Best params: {'model__learning_rate': 1.0, 'model__n_estimators': 100}
Best mean CV balanced_accuracy: 0.8268

Mean CV metrics for best model:
  balanced_accuracy: 0.8268
  accuracy:          0.8326
  f1_macro:          0.7884

PER-FOLD SCORES FOR BEST MODEL
Fold 0:
  balanced_accuracy = 0.6653
  accuracy          = 0.5691
  f1_macro          = 0.5474
Fold 1:
  balanced_accuracy = 0.9022
  accuracy          = 0.9174
  f1_macro          = 0.9080
Fold 2:
  balanced_accuracy = 0.8834
  accuracy          = 0.9277
  f1_macro          = 0.8540
Fold 3:
  balanced_accuracy = 0.6543
  accuracy          = 0.7416
  f1_macro          = 0.6226
Fold 4:
  balanced_accuracy = 0.9412
  accuracy          = 0.9391
  f1_macro          = 0.9355
Fold 5:
  balanced_accuracy = 0.9146
  accuracy          = 0.9006
  f1_macro          = 0.8628

Fold sizes: [2300 1586 1244  445  197  161]

WEIGHTED MEAN CV SCORES
Weighted balanced_accuracy: 0.

# AdaBoost - Standardised Data -> LDA

In [56]:
# =========================================================
# AdaBoost - STANDARDISED DATA -> LDA (Linear Discriminant Coordinates)
# =========================================================

# We use a shallow tree (stump) to keep computation time low
base_estimator = DecisionTreeClassifier(max_depth=2, class_weight='balanced')

# Define the pipeline of how to preprocess the data and which model to implement
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("lda", LinearDiscriminantAnalysis(n_components=5)),
    ("model", AdaBoostClassifier(estimator=base_estimator,
                                 random_state=seed)) # Set Seed for reproducibility
])

# Define the parameter grid over which we optimise
param_grid = [
    {
        "model__n_estimators": [50, 100, 200],
        "model__learning_rate": [0.01, 0.1, 1.0]
    }
]

# Define metrics we want to evaluate
scoring = {
    "balanced_accuracy": "balanced_accuracy",
    "accuracy": "accuracy",
    "f1_macro": "f1_macro"
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=gkf,
    scoring=scoring,
    refit="balanced_accuracy",   # Choose best model based on balanced accuracy
    n_jobs=-1,
    verbose=4,
    return_train_score=False
)

grid.fit(X_train, ys_train, groups=period_train)

# =========================================================
# OVERALL BEST RESULT
# =========================================================
print("=" * 60)
print("BEST MODEL")
print("=" * 60)
print("Best params:", grid.best_params_)
print(f"Best mean CV balanced_accuracy: {grid.best_score_:.4f}")

results = pd.DataFrame(grid.cv_results_)
best_idx = grid.best_index_

print("\nMean CV metrics for best model:")
print(f"  balanced_accuracy: {results.loc[best_idx, 'mean_test_balanced_accuracy']:.4f}")
print(f"  accuracy:          {results.loc[best_idx, 'mean_test_accuracy']:.4f}")
print(f"  f1_macro:          {results.loc[best_idx, 'mean_test_f1_macro']:.4f}")

# =========================================================
# PER-FOLD SCORES FOR BEST MODEL
# =========================================================
n_splits = gkf.get_n_splits(X_train, ys_train, groups=period_train)

print("\n" + "=" * 60)
print("PER-FOLD SCORES FOR BEST MODEL")
print("=" * 60)

for i in range(n_splits):
    ba = results.loc[best_idx, f"split{i}_test_balanced_accuracy"]
    acc = results.loc[best_idx, f"split{i}_test_accuracy"]
    f1 = results.loc[best_idx, f"split{i}_test_f1_macro"]

    print(f"Fold {i}:")
    print(f"  balanced_accuracy = {ba:.4f}")
    print(f"  accuracy          = {acc:.4f}")
    print(f"  f1_macro          = {f1:.4f}")


# =========================================================
# WEIGHTED MEAN (based on fold sizes) FOR BEST MODEL
# =========================================================
fold_sizes = []
for _, val_idx in gkf.split(X_train, ys_train, groups=period_train):
    fold_sizes.append(len(val_idx))
fold_sizes = np.array(fold_sizes)

print("\nFold sizes:", fold_sizes)

fold_ba = np.array([
    results.loc[best_idx, f"split{i}_test_balanced_accuracy"]
    for i in range(n_splits)
])

fold_acc = np.array([
    results.loc[best_idx, f"split{i}_test_accuracy"]
    for i in range(n_splits)
])

fold_f1 = np.array([
    results.loc[best_idx, f"split{i}_test_f1_macro"]
    for i in range(n_splits)
])

weighted_ba = np.average(fold_ba, weights=fold_sizes)
weighted_acc = np.average(fold_acc, weights=fold_sizes)
weighted_f1 = np.average(fold_f1, weights=fold_sizes)

print("\n" + "=" * 60)
print("WEIGHTED MEAN CV SCORES")
print("=" * 60)
print(f"Weighted balanced_accuracy: {weighted_ba:.4f}")
print(f"Weighted accuracy:          {weighted_acc:.4f}")
print(f"Weighted f1_macro:          {weighted_f1:.4f}")


# =========================================================
# BEST MODEL ON PUB TEST (Classification)
# =========================================================
best_model = grid.best_estimator_

y_pred = best_model.predict(X_pub_test)
cm = confusion_matrix(ys_pub_test, y_pred)

pub_acc = accuracy_score(ys_pub_test, y_pred)
pub_ba = balanced_accuracy_score(ys_pub_test, y_pred)
pub_f1 = f1_score(ys_pub_test, y_pred, average="macro")

print("\n" + "=" * 60)
print("PUBLIC TEST PERFORMANCE")
print("=" * 60)
print(f"Accuracy:          {pub_acc:.4f}")
print(f"Balanced Accuracy: {pub_ba:.4f}")
print(f"F1 Macro:          {pub_f1:.4f}")

print("\nConfusion Matrix:")
print(cm)

Fitting 6 folds for each of 9 candidates, totalling 54 fits
BEST MODEL
Best params: {'model__learning_rate': 1.0, 'model__n_estimators': 200}
Best mean CV balanced_accuracy: 0.8143

Mean CV metrics for best model:
  balanced_accuracy: 0.8143
  accuracy:          0.8104
  f1_macro:          0.7326

PER-FOLD SCORES FOR BEST MODEL
Fold 0:
  balanced_accuracy = 0.7433
  accuracy          = 0.6883
  f1_macro          = 0.6025
Fold 1:
  balanced_accuracy = 0.9834
  accuracy          = 0.9849
  f1_macro          = 0.8191
Fold 2:
  balanced_accuracy = 0.9592
  accuracy          = 0.9518
  f1_macro          = 0.8798
Fold 3:
  balanced_accuracy = 0.3010
  accuracy          = 0.3573
  f1_macro          = 0.2272
Fold 4:
  balanced_accuracy = 0.9793
  accuracy          = 0.9797
  f1_macro          = 0.9773
Fold 5:
  balanced_accuracy = 0.9196
  accuracy          = 0.9006
  f1_macro          = 0.8898

Fold sizes: [2300 1586 1244  445  197  161]

WEIGHTED MEAN CV SCORES
Weighted balanced_accuracy: 0.

# Substance Agnostic Regression Models:

# OLS Regression - Standardised Data:

In [57]:
# =========================================================
# OLS - STANDARDISED DATA
# =========================================================

# Define the pipeline of how to preprocess the data and which model to use
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

param_grid = [
    {} # OLS has no parameter to hypertune
]

# Define Regression Metrics we want to investigate 
scoring = {
    "mse": "neg_mean_squared_error",
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error", # Our final chosen metric
    "r2": "r2"
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=gkf,
    scoring=scoring,
    refit="mae", # Find the best model in the sense that it minimises MAE
    n_jobs=-1,
    verbose=1,
    return_train_score=False
)

grid.fit(X_train, yc_train, groups=period_train)

results = pd.DataFrame(grid.cv_results_)
best_idx = grid.best_index_

# =========================================================
# OVERALL BEST RESULT
# =========================================================
print("=" * 60)
print("BEST MODEL")
print("=" * 60)
print("Best params:", grid.best_params_)

# Not we get the negative score as we used neg_... <- thus multiply with (-1)
print(f"Best mean CV MAE: {-grid.best_score_:.4f}")

print("\nMean CV metrics for best model:")
print(f"  MSE : {-results.loc[best_idx, 'mean_test_mse']:.4f}")
print(f"  RMSE: {-results.loc[best_idx, 'mean_test_rmse']:.4f}")
print(f"  MAE : {-results.loc[best_idx, 'mean_test_mae']:.4f}")
print(f"  R2  : {results.loc[best_idx, 'mean_test_r2']:.4f}")

# =========================================================
# PER-FOLD SCORES FOR BEST MODEL
# =========================================================
n_splits = gkf.get_n_splits(X_train, yc_train, groups=period_train)

print("\n" + "=" * 60)
print("PER-FOLD SCORES FOR BEST MODEL")
print("=" * 60)

for i in range(n_splits):
    mse = -results.loc[best_idx, f"split{i}_test_mse"]
    rmse = -results.loc[best_idx, f"split{i}_test_rmse"]
    mae = -results.loc[best_idx, f"split{i}_test_mae"]
    r2 = results.loc[best_idx, f"split{i}_test_r2"]

    print(f"Fold {i}:")
    print(f"  MSE  = {mse:.4f}")
    print(f"  RMSE = {rmse:.4f}")
    print(f"  MAE  = {mae:.4f}")
    print(f"  R2   = {r2:.4f}")

# =========================================================
# WEIGHTED MEAN (based on number of observations per fold)
# =========================================================

# 1) Find the number of observations for each validation set
fold_sizes = []
for _, val_idx in gkf.split(X_train, yc_train, groups=period_train):
    fold_sizes.append(len(val_idx))

fold_sizes = np.array(fold_sizes)

print("\nFold sizes:", fold_sizes)

# 2) Get the results of the best model
fold_mse = np.array([
    -results.loc[best_idx, f"split{i}_test_mse"]
    for i in range(n_splits)
])

fold_rmse = np.array([
    -results.loc[best_idx, f"split{i}_test_rmse"]
    for i in range(n_splits)
])

fold_mae = np.array([
    -results.loc[best_idx, f"split{i}_test_mae"]
    for i in range(n_splits)
])

fold_r2 = np.array([
    results.loc[best_idx, f"split{i}_test_r2"]
    for i in range(n_splits)
])

print("Per-fold MSE :", fold_mse)
print("Per-fold RMSE:", fold_rmse)
print("Per-fold MAE :", fold_mae)
print("Per-fold R2  :", fold_r2)

# 3) Compute weighted means of the metrics
weighted_mse = np.average(fold_mse, weights=fold_sizes)
weighted_rmse = np.average(fold_rmse, weights=fold_sizes)
weighted_mae = np.average(fold_mae, weights=fold_sizes)
weighted_r2 = np.average(fold_r2, weights=fold_sizes)

print("\n" + "=" * 60)
print("WEIGHTED MEAN CV SCORES")
print("=" * 60)
print(f"Weighted MSE : {weighted_mse:.4f}")
print(f"Weighted RMSE: {weighted_rmse:.4f}")
print(f"Weighted MAE : {weighted_mae:.4f}")
print(f"Weighted R2  : {weighted_r2:.4f}")

# =========================================================
# PUBLIC TEST SET EVALUATION (REGRESSION)
# =========================================================
best_reg_model = grid.best_estimator_

y_pred_pub = best_reg_model.predict(X_pub_test)

pub_mse  = mean_squared_error(yc_pub_test, y_pred_pub)
pub_rmse = np.sqrt(pub_mse)
pub_mae  = mean_absolute_error(yc_pub_test, y_pred_pub)
pub_r2   = r2_score(yc_pub_test, y_pred_pub)

print("=" * 60)
print("PUBLIC TEST SET RESULTS (REGRESSION)")
print("=" * 60)
print(f"  MSE : {pub_mse:.4f}")
print(f"  RMSE: {pub_rmse:.4f}")
print(f"  MAE : {pub_mae:.4f}")
print(f"  R2  : {pub_r2:.4f}")

Fitting 6 folds for each of 1 candidates, totalling 6 fits
BEST MODEL
Best params: {}
Best mean CV MAE: 128.1233

Mean CV metrics for best model:
  MSE : 36361845.3228
  RMSE: 2523.7323
  MAE : 128.1233
  R2  : -3118.1679

PER-FOLD SCORES FOR BEST MODEL
Fold 0:
  MSE  = 2325.4239
  RMSE = 48.2226
  MAE  = 35.6142
  R2   = 0.5336
Fold 1:
  MSE  = 1111.8322
  RMSE = 33.3441
  MAE  = 24.1823
  R2   = 0.8085
Fold 2:
  MSE  = 218130765.5904
  RMSE = 14769.2507
  MAE  = 522.1680
  R2   = -18709.6236
Fold 3:
  MSE  = 18312.4346
  RMSE = 135.3234
  MAE  = 94.1667
  R2   = 0.6348
Fold 4:
  MSE  = 474.5020
  RMSE = 21.7831
  MAE  = 16.3018
  R2   = 0.9062
Fold 5:
  MSE  = 18082.1534
  RMSE = 134.4699
  MAE  = 76.3069
  R2   = -2.2667

Fold sizes: [2300 1586 1244  445  197  161]
Per-fold MSE : [2.32542394e+03 1.11183216e+03 2.18130766e+08 1.83124346e+04
 4.74502000e+02 1.80821534e+04]
Per-fold RMSE: [   48.22264962    33.34414728 14769.25067803   135.32344456
    21.78306683   134.46989783]
Per-f

# OLS Regression - Standardised Data -> PCA:

In [58]:
# =========================================================
# OLS - STANDARDISED DATA -> PCA
# =========================================================

# Define the pipeline for data preprocessing and which model training
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA()),
    ("model", LinearRegression())
])

# Define parameter grid over which we optimise
param_grid = [
    {'pca__n_components': [5, 10, 20, 50, 75]} # As there are no Hyperparameters for OLS we only need to optimise over the number of PCs
]

# Define Regression Metrics
scoring = {
    "mse": "neg_mean_squared_error",
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2"
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=gkf,
    scoring=scoring,
    refit="mae", # Make final decision based on the model that optimises MAE
    n_jobs=-1,
    verbose=1,
    return_train_score=False
)

grid.fit(X_train, yc_train, groups=period_train)

results = pd.DataFrame(grid.cv_results_)
best_idx = grid.best_index_

# =========================================================
# OVERALL BEST RESULT
# =========================================================
print("=" * 60)
print("BEST MODEL")
print("=" * 60)
print("Best params:", grid.best_params_)

# Multiply by (-1) as we used neg_...
print(f"Best mean CV MAE: {-grid.best_score_:.4f}")

print("\nMean CV metrics for best model:")
print(f"  MSE : {-results.loc[best_idx, 'mean_test_mse']:.4f}")
print(f"  RMSE: {-results.loc[best_idx, 'mean_test_rmse']:.4f}")
print(f"  MAE : {-results.loc[best_idx, 'mean_test_mae']:.4f}")
print(f"  R2  : {results.loc[best_idx, 'mean_test_r2']:.4f}")

# =========================================================
# PER-FOLD SCORES FOR BEST MODEL
# =========================================================
n_splits = gkf.get_n_splits(X_train, yc_train, groups=period_train)

print("\n" + "=" * 60)
print("PER-FOLD SCORES FOR BEST MODEL")
print("=" * 60)

for i in range(n_splits):
    mse = -results.loc[best_idx, f"split{i}_test_mse"]
    rmse = -results.loc[best_idx, f"split{i}_test_rmse"]
    mae = -results.loc[best_idx, f"split{i}_test_mae"]
    r2 = results.loc[best_idx, f"split{i}_test_r2"]

    print(f"Fold {i}:")
    print(f"  MSE  = {mse:.4f}")
    print(f"  RMSE = {rmse:.4f}")
    print(f"  MAE  = {mae:.4f}")
    print(f"  R2   = {r2:.4f}")

# =========================================================
# WEIGHTED MEAN (based on number of observations per fold)
# =========================================================

# 1) Find the number of observations per validation set
fold_sizes = []
for _, val_idx in gkf.split(X_train, yc_train, groups=period_train):
    fold_sizes.append(len(val_idx))

fold_sizes = np.array(fold_sizes)

print("\nFold sizes:", fold_sizes)

# 2) Get the results of the best model
fold_mse = np.array([
    -results.loc[best_idx, f"split{i}_test_mse"]
    for i in range(n_splits)
])

fold_rmse = np.array([
    -results.loc[best_idx, f"split{i}_test_rmse"]
    for i in range(n_splits)
])

fold_mae = np.array([
    -results.loc[best_idx, f"split{i}_test_mae"]
    for i in range(n_splits)
])

fold_r2 = np.array([
    results.loc[best_idx, f"split{i}_test_r2"]
    for i in range(n_splits)
])

print("Per-fold MSE :", fold_mse)
print("Per-fold RMSE:", fold_rmse)
print("Per-fold MAE :", fold_mae)
print("Per-fold R2  :", fold_r2)

# 3) Compute weighted means of metrics
weighted_mse = np.average(fold_mse, weights=fold_sizes)
weighted_rmse = np.average(fold_rmse, weights=fold_sizes)
weighted_mae = np.average(fold_mae, weights=fold_sizes)
weighted_r2 = np.average(fold_r2, weights=fold_sizes)

print("\n" + "=" * 60)
print("WEIGHTED MEAN CV SCORES")
print("=" * 60)
print(f"Weighted MSE : {weighted_mse:.4f}")
print(f"Weighted RMSE: {weighted_rmse:.4f}")
print(f"Weighted MAE : {weighted_mae:.4f}")
print(f"Weighted R2  : {weighted_r2:.4f}")


# =========================================================
# PUBLIC TEST SET EVALUATION (REGRESSION)
# =========================================================
best_reg_model = grid.best_estimator_

y_pred_pub = best_reg_model.predict(X_pub_test)

pub_mse  = mean_squared_error(yc_pub_test, y_pred_pub)
pub_rmse = np.sqrt(pub_mse)
pub_mae  = mean_absolute_error(yc_pub_test, y_pred_pub)
pub_r2   = r2_score(yc_pub_test, y_pred_pub)

print("=" * 60)
print("PUBLIC TEST SET RESULTS (REGRESSION)")
print("=" * 60)
print(f"  MSE : {pub_mse:.4f}")
print(f"  RMSE: {pub_rmse:.4f}")
print(f"  MAE : {pub_mae:.4f}")
print(f"  R2  : {pub_r2:.4f}")

Fitting 6 folds for each of 5 candidates, totalling 30 fits
BEST MODEL
Best params: {'pca__n_components': 20}
Best mean CV MAE: 62.0106

Mean CV metrics for best model:
  MSE : 339930.0407
  RMSE: 297.8581
  MAE : 62.0106
  R2  : -28.1898

PER-FOLD SCORES FOR BEST MODEL
Fold 0:
  MSE  = 955.7092
  RMSE = 30.9145
  MAE  = 20.9973
  R2   = 0.8083
Fold 1:
  MSE  = 1834.4738
  RMSE = 42.8308
  MAE  = 32.4602
  R2   = 0.6840
Fold 2:
  MSE  = 2003721.6991
  RMSE = 1415.5288
  MAE  = 123.0186
  R2   = -170.8734
Fold 3:
  MSE  = 20386.5478
  RMSE = 142.7815
  MAE  = 86.5166
  R2   = 0.5935
Fold 4:
  MSE  = 3534.4090
  RMSE = 59.4509
  MAE  = 34.8925
  R2   = 0.3014
Fold 5:
  MSE  = 9147.4054
  RMSE = 95.6421
  MAE  = 74.1784
  R2   = -0.6526

Fold sizes: [2300 1586 1244  445  197  161]
Per-fold MSE : [9.55709216e+02 1.83447383e+03 2.00372170e+06 2.03865478e+04
 3.53440898e+03 9.14740538e+03]
Per-fold RMSE: [  30.914547     42.83075802 1415.52877014  142.78146851   59.45089556
   95.6420691 ]
P

# Ridge Regression - Standardised Data:

In [59]:
# =========================================================
# RIDGE REGRESSION - STANDARDISED DATA
# =========================================================

# Define the pipeline of data preprocessing and model training
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(random_state=seed))
])

# Define the parameter grid over which we optimise
param_grid = [
    {
        "model__alpha": np.logspace(-2, 4, 60)
    }
]

# Define Regression metrics over which we optimise 
scoring = {
    "mse": "neg_mean_squared_error",
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2"
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=gkf,
    scoring=scoring,
    refit="mae",   # Choose the best model based on MAE
    n_jobs=-1,
    verbose=1,
    return_train_score=False
)

grid.fit(X_train, yc_train, groups=period_train)

results = pd.DataFrame(grid.cv_results_)
best_idx = grid.best_index_

# =========================================================
# OVERALL BEST RESULT
# =========================================================
print("=" * 60)
print("BEST MODEL")
print("=" * 60)
print("Best params:", grid.best_params_)

# Multiply by (-1) as we used neg_...
print(f"Best mean CV MAE: {-grid.best_score_:.4f}")

print("\nMean CV metrics for best model:")
print(f"  MSE : {-results.loc[best_idx, 'mean_test_mse']:.4f}")
print(f"  RMSE: {-results.loc[best_idx, 'mean_test_rmse']:.4f}")
print(f"  MAE : {-results.loc[best_idx, 'mean_test_mae']:.4f}")
print(f"  R2  : {results.loc[best_idx, 'mean_test_r2']:.4f}")

# =========================================================
# PER-FOLD SCORES FOR BEST MODEL
# =========================================================
n_splits = gkf.get_n_splits(X_train, yc_train, groups=period_train)

print("\n" + "=" * 60)
print("PER-FOLD SCORES FOR BEST MODEL")
print("=" * 60)

for i in range(n_splits):
    mse = -results.loc[best_idx, f"split{i}_test_mse"]
    rmse = -results.loc[best_idx, f"split{i}_test_rmse"]
    mae = -results.loc[best_idx, f"split{i}_test_mae"]
    r2 = results.loc[best_idx, f"split{i}_test_r2"]

    print(f"Fold {i}:")
    print(f"  MSE  = {mse:.4f}")
    print(f"  RMSE = {rmse:.4f}")
    print(f"  MAE  = {mae:.4f}")
    print(f"  R2   = {r2:.4f}")


# =========================================================
# WEIGHTED MEAN (based on number of observations per fold)
# =========================================================

# 1) Find the number of observations per fold
fold_sizes = []
for _, val_idx in gkf.split(X_train, yc_train, groups=period_train):
    fold_sizes.append(len(val_idx))

fold_sizes = np.array(fold_sizes)

print("\nFold sizes:", fold_sizes)

# 2) Get the results of the best model
fold_mse = np.array([
    -results.loc[best_idx, f"split{i}_test_mse"]
    for i in range(n_splits)
])

fold_rmse = np.array([
    -results.loc[best_idx, f"split{i}_test_rmse"]
    for i in range(n_splits)
])

fold_mae = np.array([
    -results.loc[best_idx, f"split{i}_test_mae"]
    for i in range(n_splits)
])

fold_r2 = np.array([
    results.loc[best_idx, f"split{i}_test_r2"]
    for i in range(n_splits)
])

print("Per-fold MSE :", fold_mse)
print("Per-fold RMSE:", fold_rmse)
print("Per-fold MAE :", fold_mae)
print("Per-fold R2  :", fold_r2)

# 3) Comput weighted means of metrics
weighted_mse = np.average(fold_mse, weights=fold_sizes)
weighted_rmse = np.average(fold_rmse, weights=fold_sizes)
weighted_mae = np.average(fold_mae, weights=fold_sizes)
weighted_r2 = np.average(fold_r2, weights=fold_sizes)

print("\n" + "=" * 60)
print("WEIGHTED MEAN CV SCORES")
print("=" * 60)
print(f"Weighted MSE : {weighted_mse:.4f}")
print(f"Weighted RMSE: {weighted_rmse:.4f}")
print(f"Weighted MAE : {weighted_mae:.4f}")
print(f"Weighted R2  : {weighted_r2:.4f}")


# =========================================================
# PUBLIC TEST SET EVALUATION (REGRESSION)
# =========================================================
best_reg_model = grid.best_estimator_

y_pred_pub = best_reg_model.predict(X_pub_test)

pub_mse  = mean_squared_error(yc_pub_test, y_pred_pub)
pub_rmse = np.sqrt(pub_mse)
pub_mae  = mean_absolute_error(yc_pub_test, y_pred_pub)
pub_r2   = r2_score(yc_pub_test, y_pred_pub)

print("=" * 60)
print("PUBLIC TEST SET RESULTS (REGRESSION)")
print("=" * 60)
print(f"  MSE : {pub_mse:.4f}")
print(f"  RMSE: {pub_rmse:.4f}")
print(f"  MAE : {pub_mae:.4f}")
print(f"  R2  : {pub_r2:.4f}")

Fitting 6 folds for each of 60 candidates, totalling 360 fits
BEST MODEL
Best params: {'model__alpha': np.float64(186.71810912919207)}
Best mean CV MAE: 46.4443

Mean CV metrics for best model:
  MSE : 69557.3105
  RMSE: 156.4685
  MAE : 46.4443
  R2  : -4.9256

PER-FOLD SCORES FOR BEST MODEL
Fold 0:
  MSE  = 931.1688
  RMSE = 30.5151
  MAE  = 22.5920
  R2   = 0.8132
Fold 1:
  MSE  = 1546.3278
  RMSE = 39.3234
  MAE  = 28.5886
  R2   = 0.7336
Fold 2:
  MSE  = 390801.8147
  RMSE = 625.1414
  MAE  = 83.0843
  R2   = -32.5218
Fold 3:
  MSE  = 17415.5228
  RMSE = 131.9679
  MAE  = 73.6115
  R2   = 0.6527
Fold 4:
  MSE  = 1757.7927
  RMSE = 41.9260
  MAE  = 20.4760
  R2   = 0.6526
Fold 5:
  MSE  = 4891.2360
  RMSE = 69.9374
  MAE  = 50.3134
  R2   = 0.1164

Fold sizes: [2300 1586 1244  445  197  161]
Per-fold MSE : [   931.16876887   1546.32776988 390801.81467797  17415.52283325
   1757.79271574   4891.23596349]
Per-fold RMSE: [ 30.51505807  39.32337435 625.14143574 131.96788561  41.9260386

# Support Vector Regression - Standardised Data:

In [60]:
# =========================================================
# SVR (rbf kernel) REGRESSION - STANDARDISED DATA
# =========================================================

# Define the pipeline of how to preprocess the data and which model to train
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVR(kernel="rbf"))
])

# Define the parameter grid over which we optimise
param_grid = [
    {'model__C': [0.1, 1, 10],
     'model__gamma': ['scale', 0.001, 0.01]}
]

# Define the Regression metrics
scoring = {
    "mse": "neg_mean_squared_error",
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2"
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=gkf,
    scoring=scoring,
    refit="mae",   # Choose the best model based on MAE
    n_jobs=-1,
    verbose=1,
    return_train_score=False
)

grid.fit(X_train, yc_train, groups=period_train)

results = pd.DataFrame(grid.cv_results_)
best_idx = grid.best_index_

# =========================================================
# OVERALL BEST RESULT
# =========================================================
print("=" * 60)
print("BEST MODEL")
print("=" * 60)
print("Best params:", grid.best_params_)

# Multiply by (-1) as we used neg_...
print(f"Best mean CV MAE: {-grid.best_score_:.4f}")

print("\nMean CV metrics for best model:")
print(f"  MSE : {-results.loc[best_idx, 'mean_test_mse']:.4f}")
print(f"  RMSE: {-results.loc[best_idx, 'mean_test_rmse']:.4f}")
print(f"  MAE : {-results.loc[best_idx, 'mean_test_mae']:.4f}")
print(f"  R2  : {results.loc[best_idx, 'mean_test_r2']:.4f}")

# =========================================================
# PER-FOLD SCORES FOR BEST MODEL
# =========================================================
n_splits = gkf.get_n_splits(X_train, yc_train, groups=period_train)

print("\n" + "=" * 60)
print("PER-FOLD SCORES FOR BEST MODEL")
print("=" * 60)

for i in range(n_splits):
    mse = -results.loc[best_idx, f"split{i}_test_mse"]
    rmse = -results.loc[best_idx, f"split{i}_test_rmse"]
    mae = -results.loc[best_idx, f"split{i}_test_mae"]
    r2 = results.loc[best_idx, f"split{i}_test_r2"]

    print(f"Fold {i}:")
    print(f"  MSE  = {mse:.4f}")
    print(f"  RMSE = {rmse:.4f}")
    print(f"  MAE  = {mae:.4f}")
    print(f"  R2   = {r2:.4f}")


# =========================================================
# WEIGHTED MEAN (based on number of observations per fold)
# =========================================================

# 1) Find the number of observations for each validation set
fold_sizes = []
for _, val_idx in gkf.split(X_train, yc_train, groups=period_train):
    fold_sizes.append(len(val_idx))

fold_sizes = np.array(fold_sizes)

print("\nFold sizes:", fold_sizes)

# 2) Get the metrics for the best model
fold_mse = np.array([
    -results.loc[best_idx, f"split{i}_test_mse"]
    for i in range(n_splits)
])

fold_rmse = np.array([
    -results.loc[best_idx, f"split{i}_test_rmse"]
    for i in range(n_splits)
])

fold_mae = np.array([
    -results.loc[best_idx, f"split{i}_test_mae"]
    for i in range(n_splits)
])

fold_r2 = np.array([
    results.loc[best_idx, f"split{i}_test_r2"]
    for i in range(n_splits)
])

print("Per-fold MSE :", fold_mse)
print("Per-fold RMSE:", fold_rmse)
print("Per-fold MAE :", fold_mae)
print("Per-fold R2  :", fold_r2)

# 3) Compute weighted mean of the regression metrics
weighted_mse = np.average(fold_mse, weights=fold_sizes)
weighted_rmse = np.average(fold_rmse, weights=fold_sizes)
weighted_mae = np.average(fold_mae, weights=fold_sizes)
weighted_r2 = np.average(fold_r2, weights=fold_sizes)

print("\n" + "=" * 60)
print("WEIGHTED MEAN CV SCORES")
print("=" * 60)
print(f"Weighted MSE : {weighted_mse:.4f}")
print(f"Weighted RMSE: {weighted_rmse:.4f}")
print(f"Weighted MAE : {weighted_mae:.4f}")
print(f"Weighted R2  : {weighted_r2:.4f}")


# =========================================================
# PUBLIC TEST SET EVALUATION (REGRESSION)
# =========================================================
best_reg_model = grid.best_estimator_

y_pred_pub = best_reg_model.predict(X_pub_test)

pub_mse  = mean_squared_error(yc_pub_test, y_pred_pub)
pub_rmse = np.sqrt(pub_mse)
pub_mae  = mean_absolute_error(yc_pub_test, y_pred_pub)
pub_r2   = r2_score(yc_pub_test, y_pred_pub)

print("=" * 60)
print("PUBLIC TEST SET RESULTS (REGRESSION)")
print("=" * 60)
print(f"  MSE : {pub_mse:.4f}")
print(f"  RMSE: {pub_rmse:.4f}")
print(f"  MAE : {pub_mae:.4f}")
print(f"  R2  : {pub_r2:.4f}")

Fitting 6 folds for each of 9 candidates, totalling 54 fits
BEST MODEL
Best params: {'model__C': 10, 'model__gamma': 'scale'}
Best mean CV MAE: 37.2634

Mean CV metrics for best model:
  MSE : 6677.6865
  RMSE: 66.2508
  MAE : 37.2634
  R2  : 0.5793

PER-FOLD SCORES FOR BEST MODEL
Fold 0:
  MSE  = 441.8193
  RMSE = 21.0195
  MAE  = 14.3978
  R2   = 0.9114
Fold 1:
  MSE  = 1166.7007
  RMSE = 34.1570
  MAE  = 25.7362
  R2   = 0.7990
Fold 2:
  MSE  = 3668.7515
  RMSE = 60.5702
  MAE  = 29.4641
  R2   = 0.6853
Fold 3:
  MSE  = 27327.6724
  RMSE = 165.3108
  MAE  = 83.3845
  R2   = 0.4551
Fold 4:
  MSE  = 1581.5599
  RMSE = 39.7688
  MAE  = 21.0368
  R2   = 0.6874
Fold 5:
  MSE  = 5879.6149
  RMSE = 76.6786
  MAE  = 49.5613
  R2   = -0.0622

Fold sizes: [2300 1586 1244  445  197  161]
Per-fold MSE : [  441.81931051  1166.70074391  3668.75147212 27327.67244879
  1581.55994961  5879.6149226 ]
Per-fold RMSE: [ 21.01949834  34.15700139  60.57021935 165.31083585  39.76883138
  76.67864711]
Per-f

# kNN Regression - Standardised Data

In [61]:
# =========================================================
# KNN REGRESSION - STANDARDISED DATA
# =========================================================

# Define the pipeline of data preprocessing and model training
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsRegressor())
])

# Define the parameter grid over which we optimise
param_grid = [
    {'model__n_neighbors': [3, 5, 11, 21],
     'model__weights': ['uniform', 'distance']}
]

# Define the regression metrics
scoring = {
    "mse": "neg_mean_squared_error",
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2"
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=gkf,
    scoring=scoring,
    refit="mae",   # Choose the best model based on MAE
    n_jobs=-1,
    verbose=1,
    return_train_score=False
)

grid.fit(X_train, yc_train, groups=period_train)

results = pd.DataFrame(grid.cv_results_)
best_idx = grid.best_index_

# =========================================================
# OVERALL BEST RESULT
# =========================================================
print("=" * 60)
print("BEST MODEL")
print("=" * 60)
print("Best params:", grid.best_params_)

# Multiply by (-1) as we used neg_...
print(f"Best mean CV MAE: {-grid.best_score_:.4f}")

print("\nMean CV metrics for best model:")
print(f"  MSE : {-results.loc[best_idx, 'mean_test_mse']:.4f}")
print(f"  RMSE: {-results.loc[best_idx, 'mean_test_rmse']:.4f}")
print(f"  MAE : {-results.loc[best_idx, 'mean_test_mae']:.4f}")
print(f"  R2  : {results.loc[best_idx, 'mean_test_r2']:.4f}")

# =========================================================
# PER-FOLD SCORES FOR BEST MODEL
# =========================================================
n_splits = gkf.get_n_splits(X_train, yc_train, groups=period_train)

print("\n" + "=" * 60)
print("PER-FOLD SCORES FOR BEST MODEL")
print("=" * 60)

for i in range(n_splits):
    mse = -results.loc[best_idx, f"split{i}_test_mse"]
    rmse = -results.loc[best_idx, f"split{i}_test_rmse"]
    mae = -results.loc[best_idx, f"split{i}_test_mae"]
    r2 = results.loc[best_idx, f"split{i}_test_r2"]

    print(f"Fold {i}:")
    print(f"  MSE  = {mse:.4f}")
    print(f"  RMSE = {rmse:.4f}")
    print(f"  MAE  = {mae:.4f}")
    print(f"  R2   = {r2:.4f}")

# =========================================================
# WEIGHTED MEAN (based on number of observations per fold)
# =========================================================

# 1) Find the number of observations per fold
fold_sizes = []
for _, val_idx in gkf.split(X_train, yc_train, groups=period_train):
    fold_sizes.append(len(val_idx))

fold_sizes = np.array(fold_sizes)

print("\nFold sizes:", fold_sizes)

# 2) Get the metrics for the best model
fold_mse = np.array([
    -results.loc[best_idx, f"split{i}_test_mse"]
    for i in range(n_splits)
])

fold_rmse = np.array([
    -results.loc[best_idx, f"split{i}_test_rmse"]
    for i in range(n_splits)
])

fold_mae = np.array([
    -results.loc[best_idx, f"split{i}_test_mae"]
    for i in range(n_splits)
])

fold_r2 = np.array([
    results.loc[best_idx, f"split{i}_test_r2"]
    for i in range(n_splits)
])

print("Per-fold MSE :", fold_mse)
print("Per-fold RMSE:", fold_rmse)
print("Per-fold MAE :", fold_mae)
print("Per-fold R2  :", fold_r2)

# 3) Compute weighted mean of the regression metrics
weighted_mse = np.average(fold_mse, weights=fold_sizes)
weighted_rmse = np.average(fold_rmse, weights=fold_sizes)
weighted_mae = np.average(fold_mae, weights=fold_sizes)
weighted_r2 = np.average(fold_r2, weights=fold_sizes)

print("\n" + "=" * 60)
print("WEIGHTED MEAN CV SCORES")
print("=" * 60)
print(f"Weighted MSE : {weighted_mse:.4f}")
print(f"Weighted RMSE: {weighted_rmse:.4f}")
print(f"Weighted MAE : {weighted_mae:.4f}")
print(f"Weighted R2  : {weighted_r2:.4f}")


# =========================================================
# PUBLIC TEST SET EVALUATION (REGRESSION)
# =========================================================
best_reg_model = grid.best_estimator_

y_pred_pub = best_reg_model.predict(X_pub_test)

pub_mse  = mean_squared_error(yc_pub_test, y_pred_pub)
pub_rmse = np.sqrt(pub_mse)
pub_mae  = mean_absolute_error(yc_pub_test, y_pred_pub)
pub_r2   = r2_score(yc_pub_test, y_pred_pub)

print("=" * 60)
print("PUBLIC TEST SET RESULTS (REGRESSION)")
print("=" * 60)
print(f"  MSE : {pub_mse:.4f}")
print(f"  RMSE: {pub_rmse:.4f}")
print(f"  MAE : {pub_mae:.4f}")
print(f"  R2  : {pub_r2:.4f}")

Fitting 6 folds for each of 8 candidates, totalling 48 fits
BEST MODEL
Best params: {'model__n_neighbors': 11, 'model__weights': 'distance'}
Best mean CV MAE: 30.6229

Mean CV metrics for best model:
  MSE : 4259.8188
  RMSE: 56.5343
  MAE : 30.6229
  R2  : 0.7051

PER-FOLD SCORES FOR BEST MODEL
Fold 0:
  MSE  = 799.7771
  RMSE = 28.2803
  MAE  = 12.8235
  R2   = 0.8396
Fold 1:
  MSE  = 1175.1601
  RMSE = 34.2806
  MAE  = 24.3268
  R2   = 0.7976
Fold 2:
  MSE  = 4121.1759
  RMSE = 64.1964
  MAE  = 29.6026
  R2   = 0.6465
Fold 3:
  MSE  = 15467.2407
  RMSE = 124.3674
  MAE  = 63.9089
  R2   = 0.6916
Fold 4:
  MSE  = 1325.7294
  RMSE = 36.4106
  MAE  = 19.6422
  R2   = 0.7380
Fold 5:
  MSE  = 2669.8298
  RMSE = 51.6704
  MAE  = 33.4331
  R2   = 0.5177

Fold sizes: [2300 1586 1244  445  197  161]
Per-fold MSE : [  799.77708475  1175.16010292  4121.17585901 15467.24066452
  1325.72937179  2669.82980466]
Per-fold RMSE: [ 28.28033035  34.28060826  64.19638509 124.36736173  36.41056676
  51.6

# Multi Layer Perceptron - Standardised Data:

In [62]:
# =========================================================
# MULTI LAYER PERCEPTRON REGRESSOR - STANDARDISED DATA
# =========================================================

# Define the pipeline of data preprocessing and model training
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", MLPRegressor(max_iter=3000,
                           random_state=seed)) # set seed for reproducibility
])

# Define the parameter grid over which we optimise
param_grid = [
    {
        "model__hidden_layer_sizes": [
            (128, 64, 64, 32),
            (128, 128, 64, 64, 32, 16)
        ],
        "model__activation": ["relu"],
        "model__alpha": np.logspace(-5, -1, 3),
        "model__learning_rate_init": [1e-4],
        "model__solver": ["adam"] # Use ADAM as it adapts the step size based on the Hyperplane
    }
]

# Define Regression metrics
scoring = {
    "mse": "neg_mean_squared_error",
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2"
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=gkf,
    scoring=scoring,
    refit="mae",   # Choose the best model based on MAE
    n_jobs=-1,
    verbose=1,
    return_train_score=False
)

grid.fit(X_train, yc_train, groups=period_train)

results = pd.DataFrame(grid.cv_results_)
best_idx = grid.best_index_

# =========================================================
# OVERALL BEST RESULT
# =========================================================
print("=" * 60)
print("BEST MODEL")
print("=" * 60)
print("Best params:", grid.best_params_)

# We get a negative value for best_score_, as we used neg_..., thus multiply by (-1)
print(f"Best mean CV MAE: {-grid.best_score_:.4f}")

print("\nMean CV metrics for best model:")
print(f"  MSE : {-results.loc[best_idx, 'mean_test_mse']:.4f}")
print(f"  RMSE: {-results.loc[best_idx, 'mean_test_rmse']:.4f}")
print(f"  MAE : {-results.loc[best_idx, 'mean_test_mae']:.4f}")
print(f"  R2  : {results.loc[best_idx, 'mean_test_r2']:.4f}")

# =========================================================
# PER-FOLD SCORES FOR BEST MODEL
# =========================================================
n_splits = gkf.get_n_splits(X_train, yc_train, groups=period_train)

print("\n" + "=" * 60)
print("PER-FOLD SCORES FOR BEST MODEL")
print("=" * 60)

for i in range(n_splits):
    mse = -results.loc[best_idx, f"split{i}_test_mse"]
    rmse = -results.loc[best_idx, f"split{i}_test_rmse"]
    mae = -results.loc[best_idx, f"split{i}_test_mae"]
    r2 = results.loc[best_idx, f"split{i}_test_r2"]

    print(f"Fold {i}:")
    print(f"  MSE  = {mse:.4f}")
    print(f"  RMSE = {rmse:.4f}")
    print(f"  MAE  = {mae:.4f}")
    print(f"  R2   = {r2:.4f}")


# =========================================================
# WEIGHTED MEAN (based on number of observations per fold)
# =========================================================

# 1) Find the number of observations per fold
fold_sizes = []
for _, val_idx in gkf.split(X_train, yc_train, groups=period_train):
    fold_sizes.append(len(val_idx))

fold_sizes = np.array(fold_sizes)

print("\nFold sizes:", fold_sizes)

# 2) Get the results of the best models
fold_mse = np.array([
    -results.loc[best_idx, f"split{i}_test_mse"]
    for i in range(n_splits)
])

fold_rmse = np.array([
    -results.loc[best_idx, f"split{i}_test_rmse"]
    for i in range(n_splits)
])

fold_mae = np.array([
    -results.loc[best_idx, f"split{i}_test_mae"]
    for i in range(n_splits)
])

fold_r2 = np.array([
    results.loc[best_idx, f"split{i}_test_r2"]
    for i in range(n_splits)
])

print("Per-fold MSE :", fold_mse)
print("Per-fold RMSE:", fold_rmse)
print("Per-fold MAE :", fold_mae)
print("Per-fold R2  :", fold_r2)

# 3) Compute weighted mean of regression metrics
weighted_mse = np.average(fold_mse, weights=fold_sizes)
weighted_rmse = np.average(fold_rmse, weights=fold_sizes)
weighted_mae = np.average(fold_mae, weights=fold_sizes)
weighted_r2 = np.average(fold_r2, weights=fold_sizes)

print("\n" + "=" * 60)
print("WEIGHTED MEAN CV SCORES")
print("=" * 60)
print(f"Weighted MSE : {weighted_mse:.4f}")
print(f"Weighted RMSE: {weighted_rmse:.4f}")
print(f"Weighted MAE : {weighted_mae:.4f}")
print(f"Weighted R2  : {weighted_r2:.4f}")


# =========================================================
# PUBLIC TEST SET EVALUATION (REGRESSION)
# =========================================================
best_reg_model = grid.best_estimator_

y_pred_pub = best_reg_model.predict(X_pub_test)

pub_mse  = mean_squared_error(yc_pub_test, y_pred_pub)
pub_rmse = np.sqrt(pub_mse)
pub_mae  = mean_absolute_error(yc_pub_test, y_pred_pub)
pub_r2   = r2_score(yc_pub_test, y_pred_pub)

print("=" * 60)
print("PUBLIC TEST SET RESULTS (REGRESSION)")
print("=" * 60)
print(f"  MSE : {pub_mse:.4f}")
print(f"  RMSE: {pub_rmse:.4f}")
print(f"  MAE : {pub_mae:.4f}")
print(f"  R2  : {pub_r2:.4f}")

Fitting 6 folds for each of 6 candidates, totalling 36 fits
BEST MODEL
Best params: {'model__activation': 'relu', 'model__alpha': np.float64(1e-05), 'model__hidden_layer_sizes': (128, 128, 64, 64, 32, 16), 'model__learning_rate_init': 0.0001, 'model__solver': 'adam'}
Best mean CV MAE: 76.9492

Mean CV metrics for best model:
  MSE : 6100983.4742
  RMSE: 1060.1393
  MAE : 76.9492
  R2  : -522.3937

PER-FOLD SCORES FOR BEST MODEL
Fold 0:
  MSE  = 6715.4994
  RMSE = 81.9482
  MAE  = 47.9071
  R2   = -0.3468
Fold 1:
  MSE  = 928.9940
  RMSE = 30.4794
  MAE  = 23.8999
  R2   = 0.8400
Fold 2:
  MSE  = 36580904.1184
  RMSE = 6048.2150
  MAE  = 250.3590
  R2   = -3136.8037
Fold 3:
  MSE  = 13030.5672
  RMSE = 114.1515
  MAE  = 83.0799
  R2   = 0.7402
Fold 4:
  MSE  = 645.8367
  RMSE = 25.4133
  MAE  = 19.1873
  R2   = 0.8723
Fold 5:
  MSE  = 3675.8298
  RMSE = 60.6286
  MAE  = 37.2618
  R2   = 0.3359

Fold sizes: [2300 1586 1244  445  197  161]
Per-fold MSE : [6.71549935e+03 9.28994008e+02 3.6

# Substance Specific Regression Models:

# Support Vector Regression (RBF Kernel) - Standardised Data:

# Substance 1

In [63]:
# =========================================================
# SVR (rbf kernel) REGRESSION - STANDARDISED DATA - Substance 1
# =========================================================

# Define the pipeline of data preprocessing and model training
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVR(kernel="rbf"))
])

# Define the parameter grid over which we want to optimise
param_grid = [
    {'model__C': [0.1, 1, 10],
     'model__gamma': ['scale', 0.001, 0.01]}
]

# Define the regression metrics
scoring = {
    "mse": "neg_mean_squared_error",
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2"
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=gkf1,
    scoring=scoring,
    refit="mae",   # Choose the best model based on MAE
    n_jobs=-1,
    verbose=1,
    return_train_score=False
)

grid.fit(X_train_s1, yc_train_s1, groups=period_train_s1) # Fit on substance based subgroups

results = pd.DataFrame(grid.cv_results_)
best_idx = grid.best_index_

# =========================================================
# OVERALL BEST RESULT
# =========================================================
print("=" * 60)
print("BEST MODEL")
print("=" * 60)
print("Best params:", grid.best_params_)

print(f"Best mean CV MAE: {-grid.best_score_:.4f}")

print("\nMean CV metrics for best model:")
print(f"  MSE : {-results.loc[best_idx, 'mean_test_mse']:.4f}")
print(f"  RMSE: {-results.loc[best_idx, 'mean_test_rmse']:.4f}")
print(f"  MAE : {-results.loc[best_idx, 'mean_test_mae']:.4f}")
print(f"  R2  : {results.loc[best_idx, 'mean_test_r2']:.4f}")

# =========================================================
# PER-FOLD SCORES FOR BEST MODEL
# =========================================================
n_splits = gkf1.get_n_splits(X_train_s1, yc_train_s1, groups=period_train_s1)

print("\n" + "=" * 60)
print("PER-FOLD SCORES FOR BEST MODEL")
print("=" * 60)

for i in range(n_splits):
    mse = -results.loc[best_idx, f"split{i}_test_mse"]
    rmse = -results.loc[best_idx, f"split{i}_test_rmse"]
    mae = -results.loc[best_idx, f"split{i}_test_mae"]
    r2 = results.loc[best_idx, f"split{i}_test_r2"]

    print(f"Fold {i}:")
    print(f"  MSE  = {mse:.4f}")
    print(f"  RMSE = {rmse:.4f}")
    print(f"  MAE  = {mae:.4f}")
    print(f"  R2   = {r2:.4f}")


# =========================================================
# WEIGHTED MEAN (based on number of observations per fold)
# =========================================================

# 1) Find the number of observations for each fold
fold_sizes = []
for _, val_idx in gkf1.split(X_train_s1, yc_train_s1, groups=period_train_s1):
    fold_sizes.append(len(val_idx))

fold_sizes = np.array(fold_sizes)

print("\nFold sizes:", fold_sizes)

# 2) Get the metrics for the best model
fold_mse = np.array([
    -results.loc[best_idx, f"split{i}_test_mse"]
    for i in range(n_splits)
])

fold_rmse = np.array([
    -results.loc[best_idx, f"split{i}_test_rmse"]
    for i in range(n_splits)
])

fold_mae = np.array([
    -results.loc[best_idx, f"split{i}_test_mae"]
    for i in range(n_splits)
])

fold_r2 = np.array([
    results.loc[best_idx, f"split{i}_test_r2"]
    for i in range(n_splits)
])

print("Per-fold MSE :", fold_mse)
print("Per-fold RMSE:", fold_rmse)
print("Per-fold MAE :", fold_mae)
print("Per-fold R2  :", fold_r2)

# 3) Compute the weighted mean of the regression metrics
weighted_mse = np.average(fold_mse, weights=fold_sizes)
weighted_rmse = np.average(fold_rmse, weights=fold_sizes)
weighted_mae = np.average(fold_mae, weights=fold_sizes)
weighted_r2 = np.average(fold_r2, weights=fold_sizes)

print("\n" + "=" * 60)
print("WEIGHTED MEAN CV SCORES")
print("=" * 60)
print(f"Weighted MSE : {weighted_mse:.4f}")
print(f"Weighted RMSE: {weighted_rmse:.4f}")
print(f"Weighted MAE : {weighted_mae:.4f}")
print(f"Weighted R2  : {weighted_r2:.4f}")

# =========================================================
# PUBLIC TEST SET EVALUATION - SUBSTANCE 1
# =========================================================

# 1) Filter Public Test for Substance 1 only
X_pub_test_s1 = X_pub_test.loc[ys_pub_test == 1, :]
yc_pub_test_s1 = yc_pub_test.loc[ys_pub_test == 1]

# 2) Use the best estimator from our Substance 1 grid
best_reg_s1 = grid.best_estimator_

# 3) Predict
y_pred_pub_s1 = best_reg_s1.predict(X_pub_test_s1)

# 4) Calculate Metrics
pub_mse_s1  = mean_squared_error(yc_pub_test_s1, y_pred_pub_s1)
pub_rmse_s1 = np.sqrt(pub_mse_s1)
pub_mae_s1  = mean_absolute_error(yc_pub_test_s1, y_pred_pub_s1)
pub_r2_s1   = r2_score(yc_pub_test_s1, y_pred_pub_s1)

# 5) Print Results
print("=" * 60)
print("PUBLIC TEST SET RESULTS - SUBSTANCE 1")
print("=" * 60)
print(f"  MSE : {pub_mse_s1:.4f}")
print(f"  RMSE: {pub_rmse_s1:.4f}")
print(f"  MAE : {pub_mae_s1:.4f}")
print(f"  R2  : {pub_r2_s1:.4f}")


Fitting 6 folds for each of 9 candidates, totalling 54 fits
BEST MODEL
Best params: {'model__C': 10, 'model__gamma': 0.001}
Best mean CV MAE: 31.3526

Mean CV metrics for best model:
  MSE : 4798.7616
  RMSE: 50.9320
  MAE : 31.3526
  R2  : 0.6579

PER-FOLD SCORES FOR BEST MODEL
Fold 0:
  MSE  = 126.0969
  RMSE = 11.2293
  MAE  = 7.6634
  R2   = 0.9324
Fold 1:
  MSE  = 407.4893
  RMSE = 20.1864
  MAE  = 17.7035
  R2   = 0.9277
Fold 2:
  MSE  = 22463.6620
  RMSE = 149.8788
  MAE  = 66.6126
  R2   = 0.0163
Fold 3:
  MSE  = 2184.7472
  RMSE = 46.7413
  MAE  = 34.1478
  R2   = 0.7176
Fold 4:
  MSE  = 3152.0347
  RMSE = 56.1430
  MAE  = 43.4280
  R2   = 0.4366
Fold 5:
  MSE  = 458.5392
  RMSE = 21.4135
  MAE  = 18.5604
  R2   = 0.9168

Fold sizes: [514 365 164  90  64  28]
Per-fold MSE : [  126.09687429   407.48928426 22463.66201558  2184.74723517
  3152.03474837   458.53924632]
Per-fold RMSE: [ 11.22928646  20.18636382 149.87882444  46.74127978  56.14298485
  21.41352952]
Per-fold MAE : [ 

# Substance 2

In [64]:
# =========================================================
# GRADIENT BOOSTING REGRESSION - STANDARDISED DATA - Substance 2
# =========================================================

# Define the pipeline of data preprocessing and model training
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", GradientBoostingRegressor(random_state=seed)) # Set seed for reproducibility
])

# Define the parameter grid over which we optimise
param_grid = [
    {'model__n_estimators': [100],
     'model__learning_rate': [0.01, 0.1],
     'model__max_depth': [3,5],
     'model__subsample':[0.8]}
]

# Define the regression metrics 
scoring = {
    "mse": "neg_mean_squared_error",
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2"
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=gkf2,
    scoring=scoring,
    refit="mae",   # Choose the best model based on MAE
    n_jobs=-1,
    verbose=1,
    return_train_score=False
)

grid.fit(X_train_s2, yc_train_s2, groups=period_train_s2)

results = pd.DataFrame(grid.cv_results_)
best_idx = grid.best_index_

# =========================================================
# OVERALL BEST RESULT
# =========================================================
print("=" * 60)
print("BEST MODEL")
print("=" * 60)
print("Best params:", grid.best_params_)

# Multiply with (-1) as we used neg_...
print(f"Best mean CV MAE: {-grid.best_score_:.4f}")

print("\nMean CV metrics for best model:")
print(f"  MSE : {-results.loc[best_idx, 'mean_test_mse']:.4f}")
print(f"  RMSE: {-results.loc[best_idx, 'mean_test_rmse']:.4f}")
print(f"  MAE : {-results.loc[best_idx, 'mean_test_mae']:.4f}")
print(f"  R2  : {results.loc[best_idx, 'mean_test_r2']:.4f}")

# =========================================================
# PER-FOLD SCORES FOR BEST MODEL
# =========================================================
n_splits = gkf2.get_n_splits(X_train_s2, yc_train_s2, groups=period_train_s2)

print("\n" + "=" * 60)
print("PER-FOLD SCORES FOR BEST MODEL")
print("=" * 60)

for i in range(n_splits):
    mse = -results.loc[best_idx, f"split{i}_test_mse"]
    rmse = -results.loc[best_idx, f"split{i}_test_rmse"]
    mae = -results.loc[best_idx, f"split{i}_test_mae"]
    r2 = results.loc[best_idx, f"split{i}_test_r2"]

    print(f"Fold {i}:")
    print(f"  MSE  = {mse:.4f}")
    print(f"  RMSE = {rmse:.4f}")
    print(f"  MAE  = {mae:.4f}")
    print(f"  R2   = {r2:.4f}")

# =========================================================
# WEIGHTED MEAN (based on number of observations per fold)
# =========================================================

# 1) Find the number of observations per fold
fold_sizes = []
for _, val_idx in gkf2.split(X_train_s2, yc_train_s2, groups=period_train_s2):
    fold_sizes.append(len(val_idx))

fold_sizes = np.array(fold_sizes)

print("\nFold sizes:", fold_sizes)

# 2) Get the results of the best model
fold_mse = np.array([
    -results.loc[best_idx, f"split{i}_test_mse"]
    for i in range(n_splits)
])

fold_rmse = np.array([
    -results.loc[best_idx, f"split{i}_test_rmse"]
    for i in range(n_splits)
])

fold_mae = np.array([
    -results.loc[best_idx, f"split{i}_test_mae"]
    for i in range(n_splits)
])

fold_r2 = np.array([
    results.loc[best_idx, f"split{i}_test_r2"]
    for i in range(n_splits)
])

print("Per-fold MSE :", fold_mse)
print("Per-fold RMSE:", fold_rmse)
print("Per-fold MAE :", fold_mae)
print("Per-fold R2  :", fold_r2)

# 3) Compute the weighted mean
weighted_mse = np.average(fold_mse, weights=fold_sizes)
weighted_rmse = np.average(fold_rmse, weights=fold_sizes)
weighted_mae = np.average(fold_mae, weights=fold_sizes)
weighted_r2 = np.average(fold_r2, weights=fold_sizes)

print("\n" + "=" * 60)
print("WEIGHTED MEAN CV SCORES")
print("=" * 60)
print(f"Weighted MSE : {weighted_mse:.4f}")
print(f"Weighted RMSE: {weighted_rmse:.4f}")
print(f"Weighted MAE : {weighted_mae:.4f}")
print(f"Weighted R2  : {weighted_r2:.4f}")

# =========================================================
# PUBLIC TEST SET EVALUATION - SUBSTANCE 2
# =========================================================
X_pub_test_s2 = X_pub_test.loc[ys_pub_test == 2, :]
yc_pub_test_s2 = yc_pub_test.loc[ys_pub_test == 2]

best_reg_s2 = grid.best_estimator_
y_pred_pub_s2 = best_reg_s2.predict(X_pub_test_s2)

print("=" * 60)
print("PUBLIC TEST SET RESULTS - SUBSTANCE 2")
print("=" * 60)
print(f"  MSE : {mean_squared_error(yc_pub_test_s2, y_pred_pub_s2):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(yc_pub_test_s2, y_pred_pub_s2)):.4f}")
print(f"  MAE : {mean_absolute_error(yc_pub_test_s2, y_pred_pub_s2):.4f}")
print(f"  R2  : {r2_score(yc_pub_test_s2, y_pred_pub_s2):.4f}")

Fitting 6 folds for each of 4 candidates, totalling 24 fits
BEST MODEL
Best params: {'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 100, 'model__subsample': 0.8}
Best mean CV MAE: 13.3493

Mean CV metrics for best model:
  MSE : 446.5788
  RMSE: 19.3168
  MAE : 13.3493
  R2  : 0.7750

PER-FOLD SCORES FOR BEST MODEL
Fold 0:
  MSE  = 218.2608
  RMSE = 14.7737
  MAE  = 9.0628
  R2   = 0.9087
Fold 1:
  MSE  = 432.3260
  RMSE = 20.7924
  MAE  = 15.3314
  R2   = 0.9141
Fold 2:
  MSE  = 1137.7657
  RMSE = 33.7308
  MAE  = 23.2158
  R2   = 0.8764
Fold 3:
  MSE  = 394.6845
  RMSE = 19.8667
  MAE  = 12.6957
  R2   = 0.9541
Fold 4:
  MSE  = 471.1117
  RMSE = 21.7051
  MAE  = 16.1676
  R2   = 0.0000
Fold 5:
  MSE  = 25.3240
  RMSE = 5.0323
  MAE  = 3.6226
  R2   = 0.9967

Fold sizes: [574 490 334  98  43  40]
Per-fold MSE : [ 218.26082334  432.32595112 1137.76567778  394.68454068  471.11169411
   25.32395536]
Per-fold RMSE: [14.77365301 20.79244938 33.73078235 19.866669

# Substance 3

In [65]:
# =========================================================
# SVR (linear kernel) REGRESSION - STANDARDISED DATA
# Substance 3
# =========================================================

# Define the pipeline of data preprocessing adn model training
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVR(kernel="linear"))
])

# Define the parameter grid over which we optimise
param_grid = [
    {'model__C': [0.1, 1, 10]}
]

# Define regression metrics
scoring = {
    "mse": "neg_mean_squared_error",
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2"
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=gkf3,
    scoring=scoring,
    refit="mae",   # Choose the best model based on MAE
    n_jobs=-1,
    verbose=1,
    return_train_score=False
)

grid.fit(X_train_s3, yc_train_s3, groups=period_train_s3)

results = pd.DataFrame(grid.cv_results_)
best_idx = grid.best_index_

# =========================================================
# OVERALL BEST RESULT
# =========================================================
print("=" * 60)
print("BEST MODEL")
print("=" * 60)
print("Best params:", grid.best_params_)

# Multiply with (-1) as we used neg_...
print(f"Best mean CV MAE: {-grid.best_score_:.4f}")

print("\nMean CV metrics for best model:")
print(f"  MSE : {-results.loc[best_idx, 'mean_test_mse']:.4f}")
print(f"  RMSE: {-results.loc[best_idx, 'mean_test_rmse']:.4f}")
print(f"  MAE : {-results.loc[best_idx, 'mean_test_mae']:.4f}")
print(f"  R2  : {results.loc[best_idx, 'mean_test_r2']:.4f}")

# =========================================================
# PER-FOLD SCORES FOR BEST MODEL
# =========================================================
n_splits = gkf3.get_n_splits(X_train_s3, yc_train_s3, groups=period_train_s3)

print("\n" + "=" * 60)
print("PER-FOLD SCORES FOR BEST MODEL")
print("=" * 60)

for i in range(n_splits):
    mse = -results.loc[best_idx, f"split{i}_test_mse"]
    rmse = -results.loc[best_idx, f"split{i}_test_rmse"]
    mae = -results.loc[best_idx, f"split{i}_test_mae"]
    r2 = results.loc[best_idx, f"split{i}_test_r2"]

    print(f"Fold {i}:")
    print(f"  MSE  = {mse:.4f}")
    print(f"  RMSE = {rmse:.4f}")
    print(f"  MAE  = {mae:.4f}")
    print(f"  R2   = {r2:.4f}")


# =========================================================
# WEIGHTED MEAN (based on number of observations per fold)
# =========================================================

# 1) Find the number of observations per fold
fold_sizes = []
for _, val_idx in gkf3.split(X_train_s3, yc_train_s3, groups=period_train_s3):
    fold_sizes.append(len(val_idx))

fold_sizes = np.array(fold_sizes)

print("\nFold sizes:", fold_sizes)

# 2) Get the results of the best model
fold_mse = np.array([
    -results.loc[best_idx, f"split{i}_test_mse"]
    for i in range(n_splits)
])

fold_rmse = np.array([
    -results.loc[best_idx, f"split{i}_test_rmse"]
    for i in range(n_splits)
])

fold_mae = np.array([
    -results.loc[best_idx, f"split{i}_test_mae"]
    for i in range(n_splits)
])

fold_r2 = np.array([
    results.loc[best_idx, f"split{i}_test_r2"]
    for i in range(n_splits)
])

print("Per-fold MSE :", fold_mse)
print("Per-fold RMSE:", fold_rmse)
print("Per-fold MAE :", fold_mae)
print("Per-fold R2  :", fold_r2)

# 3) Compute the weighted mean
weighted_mse = np.average(fold_mse, weights=fold_sizes)
weighted_rmse = np.average(fold_rmse, weights=fold_sizes)
weighted_mae = np.average(fold_mae, weights=fold_sizes)
weighted_r2 = np.average(fold_r2, weights=fold_sizes)

print("\n" + "=" * 60)
print("WEIGHTED MEAN CV SCORES")
print("=" * 60)
print(f"Weighted MSE : {weighted_mse:.4f}")
print(f"Weighted RMSE: {weighted_rmse:.4f}")
print(f"Weighted MAE : {weighted_mae:.4f}")
print(f"Weighted R2  : {weighted_r2:.4f}")


# =========================================================
# PUBLIC TEST SET EVALUATION - SUBSTANCE 3
# =========================================================
X_pub_test_s3 = X_pub_test.loc[ys_pub_test == 3, :]
yc_pub_test_s3 = yc_pub_test.loc[ys_pub_test == 3]

best_reg_s3 = grid.best_estimator_
y_pred_pub_s3 = best_reg_s3.predict(X_pub_test_s3)

print("=" * 60)
print("PUBLIC TEST SET RESULTS - SUBSTANCE 3")
print("=" * 60)
print(f"  MSE : {mean_squared_error(yc_pub_test_s3, y_pred_pub_s3):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(yc_pub_test_s2, y_pred_pub_s2)):.4f}")
print(f"  MAE : {mean_absolute_error(yc_pub_test_s3, y_pred_pub_s3):.4f}")
print(f"  R2  : {r2_score(yc_pub_test_s3, y_pred_pub_s3):.4f}")

Fitting 6 folds for each of 3 candidates, totalling 18 fits
BEST MODEL
Best params: {'model__C': 10}
Best mean CV MAE: 21.0729

Mean CV metrics for best model:
  MSE : 1527.6004
  RMSE: 28.5776
  MAE : 21.0729
  R2  : 0.6069

PER-FOLD SCORES FOR BEST MODEL
Fold 0:
  MSE  = 136.7432
  RMSE = 11.6937
  MAE  = 8.1377
  R2   = 0.9671
Fold 1:
  MSE  = 247.3600
  RMSE = 15.7277
  MAE  = 10.9653
  R2   = 0.7735
Fold 2:
  MSE  = 470.1379
  RMSE = 21.6827
  MAE  = 20.2984
  R2   = 0.9765
Fold 3:
  MSE  = 7692.1178
  RMSE = 87.7047
  MAE  = 64.0243
  R2   = 0.9246
Fold 4:
  MSE  = 415.5695
  RMSE = 20.3855
  MAE  = 13.7359
  R2   = 0.0000
Fold 5:
  MSE  = 203.6738
  RMSE = 14.2714
  MAE  = 9.2757
  R2   = 0.0000

Fold sizes: [216 110 100  83  20  12]
Per-fold MSE : [ 136.74321715  247.36002029  470.13788541 7692.11776586  415.56948804
  203.67383696]
Per-fold RMSE: [11.69372555 15.72768325 21.68266325 87.70471918 20.38552153 14.2714343 ]
Per-fold MAE : [ 8.13767795 10.96531967 20.2984273  64.024

# Substance 4

In [66]:
# =========================================================
# REGRESSION TREE REGRESSION - STANDARDISED DATA - SUBSTANCE 4
# =========================================================

# Define the pipeline of data preprocessing and model training
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", DecisionTreeRegressor(random_state=seed)) # Set seed for reproducibility
])

# Define the grid over which we want to optimise
param_grid = [
    {'model__max_depth': [None, 5, 10, 20],
     'model__min_samples_split': [2, 5, 10]}
]

# Define the metrics
scoring = {
    "mse": "neg_mean_squared_error",
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2"
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=gkf4,
    scoring=scoring,
    refit="mae",   # Choose the best model based on MAE
    n_jobs=-1,
    verbose=1,
    return_train_score=False
)

grid.fit(X_train_s4, yc_train_s4, groups=period_train_s4)

results = pd.DataFrame(grid.cv_results_)
best_idx = grid.best_index_

# =========================================================
# OVERALL BEST RESULT
# =========================================================
print("=" * 60)
print("BEST MODEL")
print("=" * 60)
print("Best params:", grid.best_params_)

# Multiply with (-1) as we used neg_...
print(f"Best mean CV MAE: {-grid.best_score_:.4f}")

print("\nMean CV metrics for best model:")
print(f"  MSE : {-results.loc[best_idx, 'mean_test_mse']:.4f}")
print(f"  RMSE: {-results.loc[best_idx, 'mean_test_rmse']:.4f}")
print(f"  MAE : {-results.loc[best_idx, 'mean_test_mae']:.4f}")
print(f"  R2  : {results.loc[best_idx, 'mean_test_r2']:.4f}")

# =========================================================
# PER-FOLD SCORES FOR BEST MODEL
# =========================================================
n_splits = gkf4.get_n_splits(X_train_s4, yc_train_s4, groups=period_train_s4)

print("\n" + "=" * 60)
print("PER-FOLD SCORES FOR BEST MODEL")
print("=" * 60)

for i in range(n_splits):
    mse = -results.loc[best_idx, f"split{i}_test_mse"]
    rmse = -results.loc[best_idx, f"split{i}_test_rmse"]
    mae = -results.loc[best_idx, f"split{i}_test_mae"]
    r2 = results.loc[best_idx, f"split{i}_test_r2"]

    print(f"Fold {i}:")
    print(f"  MSE  = {mse:.4f}")
    print(f"  RMSE = {rmse:.4f}")
    print(f"  MAE  = {mae:.4f}")
    print(f"  R2   = {r2:.4f}")


# =========================================================
# WEIGHTED MEAN (based on number of observations per fold)
# =========================================================

# 1) Get the number of observations per fold
fold_sizes = []
for _, val_idx in gkf4.split(X_train_s4, yc_train_s4, groups=period_train_s4):
    fold_sizes.append(len(val_idx))

fold_sizes = np.array(fold_sizes)

print("\nFold sizes:", fold_sizes)

# 2) Get the results of the best model
fold_mse = np.array([
    -results.loc[best_idx, f"split{i}_test_mse"]
    for i in range(n_splits)
])

fold_rmse = np.array([
    -results.loc[best_idx, f"split{i}_test_rmse"]
    for i in range(n_splits)
])

fold_mae = np.array([
    -results.loc[best_idx, f"split{i}_test_mae"]
    for i in range(n_splits)
])

fold_r2 = np.array([
    results.loc[best_idx, f"split{i}_test_r2"]
    for i in range(n_splits)
])

print("Per-fold MSE :", fold_mse)
print("Per-fold RMSE:", fold_rmse)
print("Per-fold MAE :", fold_mae)
print("Per-fold R2  :", fold_r2)

# 3) Compute the weighted mean
weighted_mse = np.average(fold_mse, weights=fold_sizes)
weighted_rmse = np.average(fold_rmse, weights=fold_sizes)
weighted_mae = np.average(fold_mae, weights=fold_sizes)
weighted_r2 = np.average(fold_r2, weights=fold_sizes)

print("\n" + "=" * 60)
print("WEIGHTED MEAN CV SCORES")
print("=" * 60)
print(f"Weighted MSE : {weighted_mse:.4f}")
print(f"Weighted RMSE: {weighted_rmse:.4f}")
print(f"Weighted MAE : {weighted_mae:.4f}")
print(f"Weighted R2  : {weighted_r2:.4f}")

# =========================================================
# PUBLIC TEST SET EVALUATION - SUBSTANCE 4
# =========================================================
X_pub_test_s4 = X_pub_test.loc[ys_pub_test == 4, :]
yc_pub_test_s4 = yc_pub_test.loc[ys_pub_test == 4]

best_reg_s4 = grid.best_estimator_
y_pred_pub_s4 = best_reg_s4.predict(X_pub_test_s4)

print("=" * 60)
print("PUBLIC TEST SET RESULTS - SUBSTANCE 4")
print("=" * 60)
print(f"  MSE : {mean_squared_error(yc_pub_test_s4, y_pred_pub_s4):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(yc_pub_test_s2, y_pred_pub_s2)):.4f}")
print(f"  MAE : {mean_absolute_error(yc_pub_test_s4, y_pred_pub_s4):.4f}")
print(f"  R2  : {r2_score(yc_pub_test_s4, y_pred_pub_s4):.4f}")

Fitting 6 folds for each of 12 candidates, totalling 72 fits
BEST MODEL
Best params: {'model__max_depth': None, 'model__min_samples_split': 5}
Best mean CV MAE: 15.6720

Mean CV metrics for best model:
  MSE : 613.6348
  RMSE: 21.3635
  MAE : 15.6720
  R2  : 0.8072

PER-FOLD SCORES FOR BEST MODEL
Fold 0:
  MSE  = 382.8125
  RMSE = 19.5656
  MAE  = 14.0625
  R2   = 0.9258
Fold 1:
  MSE  = 1115.2523
  RMSE = 33.3954
  MAE  = 27.4083
  R2   = 0.7776
Fold 2:
  MSE  = 271.7391
  RMSE = 16.4845
  MAE  = 5.4348
  R2   = 0.5208
Fold 3:
  MSE  = 0.0000
  RMSE = 0.0000
  MAE  = 0.0000
  R2   = 1.0000
Fold 4:
  MSE  = 1524.0741
  RMSE = 39.0394
  MAE  = 33.3333
  R2   = 0.8310
Fold 5:
  MSE  = 387.9310
  RMSE = 19.6960
  MAE  = 13.7931
  R2   = 0.7878

Fold sizes: [240 109  46  30  30  29]
Per-fold MSE : [ 382.8125     1115.25229358  271.73913043    0.         1524.07407407
  387.93103448]
Per-fold RMSE: [19.5655948  33.3953933  16.48451183  0.         39.03939131 19.69596493]
Per-fold MAE : [14.

# Substance 5

In [67]:
# =========================================================
# SVR (linear kernel) REGRESSION - STANDARDISED DATA -> PCA
# Substance 5
# =========================================================

# Define the pipeline of data preprocessing and model training
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA()),
    ("model", SVR(kernel="linear"))
])

# Define the parameter grid over which we optimise
param_grid = [
    {'model__C': [0.1, 1, 10],
     'pca__n_components': [5, 10, 20, 50, 75]}
]


# Define the regression metrics
scoring = {
    "mse": "neg_mean_squared_error",
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2"
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=gkf5, # Choose gkf5 for Substance 5
    scoring=scoring,
    refit="mae",   # Choose the best model based on MAE
    n_jobs=-1,
    verbose=1,
    return_train_score=False
)

grid.fit(X_train_s5, yc_train_s5, groups=period_train_s5)

results = pd.DataFrame(grid.cv_results_)
best_idx = grid.best_index_

# =========================================================
# OVERALL BEST RESULT
# =========================================================
print("=" * 60)
print("BEST MODEL")
print("=" * 60)
print("Best params:", grid.best_params_)

# Multiply with (-1) as we used neg_...
print(f"Best mean CV MAE: {-grid.best_score_:.4f}")

print("\nMean CV metrics for best model:")
print(f"  MSE : {-results.loc[best_idx, 'mean_test_mse']:.4f}")
print(f"  RMSE: {-results.loc[best_idx, 'mean_test_rmse']:.4f}")
print(f"  MAE : {-results.loc[best_idx, 'mean_test_mae']:.4f}")
print(f"  R2  : {results.loc[best_idx, 'mean_test_r2']:.4f}")

# =========================================================
# PER-FOLD SCORES FOR BEST MODEL
# =========================================================
n_splits = gkf5.get_n_splits(X_train_s5, yc_train_s5, groups=period_train_s5)

print("\n" + "=" * 60)
print("PER-FOLD SCORES FOR BEST MODEL")
print("=" * 60)

for i in range(n_splits):
    mse = -results.loc[best_idx, f"split{i}_test_mse"]
    rmse = -results.loc[best_idx, f"split{i}_test_rmse"]
    mae = -results.loc[best_idx, f"split{i}_test_mae"]
    r2 = results.loc[best_idx, f"split{i}_test_r2"]

    print(f"Fold {i}:")
    print(f"  MSE  = {mse:.4f}")
    print(f"  RMSE = {rmse:.4f}")
    print(f"  MAE  = {mae:.4f}")
    print(f"  R2   = {r2:.4f}")


# =========================================================
# WEIGHTED MEAN (based on number of observations per fold)
# =========================================================

# 1) Get the number of observations per fold
fold_sizes = []
for _, val_idx in gkf5.split(X_train_s5, yc_train_s5, groups=period_train_s5):
    fold_sizes.append(len(val_idx))

fold_sizes = np.array(fold_sizes)

print("\nFold sizes:", fold_sizes)

# 2) Get the metrics of the best model
fold_mse = np.array([
    -results.loc[best_idx, f"split{i}_test_mse"]
    for i in range(n_splits)
])

fold_rmse = np.array([
    -results.loc[best_idx, f"split{i}_test_rmse"]
    for i in range(n_splits)
])

fold_mae = np.array([
    -results.loc[best_idx, f"split{i}_test_mae"]
    for i in range(n_splits)
])

fold_r2 = np.array([
    results.loc[best_idx, f"split{i}_test_r2"]
    for i in range(n_splits)
])

print("Per-fold MSE :", fold_mse)
print("Per-fold RMSE:", fold_rmse)
print("Per-fold MAE :", fold_mae)
print("Per-fold R2  :", fold_r2)

# 3) Compute the weighted mean of the metrics
weighted_mse = np.average(fold_mse, weights=fold_sizes)
weighted_rmse = np.average(fold_rmse, weights=fold_sizes)
weighted_mae = np.average(fold_mae, weights=fold_sizes)
weighted_r2 = np.average(fold_r2, weights=fold_sizes)

print("\n" + "=" * 60)
print("WEIGHTED MEAN CV SCORES")
print("=" * 60)
print(f"Weighted MSE : {weighted_mse:.4f}")
print(f"Weighted RMSE: {weighted_rmse:.4f}")
print(f"Weighted MAE : {weighted_mae:.4f}")
print(f"Weighted R2  : {weighted_r2:.4f}")

# =========================================================
# PUBLIC TEST SET EVALUATION - SUBSTANCE 5
# =========================================================
X_pub_test_s5 = X_pub_test.loc[ys_pub_test == 5, :]
yc_pub_test_s5 = yc_pub_test.loc[ys_pub_test == 5]

best_reg_s5 = grid.best_estimator_
y_pred_pub_s5 = best_reg_s5.predict(X_pub_test_s5)

print("=" * 60)
print("PUBLIC TEST SET RESULTS - SUBSTANCE 5")
print("=" * 60)
print(f"  MSE : {mean_squared_error(yc_pub_test_s5, y_pred_pub_s5):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(yc_pub_test_s5, y_pred_pub_s5)):.4f}")
print(f"  MAE : {mean_absolute_error(yc_pub_test_s5, y_pred_pub_s5):.4f}")
print(f"  R2  : {r2_score(yc_pub_test_s5, y_pred_pub_s5):.4f}")

Fitting 6 folds for each of 15 candidates, totalling 90 fits
BEST MODEL
Best params: {'model__C': 0.1, 'pca__n_components': 50}
Best mean CV MAE: 12.1259

Mean CV metrics for best model:
  MSE : 359.0247
  RMSE: 18.2190
  MAE : 12.1259
  R2  : 0.7849

PER-FOLD SCORES FOR BEST MODEL
Fold 0:
  MSE  = 677.6701
  RMSE = 26.0321
  MAE  = 14.1869
  R2   = 0.9007
Fold 1:
  MSE  = 180.3820
  RMSE = 13.4306
  MAE  = 8.9693
  R2   = 0.9690
Fold 2:
  MSE  = 274.4033
  RMSE = 16.5651
  MAE  = 9.2290
  R2   = 0.9559
Fold 3:
  MSE  = 477.2780
  RMSE = 21.8467
  MAE  = 13.1276
  R2   = 0.9780
Fold 4:
  MSE  = 429.7031
  RMSE = 20.7293
  MAE  = 17.4871
  R2   = 0.9061
Fold 5:
  MSE  = 114.7119
  RMSE = 10.7104
  MAE  = 9.7556
  R2   = 0.0000

Fold sizes: [606 532 275  70  63  12]
Per-fold MSE : [677.67006142 180.38198731 274.40333265 477.27801671 429.70313315
 114.71194968]
Per-fold RMSE: [26.03209675 13.43063615 16.56512399 21.8466935  20.72928202 10.71036646]
Per-fold MAE : [14.18690666  8.96931956 

# Substance 6

# Ridge Regression - Polynomial Degree 2 Classifier - Standardised Data:

In [68]:
# =========================================================
# RIDGE REGRESSION - POLYNOMIAL DEGREE 2 TRANSFORMATION - STANDARDISED DATA
# SUBSTANCE 6
# =========================================================

# Define the pipeline of data preprocessing and model training
pipe = Pipeline([
    ("poly", PolynomialFeatures(degree=2)), # degree 2 includes squared terms as well as interaction terms
    ("scaler", StandardScaler()),
    ("model", Ridge(random_state=seed)) # Set seed for reproducibility
])

# Define the parameter grid over which we optimise
param_grid = [
    {
        "model__alpha": np.logspace(-2, 4, 60)
    }
]

# Define the regression metrics we are interested in
scoring = {
    "mse": "neg_mean_squared_error",
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2"
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=gkf6, # Choose gkf6 for Substance 6
    scoring=scoring,
    refit="mae",   # Choose the best model based on MAE
    n_jobs=-1,
    verbose=1,
    return_train_score=False
)

grid.fit(X_train_s6, yc_train_s6, groups=period_train_s6)

results = pd.DataFrame(grid.cv_results_)
best_idx = grid.best_index_

# =========================================================
# OVERALL BEST RESULT
# =========================================================
print("=" * 60)
print("BEST MODEL")
print("=" * 60)
print("Best params:", grid.best_params_)

# grid.best_score_ ist hier NEGATIVE RMSE
print(f"Best mean CV MAE: {-grid.best_score_:.4f}")

print("\nMean CV metrics for best model:")
print(f"  MSE : {-results.loc[best_idx, 'mean_test_mse']:.4f}")
print(f"  RMSE: {-results.loc[best_idx, 'mean_test_rmse']:.4f}")
print(f"  MAE : {-results.loc[best_idx, 'mean_test_mae']:.4f}")
print(f"  R2  : {results.loc[best_idx, 'mean_test_r2']:.4f}")

# =========================================================
# PER-FOLD SCORES FOR BEST MODEL
# =========================================================
n_splits_s6 = gkf6.get_n_splits(X_train_s6, yc_train_s6, groups=period_train_s6) 

print("\n" + "=" * 60)
print("PER-FOLD SCORES FOR BEST MODEL")
print("=" * 60)

for i in range(n_splits_s6): # Changed loop range
    mse = -results.loc[best_idx, f"split{i}_test_mse"]
    rmse = -results.loc[best_idx, f"split{i}_test_rmse"]
    mae = -results.loc[best_idx, f"split{i}_test_mae"]
    r2 = results.loc[best_idx, f"split{i}_test_r2"]

    print(f"Fold {i}:")
    print(f"  MSE  = {mse:.4f}")
    print(f"  RMSE = {rmse:.4f}")
    print(f"  MAE  = {mae:.4f}")
    print(f"  R2   = {r2:.4f}")


# =========================================================
# WEIGHTED MEAN (based on number of observations per fold)
# =========================================================

# 1) Find the number of observations per fold
fold_sizes = []
for _, val_idx in gkf6.split(X_train_s6, yc_train_s6, groups=period_train_s6):
    fold_sizes.append(len(val_idx))

fold_sizes = np.array(fold_sizes)

print("\nFold sizes:", fold_sizes)

# 2) Get the results of the best model
fold_mse = np.array([
    -results.loc[best_idx, f"split{i}_test_mse"]
    for i in range(n_splits_s6) 
])

fold_rmse = np.array([
    -results.loc[best_idx, f"split{i}_test_rmse"]
    for i in range(n_splits_s6) 
])

fold_mae = np.array([
    -results.loc[best_idx, f"split{i}_test_mae"]
    for i in range(n_splits_s6) 
])

fold_r2 = np.array([
    results.loc[best_idx, f"split{i}_test_r2"]
    for i in range(n_splits_s6) 
])

print("Per-fold MSE :", fold_mse)
print("Per-fold RMSE:", fold_rmse)
print("Per-fold MAE :", fold_mae)
print("Per-fold R2  :", fold_r2)

# 3) Compute weighted mean
weighted_mse = np.average(fold_mse, weights=fold_sizes)
weighted_rmse = np.average(fold_rmse, weights=fold_sizes)
weighted_mae = np.average(fold_mae, weights=fold_sizes)
weighted_r2 = np.average(fold_r2, weights=fold_sizes)

print("\n" + "=" * 60)
print("WEIGHTED MEAN CV SCORES")
print("=" * 60)
print(f"Weighted MSE : {weighted_mse:.4f}")
print(f"Weighted RMSE: {weighted_rmse:.4f}")
print(f"Weighted MAE : {weighted_mae:.4f}")
print(f"Weighted R2  : {weighted_r2:.4f}")

# =========================================================
# PUBLIC TEST SET EVALUATION - SUBSTANCE 6
# =========================================================
X_pub_test_s6 = X_pub_test.loc[ys_pub_test == 6, :]
yc_pub_test_s6 = yc_pub_test.loc[ys_pub_test == 6]

best_reg_s6 = grid.best_estimator_
y_pred_pub_s6 = best_reg_s6.predict(X_pub_test_s6)

print("=" * 60)
print("PUBLIC TEST SET RESULTS - SUBSTANCE 6")
print("=" * 60)
print(f"  MSE : {mean_squared_error(yc_pub_test_s6, y_pred_pub_s6):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(yc_pub_test_s6, y_pred_pub_s6)):.4f}")
print(f"  MAE : {mean_absolute_error(yc_pub_test_s6, y_pred_pub_s6):.4f}")
print(f"  R2  : {r2_score(yc_pub_test_s6, y_pred_pub_s6):.4f}")

Fitting 3 folds for each of 60 candidates, totalling 180 fits
BEST MODEL
Best params: {'model__alpha': np.float64(235.98334667821936)}
Best mean CV MAE: 14.8718

Mean CV metrics for best model:
  MSE : 470.3371
  RMSE: 21.0619
  MAE : 14.8718
  R2  : -0.0485

PER-FOLD SCORES FOR BEST MODEL
Fold 0:
  MSE  = 240.6131
  RMSE = 15.5117
  MAE  = 4.2723
  R2   = 0.5515
Fold 1:
  MSE  = 781.7334
  RMSE = 27.9595
  MAE  = 22.0743
  R2   = -0.6970
Fold 2:
  MSE  = 388.6647
  RMSE = 19.7146
  MAE  = 18.2688
  R2   = 0.0000

Fold sizes: [467  74   5]
Per-fold MSE : [240.61307349 781.73344905 388.66474602]
Per-fold RMSE: [15.51170763 27.95949658 19.71458207]
Per-fold MAE : [ 4.27227448 22.07429596 18.2687966 ]
Per-fold R2  : [ 0.5515496  -0.69703563  0.        ]

WEIGHTED MEAN CV SCORES
Weighted MSE : 315.3075
Weighted RMSE: 17.2373
Weighted MAE : 6.8132
Weighted R2  : 0.3773
PUBLIC TEST SET RESULTS - SUBSTANCE 6
  MSE : 990.5362
  RMSE: 31.4728
  MAE : 16.9577
  R2  : 0.2741


# Two Stage Process Models:

# Substance Specific Regression Models - Substance Agnostic Model Family:

In [69]:

# =========================================================
# HELPERS -> Make sure everything is numpy to avoid errors due to wrong format
# =========================================================
def to_numpy_2d(x):
    if hasattr(x, "values"):
        x = x.values
    x = np.asarray(x)
    if x.ndim == 1:
        x = x.reshape(-1, 1)
    return x

def to_numpy_1d(x):
    if hasattr(x, "values"):
        x = x.values
    return np.asarray(x).reshape(-1)

# =========================================================
# DATA
# =========================================================
X_train_np = to_numpy_2d(X_train).astype(np.float64)
ys_train_np = to_numpy_1d(ys_train).astype(int)
yc_train_np = to_numpy_1d(yc_train).astype(np.float64)

groups_np = to_numpy_1d(period_train)

# We keep labels in original form 1..6 (will have to change that for PyTorch)
unique_substances = np.sort(np.unique(ys_train_np))
# Check so we don't get confused with the PyTorch labeling of 0 to 5!
assert np.array_equal(unique_substances, np.array([1, 2, 3, 4, 5, 6])), "Expected ys_train labels 1..6" 

# =========================================================
# FIXED CLASSIFIER PIPELINE
# =========================================================
SEED = 123 # Set seed for reproducibility

classifier_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=20)),
    ("model", MLPClassifier(
        hidden_layer_sizes=(128, 96, 64),
        activation="relu",
        alpha=np.float64(0.01),
        learning_rate_init=0.0001,
        solver="adam",
        random_state=SEED, # set seed to ensure reproducibility
        max_iter=3000
    ))
])

# =========================================================
# REGRESSOR PIPELINE FACTORIES
# =========================================================
# We plugged the optimal parameters in we found based on our grid search before
def make_regressor_pipeline(name):
    if name == "svr_rbf":
        return Pipeline([
            ("scaler", StandardScaler()),
            ("model", SVR(
                kernel="rbf",
                C=10,
                gamma=0.001
            ))
        ])

    elif name == "gbr":
        return Pipeline([
            ("scaler", StandardScaler()),
            ("model", GradientBoostingRegressor(
                learning_rate=0.1,
                max_depth=3,
                n_estimators=100,
                subsample=0.8,
                random_state=SEED
            ))
        ])

    elif name == "svr_linear":
        return Pipeline([
            ("scaler", StandardScaler()),
            ("model", SVR(
                kernel="linear",
                C=10
            ))
        ])

    elif name == "tree":
        return Pipeline([
            ("scaler", StandardScaler()),
            ("model", DecisionTreeRegressor(
                max_depth=None,
                min_samples_split=5,
                random_state=SEED
            ))
        ])

    elif name == "svr_linear_pca":
        return Pipeline([
            ("scaler", StandardScaler()),
            ("pca", PCA(n_components = 50)),
            ("model", SVR(
                kernel="linear",
                C=10
            ))
        ])

    elif name == "ridge":
        return Pipeline([
            ("poly", PolynomialFeatures(degree=2)),
            ("scaler", StandardScaler()),
            ("model", Ridge(
                alpha=np.float64(235.98334667821936),
                random_state=SEED
            ))
        ])

    else:
        raise ValueError(f"Unknown regressor name: {name}")


regressor_names = [
    "svr_rbf",
    "gbr",
    "svr_linear",
    "tree",
    "svr_linear_pca",
    "ridge"
]

# =========================================================
# Period CV ROUTINE
# =========================================================
def run_period_cv_hard_routed(
    X,
    ys,
    yc,
    groups,
    classifier_template,
    regressor_name,
    verbose=True
):
    logo = LeaveOneGroupOut()

    fold_rows = []

    all_true_y = []
    all_pred_y = []

    all_true_s = []
    all_pred_s = []

    for fold_idx, (tr_idx, va_idx) in enumerate(logo.split(X, ys, groups=groups), start=1):
        X_tr, X_va = X[tr_idx], X[va_idx]
        ys_tr, ys_va = ys[tr_idx], ys[va_idx]
        yc_tr, yc_va = yc[tr_idx], yc[va_idx]

        val_periods = np.unique(groups[va_idx])

        # -------------------------------------------------
        # Step 1: fit classifier on full fold-train
        # -------------------------------------------------
        clf = clone(classifier_template)
        clf.fit(X_tr, ys_tr)

        ys_pred_va = clf.predict(X_va)

        # -------------------------------------------------
        # Step 2: fit one regressor per TRUE substance on fold-train
        # -------------------------------------------------
        substance_regressors = {}

        for s in unique_substances:
            mask_s = (ys_tr == s)
            X_tr_s = X_tr[mask_s]
            yc_tr_s = yc_tr[mask_s]

            if X_tr_s.shape[0] == 0:
                raise ValueError(f"No training samples for substance {s} in fold {fold_idx}.")

            reg = make_regressor_pipeline(regressor_name)
            reg.fit(X_tr_s, yc_tr_s)
            substance_regressors[s] = reg

        # -------------------------------------------------
        # Step 3: hard routing via PREDICTED class on validation fold
        # -------------------------------------------------
        yc_pred_va = np.empty_like(yc_va, dtype=np.float64)

        for s in unique_substances:
            route_mask = (ys_pred_va == s)

            if np.any(route_mask):
                yc_pred_va[route_mask] = substance_regressors[s].predict(X_va[route_mask])

        # -------------------------------------------------
        # Step 4: metrics
        # -------------------------------------------------
        fold_mae = mean_absolute_error(yc_va, yc_pred_va)
        fold_acc = accuracy_score(ys_va, ys_pred_va)
        fold_bal_acc = balanced_accuracy_score(ys_va, ys_pred_va)
        fold_f1 = f1_score(ys_va, ys_pred_va, average="macro")

        fold_rows.append({
            "regressor_family": regressor_name,
            "fold": fold_idx,
            "val_period": str(val_periods.tolist()),
            "n_val": len(va_idx),
            "classification_accuracy": fold_acc,
            "classification_balanced_accuracy": fold_bal_acc,
            "classification_f1_macro": fold_f1,
            "hard_routed_mae": fold_mae
        })

        all_true_y.append(yc_va)
        all_pred_y.append(yc_pred_va)

        all_true_s.append(ys_va)
        all_pred_s.append(ys_pred_va)

        if verbose:
            print("=" * 100)
            print(f"REGRESSOR: {regressor_name} | FOLD {fold_idx} | VAL PERIOD {val_periods.tolist()}")
            print("=" * 100)
            print(f"n_val                            : {len(va_idx)}")
            print(f"Classification Accuracy          : {fold_acc:.6f}")
            print(f"Classification Balanced Accuracy : {fold_bal_acc:.6f}")
            print(f"Classification F1 Macro          : {fold_f1:.6f}")
            print(f"Hard-Routed MAE                  : {fold_mae:.6f}")
            print()

    fold_df = pd.DataFrame(fold_rows)

    # unweighted mean over folds
    cv_mae_mean = fold_df["hard_routed_mae"].mean()

    # weighted mean over folds
    weights = fold_df["n_val"].values
    cv_mae_weighted_mean = np.sum(fold_df["hard_routed_mae"].values * weights) / np.sum(weights)

    # pooled OOF MAE
    all_true_y = np.concatenate(all_true_y)
    all_pred_y = np.concatenate(all_pred_y)
    pooled_mae = mean_absolute_error(all_true_y, all_pred_y)

    # pooled classifier metrics
    all_true_s = np.concatenate(all_true_s)
    all_pred_s = np.concatenate(all_pred_s)
    pooled_acc = accuracy_score(all_true_s, all_pred_s)
    pooled_bal_acc = balanced_accuracy_score(all_true_s, all_pred_s)
    pooled_f1 = f1_score(all_true_s, all_pred_s, average="macro")

    summary_row = {
        "regressor_family": regressor_name,
        "cv_unweighted_mean_hard_routed_mae": cv_mae_mean,
        "cv_weighted_mean_hard_routed_mae": cv_mae_weighted_mean,
        "cv_pooled_oof_hard_routed_mae": pooled_mae,
        "cv_pooled_classifier_accuracy": pooled_acc,
        "cv_pooled_classifier_balanced_accuracy": pooled_bal_acc,
        "cv_pooled_classifier_f1_macro": pooled_f1
    }

    return fold_df, summary_row


# =========================================================
# RUN ALL REGRESSOR FAMILIES
# =========================================================
all_fold_results = []
all_summary_rows = []

for reg_name in regressor_names:
    print("\n" + "#" * 120)
    print(f"RUNNING REGRESSOR FAMILY: {reg_name}")
    print("#" * 120)

    fold_df_reg, summary_row_reg = run_period_cv_hard_routed(
        X=X_train_np,
        ys=ys_train_np,
        yc=yc_train_np,
        groups=groups_np,
        classifier_template=classifier_pipe,
        regressor_name=reg_name,
        verbose=True
    )

    all_fold_results.append(fold_df_reg)
    all_summary_rows.append(summary_row_reg)

    print("\nFOLD RESULTS")
    print(fold_df_reg)

    print("\nSUMMARY")
    print(pd.DataFrame([summary_row_reg]).T)

all_fold_results_df = pd.concat(all_fold_results, ignore_index=True)
all_summary_df = pd.DataFrame(all_summary_rows)

print("\n" + "=" * 120)
print("ALL FOLD RESULTS")
print("=" * 120)
print(all_fold_results_df)

print("\n" + "=" * 120)
print("FINAL SUMMARY ACROSS REGRESSOR FAMILIES")
print("=" * 120)
print(all_summary_df.sort_values("cv_weighted_mean_hard_routed_mae"))

# =========================================================
# BEST REGRESSOR BY WEIGHTED HARD-ROUTED MAE
# =========================================================
best_row = all_summary_df.sort_values("cv_weighted_mean_hard_routed_mae").iloc[0]

print("\n" + "=" * 120)
print("BEST REGRESSOR FAMILY")
print("=" * 120)
print(best_row)


########################################################################################################################
RUNNING REGRESSOR FAMILY: svr_rbf
########################################################################################################################
REGRESSOR: svr_rbf | FOLD 1 | VAL PERIOD [1]
n_val                            : 445
Classification Accuracy          : 0.750562
Classification Balanced Accuracy : 0.656073
Classification F1 Macro          : 0.604168
Hard-Routed MAE                  : 82.837695

REGRESSOR: svr_rbf | FOLD 2 | VAL PERIOD [2]
n_val                            : 1244
Classification Accuracy          : 0.959807
Classification Balanced Accuracy : 0.960752
Classification F1 Macro          : 0.937958
Hard-Routed MAE                  : 24.874267

REGRESSOR: svr_rbf | FOLD 3 | VAL PERIOD [3]
n_val                            : 1586
Classification Accuracy          : 0.983607
Classification Balanced Accuracy : 0.983246
Classification F1 Macro  

# Substance Specific Model Family:

In [70]:

# =========================================================
# HELPER -> To easily compute regression metrics
# =========================================================
def compute_reg_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    try:
        r2 = r2_score(y_true, y_pred)
    except Exception:
        r2 = np.nan

    return {
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "r2": r2
    }


# =========================================================
# FIXED CLASSIFIER PIPELINE
# =========================================================
SEED = 42 # Set seed to ensure reproducibility

classifier_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=20)),
    ("model", MLPClassifier(
        hidden_layer_sizes=(128, 96, 64),
        activation="relu",
        alpha=np.float64(0.01),
        learning_rate_init=0.0001,
        solver="adam",
        random_state=SEED, # Set seed to ensure reproducibility
        max_iter=3000
    ))
])


# =========================================================
# SUBSTANCE-SPECIFIC OPTIMAL REGRESSORS
# ---------------------------------------------------------
#   1 -> SVR (RBF)
#   2 -> Gradient Boosting Regressor
#   3 -> SVR (Linear)
#   4 -> Decision Tree Regressor
#   5 -> SVR (Linear)
#   6 -> Ridge
# =========================================================
def make_optimal_regressor_for_substance(substance):
    if substance == 1:
        # SVR (RBF), standardized
        return Pipeline([
            ("scaler", StandardScaler()),
            ("model", SVR(
                kernel="rbf",
                C=10,
                gamma=0.001
            ))
        ])

    elif substance == 2:
        # Gradient Boosting
        return Pipeline([
            ("scaler", StandardScaler()),
            ("model", GradientBoostingRegressor(
                learning_rate=0.1,
                max_depth=3,
                n_estimators=100,
                subsample=0.8,
                random_state=SEED
            ))
        ])

    elif substance == 3:
        # SVR (Linear), standardized
        return Pipeline([
            ("scaler", StandardScaler()),
            ("model", SVR(
                kernel="linear",
                C=10
            ))
        ])

    elif substance == 4:
        # Regression Tree
        return Pipeline([
            ("scaler", StandardScaler()),
            ("model", DecisionTreeRegressor(
                max_depth=None,
                min_samples_split=5,
                random_state=SEED
            ))
        ])

    elif substance == 5:
        # SVR (Linear), PCA
        return Pipeline([
            ("scaler", StandardScaler()),
            ("pca", PCA(n_components=50)),
            ("model", SVR(
                kernel="linear",
                C=0.1
            ))
        ])

    elif substance == 6:
        # Ridge, Polynomial
        return Pipeline([
            ("poly", PolynomialFeatures(degree=2)),
            ("scaler", StandardScaler()),
            ("model", Ridge(
                alpha=np.float64(235.98334667821936),
                random_state=SEED
            ))
        ])

    else:
        raise ValueError(f"Unknown substance: {substance}")


# =========================================================
# CORE CV ROUTINE
# =========================================================
def run_period_cv_hard_routed_optimal_per_substance(
    X,
    ys,
    yc,
    groups,
    classifier_template,
    verbose=True
):
    logo = LeaveOneGroupOut()

    fold_rows = []

    all_true_y = []
    all_pred_y = []

    all_true_s = []
    all_pred_s = []

    for fold_idx, (tr_idx, va_idx) in enumerate(logo.split(X, ys, groups=groups), start=1):
        X_tr, X_va = X[tr_idx], X[va_idx]
        ys_tr, ys_va = ys[tr_idx], ys[va_idx]
        yc_tr, yc_va = yc[tr_idx], yc[va_idx]

        val_periods = np.unique(groups[va_idx])

        # -------------------------------------------------
        # Step 1: classification
        # -------------------------------------------------
        clf = clone(classifier_template)
        clf.fit(X_tr, ys_tr)

        ys_pred_va = clf.predict(X_va)

        # -------------------------------------------------
        # Step 2: fit optimal regressor for each TRUE substance
        # -------------------------------------------------
        substance_regressors = {}

        for s in unique_substances:
            mask_s = (ys_tr == s)
            X_tr_s = X_tr[mask_s]
            yc_tr_s = yc_tr[mask_s]

            if X_tr_s.shape[0] == 0:
                raise ValueError(f"No training samples for substance {s} in fold {fold_idx}.")

            reg = make_optimal_regressor_for_substance(s)
            reg.fit(X_tr_s, yc_tr_s)
            substance_regressors[s] = reg

        # -------------------------------------------------
        # Step 3: hard routing via PREDICTED class
        # -------------------------------------------------
        yc_pred_va = np.empty_like(yc_va, dtype=np.float64)

        for s in unique_substances:
            route_mask = (ys_pred_va == s)
            if np.any(route_mask):
                yc_pred_va[route_mask] = substance_regressors[s].predict(X_va[route_mask])

        # -------------------------------------------------
        # Step 4: metrics
        # -------------------------------------------------
        class_acc = accuracy_score(ys_va, ys_pred_va)
        class_bal_acc = balanced_accuracy_score(ys_va, ys_pred_va)
        class_f1 = f1_score(ys_va, ys_pred_va, average="macro")

        reg_metrics = compute_reg_metrics(yc_va, yc_pred_va)

        fold_rows.append({
            "fold": fold_idx,
            "val_period": str(val_periods.tolist()),
            "n_val": len(va_idx),
            "classification_accuracy": class_acc,
            "classification_balanced_accuracy": class_bal_acc,
            "classification_f1_macro": class_f1,
            "hard_routed_mse": reg_metrics["mse"],
            "hard_routed_rmse": reg_metrics["rmse"],
            "hard_routed_mae": reg_metrics["mae"],
            "hard_routed_r2": reg_metrics["r2"],
        })

        all_true_y.append(yc_va)
        all_pred_y.append(yc_pred_va)

        all_true_s.append(ys_va)
        all_pred_s.append(ys_pred_va)

        if verbose:
            print("=" * 110)
            print(f"OPTIMAL PER-SUBSTANCE REGRESSORS | FOLD {fold_idx} | VAL PERIOD {val_periods.tolist()}")
            print("=" * 110)
            print(f"n_val                            : {len(va_idx)}")
            print(f"Classification Accuracy          : {class_acc:.6f}")
            print(f"Classification Balanced Accuracy : {class_bal_acc:.6f}")
            print(f"Classification F1 Macro          : {class_f1:.6f}")
            print(f"Hard-Routed MSE                  : {reg_metrics['mse']:.6f}")
            print(f"Hard-Routed RMSE                 : {reg_metrics['rmse']:.6f}")
            print(f"Hard-Routed MAE                  : {reg_metrics['mae']:.6f}")
            print(f"Hard-Routed R2                   : {reg_metrics['r2']:.6f}")
            print()

    fold_df = pd.DataFrame(fold_rows)

    # unweighted means
    summary = {
        "cv_unweighted_mean_classification_accuracy":
            fold_df["classification_accuracy"].mean(),
        "cv_unweighted_mean_classification_balanced_accuracy":
            fold_df["classification_balanced_accuracy"].mean(),
        "cv_unweighted_mean_classification_f1_macro":
            fold_df["classification_f1_macro"].mean(),
        "cv_unweighted_mean_hard_routed_mse":
            fold_df["hard_routed_mse"].mean(),
        "cv_unweighted_mean_hard_routed_rmse":
            fold_df["hard_routed_rmse"].mean(),
        "cv_unweighted_mean_hard_routed_mae":
            fold_df["hard_routed_mae"].mean(),
        "cv_unweighted_mean_hard_routed_r2":
            fold_df["hard_routed_r2"].mean(),
    }

    # weighted means by fold size
    weights = fold_df["n_val"].values
    summary.update({
        "cv_weighted_mean_classification_accuracy":
            np.sum(fold_df["classification_accuracy"].values * weights) / np.sum(weights),
        "cv_weighted_mean_classification_balanced_accuracy":
            np.sum(fold_df["classification_balanced_accuracy"].values * weights) / np.sum(weights),
        "cv_weighted_mean_classification_f1_macro":
            np.sum(fold_df["classification_f1_macro"].values * weights) / np.sum(weights),
        "cv_weighted_mean_hard_routed_mse":
            np.sum(fold_df["hard_routed_mse"].values * weights) / np.sum(weights),
        "cv_weighted_mean_hard_routed_rmse":
            np.sum(fold_df["hard_routed_rmse"].values * weights) / np.sum(weights),
        "cv_weighted_mean_hard_routed_mae":
            np.sum(fold_df["hard_routed_mae"].values * weights) / np.sum(weights),
        "cv_weighted_mean_hard_routed_r2":
            np.sum(fold_df["hard_routed_r2"].values * weights) / np.sum(weights),
    })

    # pooled OOF metrics
    all_true_y = np.concatenate(all_true_y)
    all_pred_y = np.concatenate(all_pred_y)

    all_true_s = np.concatenate(all_true_s)
    all_pred_s = np.concatenate(all_pred_s)

    pooled_reg = compute_reg_metrics(all_true_y, all_pred_y)

    summary.update({
        "cv_pooled_oof_classifier_accuracy":
            accuracy_score(all_true_s, all_pred_s),
        "cv_pooled_oof_classifier_balanced_accuracy":
            balanced_accuracy_score(all_true_s, all_pred_s),
        "cv_pooled_oof_classifier_f1_macro":
            f1_score(all_true_s, all_pred_s, average="macro"),
        "cv_pooled_oof_hard_routed_mse":
            pooled_reg["mse"],
        "cv_pooled_oof_hard_routed_rmse":
            pooled_reg["rmse"],
        "cv_pooled_oof_hard_routed_mae":
            pooled_reg["mae"],
        "cv_pooled_oof_hard_routed_r2":
            pooled_reg["r2"],
    })

    return fold_df, summary, {
        "all_true_s": all_true_s,
        "all_pred_s": all_pred_s,
        "all_true_y": all_true_y,
        "all_pred_y": all_pred_y,
        "confusion_matrix": confusion_matrix(all_true_s, all_pred_s),
        "confusion_matrix_normalized": confusion_matrix(all_true_s, all_pred_s, normalize="true"),
    }


# =========================================================
# RUN
# =========================================================
fold_df, summary_row, detail = run_period_cv_hard_routed_optimal_per_substance(
    X=X_train_np,
    ys=ys_train_np,
    yc=yc_train_np,
    groups=groups_np,
    classifier_template=classifier_pipe,
    verbose=True
)

print("\n" + "=" * 120)
print("FOLD RESULTS")
print("=" * 120)
print(fold_df)

print("\n" + "=" * 120)
print("SUMMARY")
print("=" * 120)
print(pd.DataFrame([summary_row]).T)

print("\n" + "=" * 120)
print("CONFUSION MATRIX")
print("=" * 120)
print(detail["confusion_matrix"])

print("\n" + "=" * 120)
print("NORMALIZED CONFUSION MATRIX")
print("=" * 120)
print(detail["confusion_matrix_normalized"])

OPTIMAL PER-SUBSTANCE REGRESSORS | FOLD 1 | VAL PERIOD [1]
n_val                            : 445
Classification Accuracy          : 0.820225
Classification Balanced Accuracy : 0.724853
Classification F1 Macro          : 0.699575
Hard-Routed MSE                  : 13828.768312
Hard-Routed RMSE                 : 117.595784
Hard-Routed MAE                  : 49.500255
Hard-Routed R2                   : 0.724242

OPTIMAL PER-SUBSTANCE REGRESSORS | FOLD 2 | VAL PERIOD [2]
n_val                            : 1244
Classification Accuracy          : 0.975080
Classification Balanced Accuracy : 0.977992
Classification F1 Macro          : 0.960037
Hard-Routed MSE                  : 5173.315709
Hard-Routed RMSE                 : 71.925765
Hard-Routed MAE                  : 26.221563
Hard-Routed R2                   : 0.556248

OPTIMAL PER-SUBSTANCE REGRESSORS | FOLD 3 | VAL PERIOD [3]
n_val                            : 1586
Classification Accuracy          : 0.983607
Classification Balanced Accura

# Final Model Section

# Joint Optimisation Model

In [71]:
# =========================================================
# PERIOD-BASED CV FOR JOINT OPTIMISATION MODEL
# =========================================================

# =========================================================
# 1) SETTINGS
# =========================================================
SEED = 123 # Set seed to ensure reproducibility 

BATCH_SIZE = 128
USE_TORCH_DETERMINISTIC = True # To ensure reproducibility

NUM_CLASSES = 6
PCA40 = 40

# Number of Bootstrap Samples for Bagging
N_BOOTSTRAPS = 5

# Basic Architecture Elements
CLASS_HIDDEN = (128, 96, 64)
REG_HIDDEN = (160, 128, 96, 64)
HEAD_HIDDEN = (32, 16)

CLASS_DROPOUT = 0.08 # Dropout for Regularisation
REG_DROPOUT = 0.05
HEAD_DROPOUT = 0.05

EPOCHS = 30 # Number of Epochs we train for
LR = 3e-4
WEIGHT_DECAY = 2e-4 # Weight Decay for regularisation
LAMBDA_CLASS = 8.0 # Increased weight for the classification loss
LAMBDA_REG = 1.0
LABEL_SMOOTHING = 0.03
USE_CLASS_WEIGHTS = True # to deal with difficult classes (we saw some substances are harder to classify)
USE_REG_CLASS_WEIGHTS = False
USE_HUBER = True # We use Huber Loss and Cross Entropy Loss
HUBER_BETA = 1.0
CLIP_GRAD = 5.0 # Gradient Clipping to prevent exploding gradients as the network is relatively deep


# =========================================================
# 2) REPRODUCIBILITY -> Set seed for all sources of Python and PyTorch
# =========================================================
def set_global_seed(seed: int, deterministic: bool = True):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        try:
            torch.use_deterministic_algorithms(True)
        except Exception:
            pass

    return torch.device("cuda" if torch.cuda.is_available() else "cpu") # Try to do computations on GPU

# =========================================================
# 3) HELPERS -> Ensure that there are no errors due to data format
# =========================================================
def to_numpy_2d(x):
    if hasattr(x, "values"):
        x = x.values
    x = np.asarray(x)
    if x.ndim == 1:
        x = x.reshape(-1, 1)
    return x


def to_numpy_1d(x):
    if hasattr(x, "values"):
        x = x.values
    return np.asarray(x).reshape(-1)


# =========================================================
# 4) LOAD TRAIN DATA
# =========================================================
X_train_np = to_numpy_2d(X_train).astype(np.float32)
ys_train_np = to_numpy_1d(ys_train).astype(int) - 1
yc_train_np = to_numpy_1d(yc_train).astype(np.float32)

# Check that we use this time substance labels 0 to 5, as that is what PyTorch expects
assert ys_train_np.min() == 0 and ys_train_np.max() == 5, "ys_train must become 0..5"

period_train_np = to_numpy_1d(period_train)

# Check we didn't make a mistake when transforming the data
assert len(period_train_np) == len(X_train_np), "period/group vector must match X_train length"

# =========================================================
# 5) DATASET -> We now need tensors for PyTorch
# =========================================================
class MultiInputDataset(Dataset):
    def __init__(self, x_class, x_reg, ys, yc):
        self.x_class = torch.tensor(x_class, dtype=torch.float32)
        self.x_reg = torch.tensor(x_reg, dtype=torch.float32)
        self.ys = torch.tensor(ys, dtype=torch.long)
        self.yc = torch.tensor(yc, dtype=torch.float32)

    def __len__(self):
        return len(self.ys)

    def __getitem__(self, idx):
        return self.x_class[idx], self.x_reg[idx], self.ys[idx], self.yc[idx]


# =========================================================
# 6) LOSSES -> Define Losses
# =========================================================
class WeightedMSELoss(nn.Module):
    def __init__(self, reduction="mean"):
        super().__init__()
        self.reduction = reduction

    def forward(self, pred, target, weights=None):
        loss = (pred - target) ** 2
        if weights is not None:
            loss = loss * weights

        if self.reduction == "mean":
            if weights is not None:
                return loss.sum() / (weights.sum() + 1e-8)
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        return loss

# We decided to go with the WeightedHuberLoss as it reflects our interest in MAE rather than MSE
class WeightedHuberLoss(nn.Module):
    def __init__(self, beta=1.0, reduction="mean"):
        super().__init__()
        self.beta = beta
        self.reduction = reduction

    def forward(self, pred, target, weights=None):
        diff = pred - target
        abs_diff = torch.abs(diff)

        loss = torch.where(
            abs_diff < self.beta,
            0.5 * (diff ** 2) / self.beta,
            abs_diff - 0.5 * self.beta
        )

        if weights is not None:
            loss = loss * weights

        if self.reduction == "mean":
            if weights is not None:
                return loss.sum() / (weights.sum() + 1e-8)
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        return loss


# =========================================================
# 7) MODEL
# =========================================================
def make_mlp(input_dim, hidden_dims, dropout=0.0, output_dim=None):
    layers = []
    prev = input_dim
    for h in hidden_dims:
        layers.append(nn.Linear(prev, h))
        layers.append(nn.ReLU())
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        prev = h
    if output_dim is not None:
        layers.append(nn.Linear(prev, output_dim))
    return nn.Sequential(*layers)

# We decided to use an architecture in which the regression part, branches of into 6 different regression heads
# this is such that we can use the similarity between substances, but also model substance specific behaviour
# That is a mixture of substance agnostic and substance specific approach
class RegressionHead(nn.Module):
    def __init__(self, input_dim, hidden_dims=(32, 16), dropout=0.0):
        super().__init__()
        self.net = make_mlp(input_dim, hidden_dims, dropout=dropout, output_dim=1)

    def forward(self, x):
        return self.net(x)


class SeparateBranchMTL(nn.Module):
    def __init__(
        self,
        class_input_dim,
        reg_input_dim,
        class_hidden=(128, 96, 64),
        reg_hidden=(160, 128, 96, 64),
        head_hidden=(32, 16),
        num_classes=6,
        class_dropout=0.08,
        reg_dropout=0.05,
        head_dropout=0.05
    ):
        super().__init__()
        self.num_classes = num_classes

        # Seperate Classification Branch <- gets the scaled data
        self.class_net = make_mlp(class_input_dim, class_hidden, dropout=class_dropout)
        # Seperate Regression Branch <- gets PCA transformed data
        self.class_out = nn.Linear(class_hidden[-1], num_classes)

        self.reg_trunk = make_mlp(reg_input_dim, reg_hidden, dropout=reg_dropout)

        # Add substance specific regression heads, that branch of the regression trunk
        self.reg_heads = nn.ModuleList([
            RegressionHead(reg_hidden[-1], hidden_dims=head_hidden, dropout=head_dropout)
            for _ in range(num_classes)
        ])

    # Define the forward pass of our Neural Network
    def forward(self, x_class, x_reg):
        class_feat = self.class_net(x_class)
        logits = self.class_out(class_feat)

        reg_feat = self.reg_trunk(x_reg)
        reg_outputs = torch.cat([head(reg_feat) for head in self.reg_heads], dim=1)

        return logits, reg_outputs


# =========================================================
# 8) FEATURE FACTORY
# =========================================================
class FeatureFactory:
    def __init__(self, seed=42):
        self.seed = seed
        self.x_scaler = None
        self.y_scaler = None
        self.pca40 = None

    def fit(self, X_train, ys_train, yc_train):
        self.x_scaler = StandardScaler()
        Xs = self.x_scaler.fit_transform(X_train)

        self.y_scaler = StandardScaler()
        self.y_scaler.fit(yc_train.reshape(-1, 1))

        n40 = min(PCA40, Xs.shape[1], Xs.shape[0])
        self.pca40 = PCA(n_components=n40, random_state=self.seed)
        self.pca40.fit(Xs) # PCA on scaled Features to give this the regression part
        return self

    # We scale the yc as this tends to make the optimisation problem easier (better conditioned)
    def transform_y(self, yc):
        return self.y_scaler.transform(np.asarray(yc).reshape(-1, 1)).reshape(-1)

    # Define inverse transform such that we get our metrics on the correct scale
    def inverse_transform_y(self, yc_std):
        return self.y_scaler.inverse_transform(np.asarray(yc_std).reshape(-1, 1)).reshape(-1)

    def get_feature(self, X, name):
        Xs = self.x_scaler.transform(X)

        if name == "raw":
            return Xs.astype(np.float32)
        elif name == "pca40":
            return self.pca40.transform(Xs).astype(np.float32)
        else:
            raise ValueError(f"Unknown feature name: {name}")


# =========================================================
# 9) TRAINING HELPERS
# =========================================================
# Create helpers so I can easily change weights for difficult substances later
def make_class_weights(ys, num_classes=6, device="cpu"):
    counts = np.bincount(ys, minlength=num_classes).astype(np.float32)
    counts[counts == 0] = 1.0
    weights = counts.sum() / (num_classes * counts)
    return torch.tensor(weights, dtype=torch.float32, device=device)


def make_reg_class_weights(ys, num_classes=6, device="cpu", power=0.5):
    counts = np.bincount(ys, minlength=num_classes).astype(np.float32)
    counts[counts == 0] = 1.0
    weights = (counts.sum() / (num_classes * counts)) ** power
    return torch.tensor(weights, dtype=torch.float32, device=device)


# Define training procedure
def train_single_model_separate(model, train_loader, y_scaler, train_ys, device):
    model = model.to(device)

    class_weights = None
    if USE_CLASS_WEIGHTS:
        class_weights = make_class_weights(train_ys, num_classes=NUM_CLASSES, device=device)

    reg_class_weights = None
    if USE_REG_CLASS_WEIGHTS:
        reg_class_weights = make_reg_class_weights(train_ys, num_classes=NUM_CLASSES, device=device)

    # We use CrossEntropyLoss for Classification as it relates to Maximum Likelihood Estimation and KL-Divergence Minimisation
    class_criterion = nn.CrossEntropyLoss(
        weight=class_weights,
        label_smoothing=LABEL_SMOOTHING
    )

    # We use (weighted )Huber Loss for Regression <- this reflects our choice to use MAE instead of MSE as deciding metric
    reg_criterion = WeightedHuberLoss(beta=HUBER_BETA) if USE_HUBER else WeightedMSELoss()

    optimizer = torch.optim.AdamW( # We use AdamW as optimiser, it allows for adaptive step sizes and deals better with weight decay
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )

    for epoch in range(EPOCHS):
        model.train()

        for x_class, x_reg, ys_b, yc_b in train_loader:
            x_class = x_class.to(device)
            x_reg = x_reg.to(device)
            ys_b = ys_b.to(device)

            optimizer.zero_grad()

            logits, reg_outputs = model(x_class, x_reg)

            loss_class = class_criterion(logits, ys_b) # Classification Loss

            batch_idx = torch.arange(x_class.size(0), device=device)
            pred_c_true_std = reg_outputs[batch_idx, ys_b]

            reg_weights_batch = None
            if reg_class_weights is not None:
                reg_weights_batch = reg_class_weights[ys_b]

            yc_std = torch.tensor(
                y_scaler.transform_y(yc_b.detach().cpu().numpy()),
                dtype=torch.float32,
                device=device
            )

            loss_reg = reg_criterion(pred_c_true_std, yc_std, weights=reg_weights_batch) # Regression Loss
            loss = LAMBDA_CLASS * loss_class + LAMBDA_REG * loss_reg # Total Loss <- "weighted" sum of classification and regression loss

            loss.backward()
            if CLIP_GRAD is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD)
            optimizer.step()

    return model


# =========================================================
# 10) METRICS / EVAL
# =========================================================
# Compute the metrics we are generally interested in
def compute_metrics(y_true_s, y_pred_s, y_true_c, y_pred_c):
    mse = mean_squared_error(y_true_c, y_pred_c)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true_c, y_pred_c)
    try:
        r2 = r2_score(y_true_c, y_pred_c)
    except Exception:
        r2 = np.nan

    return {
        "accuracy": accuracy_score(y_true_s, y_pred_s),
        "balanced_accuracy": balanced_accuracy_score(y_true_s, y_pred_s),
        "f1_macro": f1_score(y_true_s, y_pred_s, average="macro"),
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
    }


# This is important, we use an ensemble approach that is we take the mean of the predictions/probs to reduce variance
# As the model is rather deep and has many parameters, this seems like a suitable form of variance reduction
def evaluate_ensemble_separate(models, loader, y_scaler, device):
    for m in models:
        m.eval()

    all_true_s = []
    all_pred_s = []
    all_true_c = []
    all_pred_c_hard = []
    all_pred_c_soft = []
    all_pred_c_true = []

    with torch.no_grad():
        for x_class, x_reg, ys_b, yc_b in loader:
            x_class = x_class.to(device)
            x_reg = x_reg.to(device)
            ys_b = ys_b.to(device)

            prob_list = []
            reg_list = []

            for model in models:
                logits, reg_outputs = model(x_class, x_reg)
                probs = torch.softmax(logits, dim=1)
                prob_list.append(probs.unsqueeze(0))
                reg_list.append(reg_outputs.unsqueeze(0))

            # Mean of probs for classification
            mean_probs = torch.mean(torch.cat(prob_list, dim=0), dim=0)
            # Mean of Predictions (Regression)
            mean_reg = torch.mean(torch.cat(reg_list, dim=0), dim=0)

            pred_s = torch.argmax(mean_probs, dim=1)
            batch_idx = torch.arange(x_class.size(0), device=device)

            pred_c_hard_std = mean_reg[batch_idx, pred_s]
            pred_c_true_std = mean_reg[batch_idx, ys_b]
            pred_c_soft_std = torch.sum(mean_probs * mean_reg, dim=1)

            pred_c_hard = y_scaler.inverse_transform_y(pred_c_hard_std.cpu().numpy())
            pred_c_true = y_scaler.inverse_transform_y(pred_c_true_std.cpu().numpy())
            pred_c_soft = y_scaler.inverse_transform_y(pred_c_soft_std.cpu().numpy())

            all_true_s.extend(ys_b.cpu().numpy())
            all_pred_s.extend(pred_s.cpu().numpy())
            all_true_c.extend(yc_b.cpu().numpy())
            all_pred_c_hard.extend(pred_c_hard)
            all_pred_c_soft.extend(pred_c_soft)
            all_pred_c_true.extend(pred_c_true)

    all_true_s = np.array(all_true_s)
    all_pred_s = np.array(all_pred_s)
    all_true_c = np.array(all_true_c)
    all_pred_c_hard = np.array(all_pred_c_hard)
    all_pred_c_soft = np.array(all_pred_c_soft)
    all_pred_c_true = np.array(all_pred_c_true)

    return {
        "all_true_s": all_true_s,
        "all_pred_s": all_pred_s,
        "all_true_c": all_true_c,
        "all_pred_c_hard": all_pred_c_hard,
        "all_pred_c_soft": all_pred_c_soft,
        "all_pred_c_true": all_pred_c_true,
        "hard": compute_metrics(all_true_s, all_pred_s, all_true_c, all_pred_c_hard),
        "soft": compute_metrics(all_true_s, all_pred_s, all_true_c, all_pred_c_soft),
        "true": compute_metrics(all_true_s, all_pred_s, all_true_c, all_pred_c_true),
    }


# =========================================================
# 11) BUILD HELPERS
# =========================================================
def build_separate_datasets(factory, X_tr, ys_tr, yc_tr, X_va, ys_va, yc_va):
    xclass_tr = factory.get_feature(X_tr, "raw")
    xreg_tr = factory.get_feature(X_tr, "pca40")

    xclass_va = factory.get_feature(X_va, "raw")
    xreg_va = factory.get_feature(X_va, "pca40")

    ds_tr = MultiInputDataset(xclass_tr, xreg_tr, ys_tr, yc_tr)
    ds_va = MultiInputDataset(xclass_va, xreg_va, ys_va, yc_va)

    return ds_tr, ds_va, xclass_tr.shape[1], xreg_tr.shape[1]


def build_separate_model(class_dim, reg_dim):
    return SeparateBranchMTL(
        class_input_dim=class_dim,
        reg_input_dim=reg_dim,
        class_hidden=CLASS_HIDDEN,
        reg_hidden=REG_HIDDEN,
        head_hidden=HEAD_HIDDEN,
        num_classes=NUM_CLASSES,
        class_dropout=CLASS_DROPOUT,
        reg_dropout=REG_DROPOUT,
        head_dropout=HEAD_DROPOUT,
    )


# =========================================================
# 12) PERIOD-CV
# =========================================================
def run_period_cv():
    logo = LeaveOneGroupOut()
    fold_rows = []

    for fold_idx, (tr_idx, va_idx) in enumerate(logo.split(X_train_np, ys_train_np, groups=period_train_np), start=1):
        fold_seed = SEED + 1000 * fold_idx # Make sure it's reproducible <- have seperate SEED every time
        set_global_seed(fold_seed, deterministic=USE_TORCH_DETERMINISTIC)

        X_tr, X_va = X_train_np[tr_idx], X_train_np[va_idx]
        ys_tr, ys_va = ys_train_np[tr_idx], ys_train_np[va_idx]
        yc_tr, yc_va = yc_train_np[tr_idx], yc_train_np[va_idx]
        val_period = np.unique(period_train_np[va_idx]).tolist()

        factory = FeatureFactory(seed=fold_seed)
        factory.fit(X_tr, ys_tr, yc_tr)

        ds_tr, ds_va, class_dim, reg_dim = build_separate_datasets(
            factory, X_tr, ys_tr, yc_tr, X_va, ys_va, yc_va
        )

        va_loader = DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False)

        models = []
        n_train = len(ds_tr)

        # Run the bootstrap <- That is draw bootstrap samples, and take the average of the outputs of the models afterwards
        for b in range(N_BOOTSTRAPS):
            boot_seed = fold_seed + 10000 + b # Ensure reproducibility, set its own seed per run
            rng = np.random.default_rng(boot_seed)
            boot_idx = rng.integers(0, n_train, size=n_train)

            boot_subset = Subset(ds_tr, boot_idx.tolist())

            boot_generator = torch.Generator()
            boot_generator.manual_seed(boot_seed)

            boot_loader = DataLoader(
                boot_subset,
                batch_size=BATCH_SIZE,
                shuffle=True,
                generator=boot_generator
            )

            set_global_seed(boot_seed, deterministic=USE_TORCH_DETERMINISTIC)

            model = build_separate_model(class_dim, reg_dim)
            model = train_single_model_separate(
                model=model,
                train_loader=boot_loader,
                y_scaler=factory,
                train_ys=ys_tr[boot_idx],
                device=device
            )
            models.append(model)

        # Evaluate
        eval_out = evaluate_ensemble_separate(models, va_loader, factory, device)

        # Get everything we might be interested in
        row = {
            "fold": fold_idx,
            "val_period": str(val_period),
            "n_val": len(va_idx),
            "hard_accuracy": eval_out["hard"]["accuracy"],
            "hard_balanced_accuracy": eval_out["hard"]["balanced_accuracy"],
            "hard_f1_macro": eval_out["hard"]["f1_macro"],
            "hard_mse": eval_out["hard"]["mse"],
            "hard_rmse": eval_out["hard"]["rmse"],
            "hard_mae": eval_out["hard"]["mae"],
            "hard_r2": eval_out["hard"]["r2"],
            "soft_rmse": eval_out["soft"]["rmse"],
            "soft_mae": eval_out["soft"]["mae"],
            "soft_r2": eval_out["soft"]["r2"],
            "true_rmse": eval_out["true"]["rmse"],
            "true_mae": eval_out["true"]["mae"],
            "true_r2": eval_out["true"]["r2"],
        }
        fold_rows.append(row)

        # Print everything we might interested in <- We are probably going with MAE and Balanced Accuracy
        print("=" * 95)
        print(f"MODEL: bag_sep_raw_pca40_d4 | SEED {SEED} | FOLD {fold_idx} | VAL PERIOD {val_period}")
        print("=" * 95)
        print(f"n_val                  : {row['n_val']}")
        print(f"Hard Accuracy          : {row['hard_accuracy']:.6f}")
        print(f"Hard Balanced Accuracy : {row['hard_balanced_accuracy']:.6f}")
        print(f"Hard F1 Macro          : {row['hard_f1_macro']:.6f}")
        print(f"Hard RMSE              : {row['hard_rmse']:.6f}")
        print(f"Hard MAE               : {row['hard_mae']:.6f}")
        print(f"Hard R2                : {row['hard_r2']:.6f}")
        print(f"Soft RMSE              : {row['soft_rmse']:.6f}")
        print(f"Soft MAE               : {row['soft_mae']:.6f}")
        print(f"Soft R2                : {row['soft_r2']:.6f}")
        print(f"True RMSE              : {row['true_rmse']:.6f}")
        print(f"True MAE               : {row['true_mae']:.6f}")
        print(f"True R2                : {row['true_r2']:.6f}")
        print()

    fold_df = pd.DataFrame(fold_rows)

    print("\nCV FOLD RESULTS")
    print(fold_df)

    print("\nCV MEAN")
    print(fold_df.mean(numeric_only=True))

    return fold_df


# =========================================================
# 13) RUN
# =========================================================
device = set_global_seed(SEED, deterministic=USE_TORCH_DETERMINISTIC)
cv_results = run_period_cv()

MODEL: bag_sep_raw_pca40_d4 | SEED 123 | FOLD 1 | VAL PERIOD [1]
n_val                  : 445
Hard Accuracy          : 0.865169
Hard Balanced Accuracy : 0.777441
Hard F1 Macro          : 0.761248
Hard RMSE              : 85.007559
Hard MAE               : 45.739910
Hard R2                : 0.855901
Soft RMSE              : 94.221818
Soft MAE               : 52.291180
Soft R2                : 0.822970
True RMSE              : 78.528934
True MAE               : 45.315308
True R2                : 0.877029

MODEL: bag_sep_raw_pca40_d4 | SEED 123 | FOLD 2 | VAL PERIOD [2]
n_val                  : 1244
Hard Accuracy          : 0.979100
Hard Balanced Accuracy : 0.979453
Hard F1 Macro          : 0.963224
Hard RMSE              : 255.484229
Hard MAE               : 32.285477
Hard R2                : -4.598859
Soft RMSE              : 254.631543
Soft MAE               : 32.048912
Soft R2                : -4.561549
True RMSE              : 254.816961
True MAE               : 31.534704
True R2    

In [72]:
# Evaluate on Public Test data set.
X_eval = X_pub_test
ys_eval = ys_pub_test
yc_eval = yc_pub_test

In [76]:
# =========================================================
# TRAIN JOINT OPTIMISATION MODEL ON X_train + EVALUATE ON X_pub_test
# =========================================================

# =========================================================
# 1) SETTINGS
# =========================================================
# The following has a similar structure to the code above, but again we should probably allow for doubling such that 
# we avoid any incoherencies from small, easily overlooked changes, and also everything should run on any device and be as modular as possible
# But most of the following is the same as in the code block above
SEED = 42 # Set seed so everything is reproducible

BATCH_SIZE = 128
USE_TORCH_DETERMINISTIC = True # Make everything reproducible

NUM_CLASSES = 6
PCA40 = 40
N_BOOTSTRAPS = 5

CLASS_HIDDEN = (128, 96, 64)
REG_HIDDEN = (160, 128, 96, 64)
HEAD_HIDDEN = (32, 16)

CLASS_DROPOUT = 0.08
REG_DROPOUT = 0.05
HEAD_DROPOUT = 0.05

EPOCHS = 30
LR = 3e-4
WEIGHT_DECAY = 2e-4
LAMBDA_CLASS = 8.0
LAMBDA_REG = 1.0
LABEL_SMOOTHING = 0.03
USE_CLASS_WEIGHTS = True
USE_REG_CLASS_WEIGHTS = False
USE_HUBER = True
HUBER_BETA = 1.0
CLIP_GRAD = 5.0


# =========================================================
# 2) REPRODUCIBILITY <- Make sure we control all sources of randomness
# =========================================================
def set_global_seed(seed: int, deterministic: bool = True):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        try:
            torch.use_deterministic_algorithms(True)
        except Exception:
            pass

    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def to_numpy_2d(x):
    if hasattr(x, "values"):
        x = x.values
    x = np.asarray(x)
    if x.ndim == 1:
        x = x.reshape(-1, 1)
    return x


def to_numpy_1d(x):
    if hasattr(x, "values"):
        x = x.values
    return np.asarray(x).reshape(-1)


# =========================================================
# 3) LOAD DATA
# =========================================================
X_train_np = to_numpy_2d(X_train).astype(np.float32)
ys_train_np = to_numpy_1d(ys_train).astype(int) - 1
yc_train_np = to_numpy_1d(yc_train).astype(np.float32)

X_eval_np = to_numpy_2d(X_eval).astype(np.float32)
ys_eval_np = to_numpy_1d(ys_eval).astype(int) - 1
yc_eval_np = to_numpy_1d(yc_eval).astype(np.float32)


# =========================================================
# 4) DATASET
# =========================================================
class MultiInputDataset(Dataset):
    def __init__(self, x_class, x_reg, ys, yc):
        self.x_class = torch.tensor(x_class, dtype=torch.float32)
        self.x_reg = torch.tensor(x_reg, dtype=torch.float32)
        self.ys = torch.tensor(ys, dtype=torch.long)
        self.yc = torch.tensor(yc, dtype=torch.float32)

    def __len__(self):
        return len(self.ys)

    def __getitem__(self, idx):
        return self.x_class[idx], self.x_reg[idx], self.ys[idx], self.yc[idx]


class WeightedMSELoss(nn.Module):
    def __init__(self, reduction="mean"):
        super().__init__()
        self.reduction = reduction

    def forward(self, pred, target, weights=None):
        loss = (pred - target) ** 2
        if weights is not None:
            loss = loss * weights
        if self.reduction == "mean":
            if weights is not None:
                return loss.sum() / (weights.sum() + 1e-8)
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        return loss


class WeightedHuberLoss(nn.Module):
    def __init__(self, beta=1.0, reduction="mean"):
        super().__init__()
        self.beta = beta
        self.reduction = reduction

    def forward(self, pred, target, weights=None):
        diff = pred - target
        abs_diff = torch.abs(diff)

        loss = torch.where(
            abs_diff < self.beta,
            0.5 * (diff ** 2) / self.beta,
            abs_diff - 0.5 * self.beta
        )

        if weights is not None:
            loss = loss * weights

        if self.reduction == "mean":
            if weights is not None:
                return loss.sum() / (weights.sum() + 1e-8)
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        return loss


def make_mlp(input_dim, hidden_dims, dropout=0.0, output_dim=None):
    layers = []
    prev = input_dim
    for h in hidden_dims:
        layers.append(nn.Linear(prev, h))
        layers.append(nn.ReLU())
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        prev = h
    if output_dim is not None:
        layers.append(nn.Linear(prev, output_dim))
    return nn.Sequential(*layers)


class RegressionHead(nn.Module):
    def __init__(self, input_dim, hidden_dims=(32, 16), dropout=0.0):
        super().__init__()
        self.net = make_mlp(input_dim, hidden_dims, dropout=dropout, output_dim=1)

    def forward(self, x):
        return self.net(x)


class SeparateBranchMTL(nn.Module):
    def __init__(
        self,
        class_input_dim,
        reg_input_dim,
        class_hidden=(128, 96, 64),
        reg_hidden=(160, 128, 96, 64),
        head_hidden=(32, 16),
        num_classes=6,
        class_dropout=0.08,
        reg_dropout=0.05,
        head_dropout=0.05
    ):
        super().__init__()
        self.class_net = make_mlp(class_input_dim, class_hidden, dropout=class_dropout)
        self.class_out = nn.Linear(class_hidden[-1], num_classes)

        self.reg_trunk = make_mlp(reg_input_dim, reg_hidden, dropout=reg_dropout)
        self.reg_heads = nn.ModuleList([
            RegressionHead(reg_hidden[-1], hidden_dims=head_hidden, dropout=head_dropout)
            for _ in range(num_classes)
        ])

    def forward(self, x_class, x_reg):
        class_feat = self.class_net(x_class)
        logits = self.class_out(class_feat)

        reg_feat = self.reg_trunk(x_reg)
        reg_outputs = torch.cat([head(reg_feat) for head in self.reg_heads], dim=1)
        return logits, reg_outputs


class FeatureFactory:
    def __init__(self, seed=42):
        self.seed = seed
        self.x_scaler = None
        self.y_scaler = None
        self.pca40 = None

    def fit(self, X_train, yc_train):
        self.x_scaler = StandardScaler()
        Xs = self.x_scaler.fit_transform(X_train)

        self.y_scaler = StandardScaler()
        self.y_scaler.fit(yc_train.reshape(-1, 1))

        n40 = min(PCA40, Xs.shape[1], Xs.shape[0])
        self.pca40 = PCA(n_components=n40, random_state=self.seed)
        self.pca40.fit(Xs)
        return self

    def transform_y(self, yc):
        return self.y_scaler.transform(np.asarray(yc).reshape(-1, 1)).reshape(-1)

    def inverse_transform_y(self, yc_std):
        return self.y_scaler.inverse_transform(np.asarray(yc_std).reshape(-1, 1)).reshape(-1)

    def transform_class(self, X):
        return self.x_scaler.transform(X).astype(np.float32)

    def transform_reg(self, X):
        Xs = self.x_scaler.transform(X)
        return self.pca40.transform(Xs).astype(np.float32)


def make_class_weights(ys, num_classes=6, device="cpu"):
    counts = np.bincount(ys, minlength=num_classes).astype(np.float32)
    counts[counts == 0] = 1.0
    weights = counts.sum() / (num_classes * counts)
    return torch.tensor(weights, dtype=torch.float32, device=device)


def train_single_model_separate(model, train_loader, y_scaler, train_ys, device):
    model = model.to(device)

    class_weights = make_class_weights(train_ys, num_classes=NUM_CLASSES, device=device) if USE_CLASS_WEIGHTS else None

    class_criterion = nn.CrossEntropyLoss(
        weight=class_weights,
        label_smoothing=LABEL_SMOOTHING
    )

    reg_criterion = WeightedHuberLoss(beta=HUBER_BETA) if USE_HUBER else WeightedMSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    for epoch in range(EPOCHS):
        model.train()
        for x_class, x_reg, ys_b, yc_b in train_loader:
            x_class = x_class.to(device)
            x_reg = x_reg.to(device)
            ys_b = ys_b.to(device)

            optimizer.zero_grad()

            logits, reg_outputs = model(x_class, x_reg)
            loss_class = class_criterion(logits, ys_b)

            batch_idx = torch.arange(x_class.size(0), device=device)
            pred_c_true_std = reg_outputs[batch_idx, ys_b]

            yc_std = torch.tensor(
                y_scaler.transform_y(yc_b.detach().cpu().numpy()),
                dtype=torch.float32,
                device=device
            )

            loss_reg = reg_criterion(pred_c_true_std, yc_std)
            loss = LAMBDA_CLASS * loss_class + LAMBDA_REG * loss_reg

            loss.backward()
            if CLIP_GRAD is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD)
            optimizer.step()

    return model


# Define Function for evaluating the ensemble
# We want to train on X_train and evaluate on pub_test
def evaluate_ensemble(models, loader, y_scaler, device):
    for m in models:
        m.eval()

    all_true_s = []
    all_pred_s = []
    all_true_c = []
    all_pred_c_hard = []
    all_pred_c_soft = []

    with torch.no_grad():
        for x_class, x_reg, ys_b, yc_b in loader:
            x_class = x_class.to(device)
            x_reg = x_reg.to(device)
            ys_b = ys_b.to(device)

            prob_list = []
            reg_list = []

            for model in models:
                logits, reg_outputs = model(x_class, x_reg)
                probs = torch.softmax(logits, dim=1)
                prob_list.append(probs.unsqueeze(0))
                reg_list.append(reg_outputs.unsqueeze(0))

            mean_probs = torch.mean(torch.cat(prob_list, dim=0), dim=0)
            mean_reg = torch.mean(torch.cat(reg_list, dim=0), dim=0)

            pred_s = torch.argmax(mean_probs, dim=1)
            batch_idx = torch.arange(x_class.size(0), device=device)

            pred_c_hard_std = mean_reg[batch_idx, pred_s]
            pred_c_soft_std = torch.sum(mean_probs * mean_reg, dim=1)

            pred_c_hard = y_scaler.inverse_transform_y(pred_c_hard_std.cpu().numpy())
            pred_c_soft = y_scaler.inverse_transform_y(pred_c_soft_std.cpu().numpy())

            all_true_s.extend(ys_b.cpu().numpy())
            all_pred_s.extend(pred_s.cpu().numpy())
            all_true_c.extend(yc_b.cpu().numpy())
            all_pred_c_hard.extend(pred_c_hard)
            all_pred_c_soft.extend(pred_c_soft)

    all_true_s = np.array(all_true_s)
    all_pred_s = np.array(all_pred_s)
    all_true_c = np.array(all_true_c)
    all_pred_c_hard = np.array(all_pred_c_hard)
    all_pred_c_soft = np.array(all_pred_c_soft)

    return {
        "balanced_accuracy": balanced_accuracy_score(all_true_s, all_pred_s),
        "hard_mae": mean_absolute_error(all_true_c, all_pred_c_hard),
        "soft_mae": mean_absolute_error(all_true_c, all_pred_c_soft),
        "accuracy": accuracy_score(all_true_s, all_pred_s),
        "f1_macro": f1_score(all_true_s, all_pred_s, average="macro"),
        "hard_rmse": np.sqrt(mean_squared_error(all_true_c, all_pred_c_hard)),
        "soft_rmse": np.sqrt(mean_squared_error(all_true_c, all_pred_c_soft)),
        "hard_r2": r2_score(all_true_c, all_pred_c_hard),
        "soft_r2": r2_score(all_true_c, all_pred_c_soft),
    }


# =========================================================
# 5) TRAIN FULL ENSEMBLE + EVALUATE
# =========================================================
# So now we get to evaluate the model <- we want to train on X_train and evaluate on pub_test
device = set_global_seed(SEED, deterministic=USE_TORCH_DETERMINISTIC)

factory = FeatureFactory(seed=SEED)
factory.fit(X_train_np, yc_train_np)

# Train on X_train
xclass_train = factory.transform_class(X_train_np)
xreg_train = factory.transform_reg(X_train_np)

# Evaluate on X_pub_test <- was renamed above to X_eval
xclass_eval = factory.transform_class(X_eval_np)
xreg_eval = factory.transform_reg(X_eval_np)

ds_train = MultiInputDataset(xclass_train, xreg_train, ys_train_np, yc_train_np)
ds_eval = MultiInputDataset(xclass_eval, xreg_eval, ys_eval_np, yc_eval_np)

eval_loader = DataLoader(ds_eval, batch_size=BATCH_SIZE, shuffle=False)

class_dim = xclass_train.shape[1]
reg_dim = xreg_train.shape[1]
n_train = len(ds_train)

models = []

# Run the Bagging for the N Bootstrap data sets (samples)
for b in range(N_BOOTSTRAPS):
    boot_seed = SEED + 10000 + b
    rng = np.random.default_rng(boot_seed)
    boot_idx = rng.integers(0, n_train, size=n_train)

    boot_subset = Subset(ds_train, boot_idx.tolist())

    boot_generator = torch.Generator()
    boot_generator.manual_seed(boot_seed)

    boot_loader = DataLoader(
        boot_subset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=boot_generator
    )

    set_global_seed(boot_seed, deterministic=USE_TORCH_DETERMINISTIC) # Ensure reproducibility

    model = SeparateBranchMTL(
        class_input_dim=class_dim,
        reg_input_dim=reg_dim,
        class_hidden=CLASS_HIDDEN,
        reg_hidden=REG_HIDDEN,
        head_hidden=HEAD_HIDDEN,
        num_classes=NUM_CLASSES,
        class_dropout=CLASS_DROPOUT,
        reg_dropout=REG_DROPOUT,
        head_dropout=HEAD_DROPOUT
    )

    model = train_single_model_separate(
        model=model,
        train_loader=boot_loader,
        y_scaler=factory,
        train_ys=ys_train_np[boot_idx],
        device=device
    )

    models.append(model)

# Finally, evaluate the model, trained on X_train on X_pub_test 
eval_results = evaluate_ensemble(models, eval_loader, factory, device)

# Print everything we might be interested in
print("Evaluation results")
print("Balanced Accuracy :", eval_results["balanced_accuracy"])
print("Hard routed MAE   :", eval_results["hard_mae"])
print("Soft routed MAE   :", eval_results["soft_mae"])
print("Accuracy          :", eval_results["accuracy"])
print("F1 Macro          :", eval_results["f1_macro"])
print("Hard routed RMSE  :", eval_results["hard_rmse"])
print("Soft routed RMSE  :", eval_results["soft_rmse"])
print("Hard routed R2  :", eval_results["hard_r2"])
print("Soft routed R2  :", eval_results["soft_r2"])

Evaluation results
Balanced Accuracy : 0.9155925268155221
Hard routed MAE   : 21.067590713500977
Soft routed MAE   : 20.69918441772461
Accuracy          : 0.9097702740105176
F1 Macro          : 0.9139444731054344
Hard routed RMSE  : 35.38261843758882
Soft routed RMSE  : 34.54887841884307
Hard routed R2  : 0.8103026151657104
Soft routed R2  : 0.8191372156143188


# Applying the Final Model to get Predictions for X_priv_test

In [80]:
# =========================================================
# TRAIN JOINT OPTIMISATION MODEL ON X_train + X_pub_test & PREDICT ON X_priv_test
# SAVE FORECASTS TO CSV
# =========================================================

# =========================================================
# 1) SETTINGS
# =========================================================
# First Part again basically the same as before, but just to make sure everything is modular
SEED = 42 # Set seed for reproducibility

BATCH_SIZE = 128
USE_TORCH_DETERMINISTIC = True # Make sure we can reproduce everything

NUM_CLASSES = 6
PCA40 = 40
N_BOOTSTRAPS = 5

CLASS_HIDDEN = (128, 96, 64)
REG_HIDDEN = (160, 128, 96, 64)
HEAD_HIDDEN = (32, 16)

CLASS_DROPOUT = 0.08
REG_DROPOUT = 0.05
HEAD_DROPOUT = 0.05

EPOCHS = 30
LR = 3e-4
WEIGHT_DECAY = 2e-4
LAMBDA_CLASS = 8.0
LAMBDA_REG = 1.0
LABEL_SMOOTHING = 0.03
USE_CLASS_WEIGHTS = True
USE_HUBER = True
HUBER_BETA = 1.0
CLIP_GRAD = 5.0

# Name from the original search for potential models <- Describes model architecture
OUTPUT_CSV_NAME = "priv_test_forecasts_bag_sep_raw_pca40_d4.csv" 


# =========================================================
# 2) REPRODUCIBILITY <- Make sure everything is again nicely reproducible
# =========================================================
def set_global_seed(seed: int, deterministic: bool = True):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        try:
            torch.use_deterministic_algorithms(True)
        except Exception:
            pass

    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Make sure we don't get problems with data types
def to_numpy_2d(x):
    if hasattr(x, "values"):
        x = x.values
    x = np.asarray(x)
    if x.ndim == 1:
        x = x.reshape(-1, 1)
    return x


def to_numpy_1d(x):
    if hasattr(x, "values"):
        x = x.values
    return np.asarray(x).reshape(-1)


# =========================================================
# 3) LOAD AND COMBINE DATA
# =========================================================
# We train on X_train AND X_pub_test to maximise data usage (use all data potentially available to us)
# Then makes substance and concentration predictions with that model

# train
X_train_np = to_numpy_2d(X_train).astype(np.float32)
ys_train_np = to_numpy_1d(ys_train).astype(int) - 1
yc_train_np = to_numpy_1d(yc_train).astype(np.float32)

# public test 
X_pub_test_np = to_numpy_2d(X_pub_test).astype(np.float32)
ys_pub_test_np = to_numpy_1d(ys_pub_test).astype(int) - 1
yc_pub_test_np = to_numpy_1d(yc_pub_test).astype(np.float32)

# private test 
X_priv_test_np = to_numpy_2d(X_priv_test).astype(np.float32)

# Make sure we are still using the now transformed labels 0 to 5 instead of 1 to 6 for PyTorch
assert ys_train_np.min() == 0 and ys_train_np.max() == 5
assert ys_pub_test_np.min() == 0 and ys_pub_test_np.max() == 5

# combine all labeled data
X_all_np = np.vstack([X_train_np, X_pub_test_np]).astype(np.float32)
ys_all_np = np.concatenate([ys_train_np, ys_pub_test_np]).astype(int)
yc_all_np = np.concatenate([yc_train_np, yc_pub_test_np]).astype(np.float32)

print("Combined labeled training shape:", X_all_np.shape)
print("Private test shape:", X_priv_test_np.shape)


# =========================================================
# 4) DATASETS
# =========================================================
class MultiInputDatasetTrain(Dataset):
    def __init__(self, x_class, x_reg, ys, yc):
        self.x_class = torch.tensor(x_class, dtype=torch.float32)
        self.x_reg = torch.tensor(x_reg, dtype=torch.float32)
        self.ys = torch.tensor(ys, dtype=torch.long)
        self.yc = torch.tensor(yc, dtype=torch.float32)

    def __len__(self):
        return len(self.ys)

    def __getitem__(self, idx):
        return self.x_class[idx], self.x_reg[idx], self.ys[idx], self.yc[idx]


class MultiInputDatasetPredict(Dataset):
    def __init__(self, x_class, x_reg):
        self.x_class = torch.tensor(x_class, dtype=torch.float32)
        self.x_reg = torch.tensor(x_reg, dtype=torch.float32)

    def __len__(self):
        return len(self.x_class)

    def __getitem__(self, idx):
        return self.x_class[idx], self.x_reg[idx]


# =========================================================
# 5) LOSSES / MODEL
# =========================================================
class WeightedHuberLoss(nn.Module):
    def __init__(self, beta=1.0, reduction="mean"):
        super().__init__()
        self.beta = beta
        self.reduction = reduction

    def forward(self, pred, target, weights=None):
        diff = pred - target
        abs_diff = torch.abs(diff)

        loss = torch.where(
            abs_diff < self.beta,
            0.5 * (diff ** 2) / self.beta,
            abs_diff - 0.5 * self.beta
        )

        if weights is not None:
            loss = loss * weights

        if self.reduction == "mean":
            if weights is not None:
                return loss.sum() / (weights.sum() + 1e-8)
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        return loss


def make_mlp(input_dim, hidden_dims, dropout=0.0, output_dim=None):
    layers = []
    prev = input_dim
    for h in hidden_dims:
        layers.append(nn.Linear(prev, h))
        layers.append(nn.ReLU())
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        prev = h
    if output_dim is not None:
        layers.append(nn.Linear(prev, output_dim))
    return nn.Sequential(*layers)


class RegressionHead(nn.Module):
    def __init__(self, input_dim, hidden_dims=(32, 16), dropout=0.0):
        super().__init__()
        self.net = make_mlp(input_dim, hidden_dims, dropout=dropout, output_dim=1)

    def forward(self, x):
        return self.net(x)


class SeparateBranchMTL(nn.Module):
    def __init__(
        self,
        class_input_dim,
        reg_input_dim,
        class_hidden=(128, 96, 64),
        reg_hidden=(160, 128, 96, 64),
        head_hidden=(32, 16),
        num_classes=6,
        class_dropout=0.08,
        reg_dropout=0.05,
        head_dropout=0.05
    ):
        super().__init__()
        self.class_net = make_mlp(class_input_dim, class_hidden, dropout=class_dropout)
        self.class_out = nn.Linear(class_hidden[-1], num_classes)

        self.reg_trunk = make_mlp(reg_input_dim, reg_hidden, dropout=reg_dropout)
        self.reg_heads = nn.ModuleList([
            RegressionHead(reg_hidden[-1], hidden_dims=head_hidden, dropout=head_dropout)
            for _ in range(num_classes)
        ])

    def forward(self, x_class, x_reg):
        class_feat = self.class_net(x_class)
        logits = self.class_out(class_feat)

        reg_feat = self.reg_trunk(x_reg)
        reg_outputs = torch.cat([head(reg_feat) for head in self.reg_heads], dim=1)

        return logits, reg_outputs


# =========================================================
# 6) FEATURE FACTORY
# =========================================================
class FeatureFactory:
    def __init__(self, seed=42):
        self.seed = seed
        self.x_scaler = None
        self.y_scaler = None
        self.pca40 = None

    def fit(self, X_train, yc_train):
        self.x_scaler = StandardScaler()
        Xs = self.x_scaler.fit_transform(X_train)

        self.y_scaler = StandardScaler()
        self.y_scaler.fit(yc_train.reshape(-1, 1))

        n40 = min(PCA40, Xs.shape[1], Xs.shape[0])
        self.pca40 = PCA(n_components=n40, random_state=self.seed)
        self.pca40.fit(Xs)
        return self

    def transform_y(self, yc):
        return self.y_scaler.transform(np.asarray(yc).reshape(-1, 1)).reshape(-1)

    def inverse_transform_y(self, yc_std):
        return self.y_scaler.inverse_transform(np.asarray(yc_std).reshape(-1, 1)).reshape(-1)

    def transform_class(self, X):
        return self.x_scaler.transform(X).astype(np.float32)

    def transform_reg(self, X):
        Xs = self.x_scaler.transform(X)
        return self.pca40.transform(Xs).astype(np.float32)


# =========================================================
# 7) TRAINING HELPERS
# =========================================================
def make_class_weights(ys, num_classes=6, device="cpu"):
    counts = np.bincount(ys, minlength=num_classes).astype(np.float32)
    counts[counts == 0] = 1.0
    weights = counts.sum() / (num_classes * counts)
    return torch.tensor(weights, dtype=torch.float32, device=device)


def train_single_model_separate(model, train_loader, y_scaler, train_ys, device):
    model = model.to(device)

    class_weights = make_class_weights(train_ys, num_classes=NUM_CLASSES, device=device) if USE_CLASS_WEIGHTS else None
    class_criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
    reg_criterion = WeightedHuberLoss(beta=HUBER_BETA)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    for epoch in range(EPOCHS):
        model.train()

        for x_class, x_reg, ys_b, yc_b in train_loader:
            x_class = x_class.to(device)
            x_reg = x_reg.to(device)
            ys_b = ys_b.to(device)

            optimizer.zero_grad()

            logits, reg_outputs = model(x_class, x_reg)
            loss_class = class_criterion(logits, ys_b)

            batch_idx = torch.arange(x_class.size(0), device=device)
            pred_c_true_std = reg_outputs[batch_idx, ys_b]

            yc_std = torch.tensor(
                y_scaler.transform_y(yc_b.detach().cpu().numpy()),
                dtype=torch.float32,
                device=device
            )

            loss_reg = reg_criterion(pred_c_true_std, yc_std)
            loss = LAMBDA_CLASS * loss_class + LAMBDA_REG * loss_reg

            loss.backward()
            if CLIP_GRAD is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD)
            optimizer.step()

    return model


# =========================================================
# 8) TRAIN FULL ENSEMBLE ON ALL LABELED DATA
# =========================================================
device = set_global_seed(SEED, deterministic=USE_TORCH_DETERMINISTIC) # Set seed to ensure reproducibility

factory = FeatureFactory(seed=SEED)
factory.fit(X_all_np, yc_all_np)

xclass_all = factory.transform_class(X_all_np)
xreg_all = factory.transform_reg(X_all_np)

ds_train = MultiInputDatasetTrain(xclass_all, xreg_all, ys_all_np, yc_all_np)

class_dim = xclass_all.shape[1]
reg_dim = xreg_all.shape[1]
n_train = len(ds_train)

models = []

for b in range(N_BOOTSTRAPS):
    boot_seed = SEED + 10000 + b
    rng = np.random.default_rng(boot_seed)
    boot_idx = rng.integers(0, n_train, size=n_train)

    boot_subset = Subset(ds_train, boot_idx.tolist())

    boot_generator = torch.Generator()
    boot_generator.manual_seed(boot_seed)

    boot_loader = DataLoader(
        boot_subset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=boot_generator
    )

    set_global_seed(boot_seed, deterministic=USE_TORCH_DETERMINISTIC)

    model = SeparateBranchMTL(
        class_input_dim=class_dim,
        reg_input_dim=reg_dim,
        class_hidden=CLASS_HIDDEN,
        reg_hidden=REG_HIDDEN,
        head_hidden=HEAD_HIDDEN,
        num_classes=NUM_CLASSES,
        class_dropout=CLASS_DROPOUT,
        reg_dropout=REG_DROPOUT,
        head_dropout=HEAD_DROPOUT
    )

    model = train_single_model_separate(
        model=model,
        train_loader=boot_loader,
        y_scaler=factory,
        train_ys=ys_all_np[boot_idx],
        device=device
    )

    models.append(model)


# =========================================================
# 9) PREDICT ON X_priv_test <- Make our final predictions for X_priv_test
# =========================================================
xclass_priv = factory.transform_class(X_priv_test_np)
xreg_priv = factory.transform_reg(X_priv_test_np)

ds_priv = MultiInputDatasetPredict(xclass_priv, xreg_priv)
priv_loader = DataLoader(
    ds_priv,
    batch_size=BATCH_SIZE,
    shuffle=False
)

for model in models:
    model.eval()

all_pred_substance = []
all_pred_conc_hard = []
all_pred_conc_soft = []

with torch.no_grad():
    for x_class, x_reg in priv_loader:
        x_class = x_class.to(device)
        x_reg = x_reg.to(device)

        prob_list = []
        reg_list = []

        for model in models:
            logits, reg_outputs = model(x_class, x_reg)
            probs = torch.softmax(logits, dim=1)

            prob_list.append(probs.unsqueeze(0))
            reg_list.append(reg_outputs.unsqueeze(0))

        mean_probs = torch.mean(torch.cat(prob_list, dim=0), dim=0)
        mean_reg = torch.mean(torch.cat(reg_list, dim=0), dim=0)

        pred_s = torch.argmax(mean_probs, dim=1)
        batch_idx = torch.arange(x_class.size(0), device=device)

        pred_c_hard_std = mean_reg[batch_idx, pred_s]
        pred_c_soft_std = torch.sum(mean_probs * mean_reg, dim=1)

        pred_c_hard = factory.inverse_transform_y(pred_c_hard_std.cpu().numpy())
        pred_c_soft = factory.inverse_transform_y(pred_c_soft_std.cpu().numpy())

        all_pred_substance.extend(pred_s.cpu().numpy())
        all_pred_conc_hard.extend(pred_c_hard)
        all_pred_conc_soft.extend(pred_c_soft)

all_pred_substance = np.array(all_pred_substance)
all_pred_conc_hard = np.array(all_pred_conc_hard)
all_pred_conc_soft = np.array(all_pred_conc_soft)

# Go back to original labels 1..6 so no confusion is created
pred_substance_1to6 = all_pred_substance + 1

priv_ids = X_priv_test.index.to_numpy() # ID ordering should stay exactly the same
prediction_df = pd.DataFrame({
    "id": priv_ids,
    "pred_substance": pred_substance_1to6,
    "pred_concentration_soft": all_pred_conc_soft,
})

print(prediction_df.head())
print("Number of predictions:", len(prediction_df))


# =========================================================
# 10) SAVE CSV <- Get our final CSV file!
# =========================================================
prediction_df.to_csv(OUTPUT_CSV_NAME, # Includes the ID, the predicted Substance and Concentration (soft routed)
                     index=False,
                     sep=",",
                     encoding="utf-8-sig") # So Excel recognises it correctly

print(f"\nCSV file saved as: {OUTPUT_CSV_NAME}") 
print(os.path.abspath(OUTPUT_CSV_NAME))

Combined labeled training shape: (9546, 120)
Private test shape: (3600, 120)
   id  pred_substance  pred_concentration_soft
0   0               6                13.825523
1   1               2                24.778816
2   2               6               108.212784
3   3               2                39.891937
4   4               6                53.166672
Number of predictions: 3600

CSV file saved as: priv_test_forecasts_bag_sep_raw_pca40_d4.csv
C:\Users\shug8381\priv_test_forecasts_bag_sep_raw_pca40_d4.csv
